# Makilala FoS Pipeline — v11

**What changed from v10**

1. `§27 BLOCK SIZE` — the practical-range estimator is corrected. v10 took the
   variogram sill from lags beyond 8 km, where both variograms *decline* because
   of large-scale non-stationarity across an elongated domain. That put the sill
   below the true plateau, so 95% of it was crossed in the first bin and the range
   printed as 50–250 m. The sill is now the mean of the 1.6–4.5 km plateau and the
   range is the lag from which gamma *stays* above the threshold. Corrected values:
   residual range 750 m (99% at 1.65 km), indicator 1.15 km (99% at 1.95 km) —
   so the 2 km block exceeds both. **Re-run this cell**; its v10 output was cleared.
2. `§28 BLOCKED VALIDATION FIGURE` — new. Writes `fig7_blocked_validation.png`
   and `table_S_blocked_tiles.tex`, the figure and per-tile table Section 6.4 was
   missing.
3. Section numbers made unique so the outline can be navigated: the four repeated
   point-level cells become §22/§22a/§22b/§22c, and the four trailing analysis
   cells are renumbered §25 threshold transfer, §26 tile selection, §27 block size,
   §28 blocked-validation figure. Comment headers only; no code changed.

Cells 1–28 keep their v10 outputs. Run All reproduces everything.

---


**Run All, top to bottom.** Every number the manuscript reports is recomputed here
and checked against its published value; the last cell prints one block you can
paste back in full.

### What changed from v9

v9 is *correct* — v10 reproduces it, it does not replace it. Two structural
changes only:

1. **No global rebinding.** v9 bound `alpha`, `c`, `scored` … as loose globals and
   later cells reassigned some of them, so by the end of a Run All
   `alpha[scored].min()` came back `0.10` when the scored mask enforces
   `alpha >= 10`. Here the canonical arrays live in a frozen namespace `P`, the
   evaluation mask is the module constant `SCORED`, and `guard()` at the top of
   every downstream cell re-asserts both. Drift becomes an exception, not a
   silent wrong number.
2. **Correct domain for the parameter table.** v9's Table 3 "Footprint" column
   was computed over the full raster extent (21,106,173 px) instead of the
   municipal footprint (13,728,083 px), which is why saturation appeared to vary
   (0.50–0.60) when it is identically 0.60 inside the municipality — the 0.50
   being `estimate_m`'s missing-data fallback on pixels outside it.

Outputs go to `output_maps_v10/`. Nothing in `output_maps_v9/` is touched.

In [1]:
# ============================================================================
#  §0  CONFIGURATION
#  The only cell that names a path, a CRS or a constant.
# ============================================================================
import os, sys, json, warnings, time
import numpy as np

# Override with the MAKILALA_DIR environment variable if the data lives
# elsewhere (e.g. running this script outside the desktop notebook).
INPUT_DIR  = os.environ.get("MAKILALA_DIR",
                            "/Users/jayroldarcede/Downloads/makilala_data")
OUTPUT_DIR = os.path.join(INPUT_DIR, "output_maps_v10")
os.makedirs(OUTPUT_DIR, exist_ok=True)

FILE_SLOPE = "Slope_PRSTM51N.tif";        FILE_DEPTH = "Soil Depth.tif"
FILE_PGA   = "PGA 2019 EQ Events.tif";    FILE_GAMMA = "Mak_BulkUnitWeight.tif"
FILE_WATER = "Mak_WaterContent.tif";      FILE_PI    = "Mak_PlasticityIndex.tif"
FILE_FINES = "Mak_Fines.tif";             FILE_SAND  = "Mak_Sand.tif"
FILE_LSI   = "LSI_North Cotabato.shp"

REFERENCE_GRID = 'gamma'      # gamma DEFINES the grid; it is never resampled
ALPHA_MIN      = 10           # degrees
GAMMA_W        = 9.81
NODATA         = -9999.0
ALL_TOUCHED    = False        # `observed` is centre-only; poly_id is touched-cell
Z_FILL_M       = 1.2

# CRS. See the v9 §0 note: the Mak_*.tif tags are mutually inconsistent and the
# EPSG:32651 reading is the verified one; the inventory's .prj says ESRI:102457
# but the GPS ground truth says WGS 84, so it is RELABELLED, never reprojected.
GRID_CRS = "EPSG:32651"
LSI_CRS  = "EPSG:32651"
CRS_OVERRIDE = {'slope': None, 'depth': None, 'pga': None}

# Threshold sweep
THRESHOLDS = np.arange(0.40, 2.05, 0.05)
THRESHOLDS_EXT = np.arange(0.05, 2.05, 0.05)   # floor test

# Primary configuration
PRIMARY_C = 'c_plus10'
PRIMARY_Z = 'z_cap_5m'

# ---- published values, for the PASS/FAIL checks ----------------------------
PUBLISHED = dict(
    footprint_px=13_728_083, scored_px=7_104_440, extent_px=21_106_173,
    poly_province=3140, poly_makilala=1703, poly_scored=1682,
    px_makilala=177_589, px_scored=170_547,
    alpha_ge10_px=7_158_603,   # corrected 2026-09-15; the table footnote
                               # (54,163 removed by depth) already implied it
    theta_star=0.90, youden=0.409, auc=0.7503, hit=0.642, far=0.233,
    blocked_auc=0.667, blocked_sd=0.155,   # 0.1548; was 0.153, stale blocked_tiles=45, blocked_total=108,
    formb_mean=0.387, formb_frac=0.831, depth_share=0.715,
)

print(f"input : {INPUT_DIR}")
print(f"output: {OUTPUT_DIR}")
print(f"python: {sys.version.split()[0]}   numpy: {np.__version__}")

input : /Users/jayroldarcede/Downloads/makilala_data
output: /Users/jayroldarcede/Downloads/makilala_data/output_maps_v10
python: 3.10.20   numpy: 1.23.5


In [2]:
# ============================================================================
#  §1  HELPERS
#  Model functions are byte-for-byte the v9 definitions.
# ============================================================================
import rasterio
from rasterio.crs import CRS
from rasterio.warp import reproject, Resampling
from rasterio.features import rasterize
try:
    import geopandas as gpd
    from shapely.geometry import mapping
except ImportError:
    gpd = None; print("!! geopandas missing -- inventory steps will fail")


def estimate_phi(pi, sand, fines):
    phi = np.full_like(pi, 30.0, dtype=np.float64)
    v = (pi > 5) & np.isfinite(pi)
    phi[v] = 35.0 - 0.1 * pi[v]
    no = ~v & np.isfinite(sand)
    phi[no & (sand > 85) & (fines < 10)] = 33.0
    phi[no & (sand > 70) & (fines >= 10) & (fines < 20)] = 31.0
    phi[no & (fines >= 20)] = 28.0
    return np.clip(phi, 20.0, 38.0)


def estimate_c(fines, sand):
    c = np.full_like(fines, 5.0, dtype=np.float64)
    v = np.isfinite(fines) & np.isfinite(sand)
    c[v & (fines < 5)] = 1.0
    c[v & (fines >= 5) & (fines < 12)] = 3.0
    c[v & (fines >= 12) & (fines < 20)] = 6.0
    c[v & (fines >= 20)] = 10.0
    return c


def estimate_m(water):
    '''NB: 0.50 is the MISSING-DATA FALLBACK, not a saturation band.
    The bands are 0.2 / 0.4 / 0.6 / 0.8.'''
    m = np.full_like(water, 0.5, dtype=np.float64)
    v = np.isfinite(water) & (water > 0)
    m[v & (water < 20)] = 0.2
    m[v & (water >= 20) & (water < 35)] = 0.4
    m[v & (water >= 35) & (water < 50)] = 0.6
    m[v & (water >= 50)] = 0.8
    return m


def compute_FoS(c, phi_deg, alpha_deg, m, gamma, z, k_h):
    phi, alpha = np.radians(phi_deg), np.radians(alpha_deg)
    R = c + (gamma - m * GAMMA_W) * z * np.cos(alpha)**2 * np.tan(phi)
    D = gamma * z * np.sin(alpha) * np.cos(alpha) + k_h * gamma * z * np.cos(alpha)**2
    return np.where(D > 0, R / D, np.nan)


def compute_kc(c, phi_deg, alpha_deg, m, gamma, z):
    phi, alpha = np.radians(phi_deg), np.radians(alpha_deg)
    d = gamma * z * np.cos(alpha)**2
    return np.where(d > 0, c/d + (1 - m*GAMMA_W/gamma)*np.tan(phi) - np.tan(alpha), np.nan)


def sensitivity_FormB(fs):
    return np.where(fs > 1, 1.0 - 1.0/fs, np.nan)


def ok(label, got, want, tol, unit=""):
    '''PASS/FAIL against a published value.'''
    good = abs(got - want) <= tol
    flag = "PASS" if good else "**FAIL**"
    gs = f"{got:,}" if isinstance(got, (int, np.integer)) else f"{got:.4f}"
    ws = f"{want:,}" if isinstance(want, (int, np.integer)) else f"{want:.4f}"
    print(f"  [{flag:8s}] {label:<44} {gs:>14}{unit}   published {ws}{unit}")
    CHECKS.append((label, good, gs, ws))
    return good


CHECKS = []
print("helpers defined")

helpers defined


In [3]:
# ============================================================================
#  §2  LOAD AND REGISTER
#  Layers already on the reference grid PASS THROUGH untouched. Reprojecting
#  them would apply a datum shift to some and not others -- the Mak_* tags are
#  mutually inconsistent, so at most one can be right.
# ============================================================================
t0 = time.time()

def load_raster(path, label=""):
    with rasterio.open(path) as src:
        d = src.read(1).astype(np.float64)
        p = src.profile.copy()
        if src.nodata is not None:
            d[d == src.nodata] = np.nan
        v = np.isfinite(d) & (d != 0)
        print(f"  {label:<7}: {str(d.shape):>14} @ {src.res[0]:>8.4g}"
              f"  {str(src.crs):<14} valid={int(v.sum()):>11,}")
        return d, p


FILES = {'slope': FILE_SLOPE, 'depth': FILE_DEPTH, 'pga': FILE_PGA,
         'gamma': FILE_GAMMA, 'water': FILE_WATER, 'pi': FILE_PI,
         'fines': FILE_FINES, 'sand': FILE_SAND}

rasters, profiles = {}, {}
for key, fn in FILES.items():
    _path = os.path.join(INPUT_DIR, fn)
    if not os.path.exists(_path):
        raise FileNotFoundError(f"{key}: {_path}")
    rasters[key], profiles[key] = load_raster(_path, key)

ref_shape     = rasters[REFERENCE_GRID].shape
ref_transform = profiles[REFERENCE_GRID]['transform']
ref_crs       = CRS.from_string(GRID_CRS)
ref_profile   = {**profiles[REFERENCE_GRID], 'crs': ref_crs}

print(f"\nreference grid: {ref_shape[0]} x {ref_shape[1]} @ {ref_transform.a:g} m"
      f"  = {ref_shape[0]*ref_shape[1]:,} cells")
print(f"  file tag : {profiles[REFERENCE_GRID].get('crs')}")
print(f"  GRID_CRS : {ref_crs}  <- used\n")

aligned = {}
for key, data in rasters.items():
    same = (data.shape == ref_shape
            and profiles[key]['transform'].almost_equals(ref_transform, precision=1e-6))
    if same:
        aligned[key] = data
        print(f"  = {key:<7}: on the reference grid, passed through untouched")
        continue
    src_crs = CRS_OVERRIDE.get(key) or profiles[key]['crs']
    if src_crs is None:
        raise RuntimeError(f"{key}: no CRS and no override")
    dst = np.full(ref_shape, np.nan, dtype=np.float64)
    reproject(source=data, destination=dst,
              src_transform=profiles[key]['transform'], src_crs=CRS.from_user_input(src_crs),
              dst_transform=ref_transform, dst_crs=ref_crs,
              resampling=Resampling.bilinear, src_nodata=np.nan, dst_nodata=np.nan)
    aligned[key] = dst
    print(f"  > {key:<7}: reprojected from {src_crs}, coverage "
          f"{100*np.isfinite(dst).mean():.1f}% of the grid")

del rasters
print(f"\nregistration complete in {time.time()-t0:.1f}s")

  slope  :   (3716, 5731) @        5  ESRI:102457    valid= 13,743,753
  depth  :     (444, 461) @      229  ESRI:102457    valid=    110,490
  pga    :     (702, 701) @      555  ESRI:102457    valid=    487,969
  gamma  :   (3697, 5709) @        5  ESRI:102457    valid= 13,762,464
  water  :   (3697, 5709) @        5  EPSG:32651     valid= 13,762,464
  pi     :   (3697, 5709) @        5  ESRI:102457    valid= 13,762,464
  fines  :   (3697, 5709) @        5  EPSG:32651     valid= 13,762,464
  sand   :   (3697, 5709) @        5  EPSG:32651     valid= 13,762,464

reference grid: 3697 x 5709 @ 5 m  = 21,106,173 cells
  file tag : ESRI:102457
  GRID_CRS : EPSG:32651  <- used

  > slope  : reprojected from ESRI:102457, coverage 65.1% of the grid
  > depth  : reprojected from ESRI:102457, coverage 80.0% of the grid
  > pga    : reprojected from ESRI:102457, coverage 100.0% of the grid
  = gamma  : on the reference grid, passed through untouched
  = water  : on the reference grid, passed thr

In [4]:
# ============================================================================
#  §3  DERIVED PARAMETERS  ->  frozen namespace P
#
#  Everything downstream reads P.<name>. Nothing rebinds a bare `alpha`, `c`
#  or `m` at module level, which is the v9 failure this version exists to stop.
# ============================================================================
from types import SimpleNamespace

_alpha = np.clip(aligned['slope'], 0.1, 80.0)
_gamma = np.clip(aligned['gamma'], GAMMA_W + 0.1, 30.0)
_phi   = estimate_phi(aligned['pi'], aligned['sand'], aligned['fines'])
_c     = estimate_c(aligned['fines'], aligned['sand'])
_m     = estimate_m(aligned['water'])
_z_raw = aligned['depth']

_pga = aligned['pga']
_pmax = np.nanmax(_pga[np.isfinite(_pga) & (_pga > 0)])
_pga_g = _pga/981.0 if _pmax > 100 else (_pga/9.81 if _pmax > 2 else _pga)
_k_h = np.clip(0.5 * _pga_g, 0.0, 1.0)

# FOOTPRINT: where the interpolated geotechnical rasters carry values.
# v9 uses isfinite(gamma) & isfinite(alpha); water gives the identical set.
FOOTPRINT = np.isfinite(_gamma) & np.isfinite(_alpha)

# gap-filled depth, for isolating the coverage effect
_z_missing = FOOTPRINT & (~np.isfinite(_z_raw) | (_z_raw <= 0))

# Only what is read downstream is retained. At 169 MB per float64 array on a
# 21.1 M-cell grid, keeping the full `aligned` dict as well costs 1.3 GB for
# nothing, and the sweep in §7 needs that headroom.
P = SimpleNamespace(alpha=_alpha, gamma=_gamma, phi=_phi, c=_c, m=_m,
                    z_raw=_z_raw, z_missing=_z_missing,
                    k_h=_k_h, pga_g=_pga_g, water=aligned['water'])
for _a in vars(P).values():
    if isinstance(_a, np.ndarray):
        _a.flags.writeable = False          # make accidental mutation raise

del aligned, _alpha, _gamma, _phi, _c, _m, _z_raw, _k_h, _pga_g
import gc; gc.collect()

_PIN = {'scored_sig': None}
_FINGERPRINT = {k: (float(np.nansum(v)), v.shape)
                for k, v in vars(P).items() if isinstance(v, np.ndarray)}


def guard():
    '''Re-assert that P and SCORED are intact. Call at the top of every cell.'''
    for k, (s, sh) in _FINGERPRINT.items():
        a = getattr(P, k)
        if a.shape != sh or not np.isclose(float(np.nansum(a)), s, rtol=0, atol=1e-6):
            raise RuntimeError(f"P.{k} has been mutated since §3 -- rerun from the top")
    if 'SCORED' in globals():
        n = int(SCORED.sum())
        # Count alone is not enough: the v9 failure was a mask with the RIGHT
        # cardinality and the WRONG membership. Two cheap spatial moments (row
        # and column profiles) pin the actual footprint of the mask, not just
        # its size.
        _rs = SCORED.sum(axis=1); _cs = SCORED.sum(axis=0)
        sig = (n,
               int((_rs * np.arange(_rs.size)).sum()),
               int((_cs * np.arange(_cs.size)).sum()))
        if _PIN.get('scored_sig') is None:
            _PIN['scored_sig'] = sig
        elif sig != _PIN['scored_sig']:
            was, now = _PIN['scored_sig'], sig
            what = ("count" if was[0] != now[0] else "membership (same count)")
            raise RuntimeError(
                f"SCORED has changed -- {what}. Was {was}, now {now}. "
                f"Something rebound or mutated it; rerun from the top.")
        # NaN-safe: a plain .min() over a mask that includes nodata pixels
        # returns nan, and `nan < ALPHA_MIN` is False -- the check would pass
        # silently on exactly the corruption it is meant to catch.
        _av = P.alpha[SCORED]
        _nn = int(np.count_nonzero(~np.isfinite(_av)))
        if _nn:
            raise RuntimeError(f"SCORED selects {_nn:,} px where alpha is not "
                               f"finite -- it is not the evaluation domain")
        am = float(np.nanmin(_av))
        if am < ALPHA_MIN - 1e-9:
            raise RuntimeError(f"SCORED contains alpha={am:.2f} deg < {ALPHA_MIN} "
                               f"-- it is not the evaluation domain any more")
    return True


print(f"footprint (isfinite gamma & alpha): {int(FOOTPRINT.sum()):,}")
_alt = int((np.isfinite(P.water) & np.isfinite(P.alpha)).sum())
print(f"footprint (isfinite water & alpha): {_alt:,}   {'(identical)' if _alt==int(FOOTPRINT.sum()) else '(DIFFERS)'}")
print(f"P frozen with {len(_FINGERPRINT)} arrays; guard() armed")

footprint (isfinite gamma & alpha): 13,728,083
footprint (isfinite water & alpha): 13,728,083   (identical)
P frozen with 10 arrays; guard() armed


In [5]:
# ============================================================================
#  §4  INVENTORY
#  The shapefile's .prj says ESRI:102457. It is RELABELLED to EPSG:32651, not
#  reprojected -- see the §0 note. poly_id is touched-cell; `observed` is
#  centre-only. That asymmetry is deliberate and is reported in Table 2.
# ============================================================================
guard()

lsi = gpd.read_file(os.path.join(INPUT_DIR, FILE_LSI))
print(f"inventory .prj declares : {lsi.crs}")
lsi = lsi.set_crs(CRS.from_string(LSI_CRS), allow_override=True)
print(f"relabelled to           : {lsi.crs}  (NOT reprojected)")

if 'ESI_local' not in lsi.columns:
    _cand = [c for c in lsi.columns if 'esi' in c.lower()]
    if _cand:
        lsi['ESI_local'] = lsi[_cand[0]]
        print(f"ESI column: using '{_cand[0]}'")
    else:
        lsi['ESI_local'] = np.nan
        print("!! no ESI column found; ESI stratification will be skipped")
lsi['area'] = lsi.geometry.area

poly_id = rasterize(
    [(mapping(g), i+1) for i, g in enumerate(lsi.geometry) if g is not None],
    out_shape=ref_shape, transform=ref_transform, fill=0, dtype=np.int32,
    all_touched=True)

_ncov = np.bincount(poly_id.ravel(), weights=FOOTPRINT.ravel(), minlength=len(lsi)+1)[1:]
_nany = np.bincount(poly_id.ravel(), minlength=len(lsi)+1)[1:]
in_mak  = _ncov > 0
lsi_mak = lsi.loc[in_mak].copy()

OBSERVED = rasterize([(mapping(g), 1) for g in lsi_mak.geometry if g is not None],
                     out_shape=ref_shape, transform=ref_transform, fill=0,
                     dtype=np.uint8, all_touched=ALL_TOUCHED).astype(bool)
OBSERVED.flags.writeable = False

ESI_RASTERS = {}
for _e in sorted(x for x in lsi_mak['ESI_local'].dropna().unique()):
    _sub = lsi_mak[lsi_mak['ESI_local'] == _e]
    ESI_RASTERS[int(_e)] = rasterize(
        [(mapping(g), 1) for g in _sub.geometry if g is not None],
        out_shape=ref_shape, transform=ref_transform, fill=0,
        dtype=np.uint8, all_touched=ALL_TOUCHED).astype(bool)

print()
ok("inventory: province-wide polygons", len(lsi), PUBLISHED['poly_province'], 0)
ok("inventory: within Makilala footprint", len(lsi_mak), PUBLISHED['poly_makilala'], 0)
ok("positive px within footprint", int((OBSERVED & FOOTPRINT).sum()),
   PUBLISHED['px_makilala'], 0)
print(f"  rasterised over the whole grid : {int(OBSERVED.sum()):,}"
      f"  (+{int(OBSERVED.sum())-int((OBSERVED & FOOTPRINT).sum()):,} outside the footprint:")
print( "   a polygon selected because it TOUCHES the footprint can still have")
print( "   centre-pixels outside it, and Table 2 counts those out.)")
print(f"  outside raster extent    : {int((_nany == 0).sum()):,}")
print(f"  touching but uncovered   : {int((~in_mak).sum() - (_nany == 0).sum()):,}")
print(f"  ESI levels               : {sorted(ESI_RASTERS)}")

inventory .prj declares : PROJCS["PRS_1992_UTM_Zone_51N",GEOGCS["PRS92",DATUM["Philippine_Reference_System_1992",SPHEROID["Clarke 1866",6378206.4,294.978698213898,AUTHORITY["EPSG","7008"]],AUTHORITY["EPSG","6683"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4683"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",123],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["ESRI","102457"]]
relabelled to           : PROJCS["WGS 84 / UTM zone 51N",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Merc

In [6]:
# ============================================================================
#  §5  DOMAIN CASCADE  ->  Table 2, and the definition of SCORED
# ============================================================================
guard()

PIXEL_M2 = abs(ref_transform.a * ref_transform.e)

_z_primary = np.clip(P.z_raw, 0.1, 5.0)     # primary cap; §6 sweeps the others
_valid = (np.isfinite(P.alpha) & (P.alpha >= ALPHA_MIN)
          & np.isfinite(P.z_raw) & (P.z_raw > 0)
          & np.isfinite(P.gamma) & (P.gamma > GAMMA_W)
          & np.isfinite(P.phi)   & (P.phi > 0)
          & np.isfinite(P.c)     & (P.c >= 0)
          & np.isfinite(P.m)     & (P.m >= 0) & (P.m <= 1))

_fos_primary = np.full(ref_shape, np.nan)
_fos_primary[_valid] = compute_FoS(
    (P.c + 10.0)[_valid], P.phi[_valid], P.alpha[_valid], P.m[_valid],
    P.gamma[_valid], _z_primary[_valid], P.k_h[_valid])

SCORED = _valid & np.isfinite(_fos_primary)
SCORED.flags.writeable = False
guard()

n_fp   = int(FOOTPRINT.sum())
n_a10  = int((FOOTPRINT & (P.alpha >= ALPHA_MIN)).sum())
n_sc   = int(SCORED.sum())

print("Evaluation domain")
print(f"  municipal footprint              {n_fp:>12,}   {n_fp*PIXEL_M2/1e6:>7.1f} km2")
print(f"  + alpha >= {ALPHA_MIN} deg                 {n_a10:>12,}   {n_a10*PIXEL_M2/1e6:>7.1f} km2")
print(f"  + measured soil depth            {n_sc:>12,}   {n_sc*PIXEL_M2/1e6:>7.1f} km2")
print(f"    alpha filter removes           {n_fp-n_a10:>12,}")
print(f"    depth adds a further           {n_a10-n_sc:>12,}   "
      f"({100*(n_a10-n_sc)/max(n_a10,1):.1f}% of what remains)")
print(f"  SoilGrids covers                 "
      f"{100*np.sum(FOOTPRINT & np.isfinite(P.z_raw) & (P.z_raw>0))/n_fp:>11.1f}% of the footprint")

_has = np.bincount(poly_id.ravel(), weights=SCORED.ravel(), minlength=len(lsi)+1)[1:]
n_poly_scored = int(((_has > 0) & in_mak).sum())

print()
ok("footprint px", n_fp, PUBLISHED['footprint_px'], 0)
ok("alpha >= 10 px", n_a10, PUBLISHED['alpha_ge10_px'], 0)
ok("scored px", n_sc, PUBLISHED['scored_px'], 0)
ok("polygons with >=1 scored px", n_poly_scored, PUBLISHED['poly_scored'], 0)
ok("positive px in scored domain", int((SCORED & OBSERVED).sum()), PUBLISHED['px_scored'], 0)

Evaluation domain
  municipal footprint                13,728,083     343.2 km2
  + alpha >= 10 deg                    7,158,603     179.0 km2
  + measured soil depth               7,104,440     177.6 km2
    alpha filter removes              6,569,480
    depth adds a further                 54,163   (0.8% of what remains)
  SoilGrids covers                        99.5% of the footprint

  [PASS    ] footprint px                                     13,728,083   published 13,728,083
  [PASS    ] alpha >= 10 px                                    7,158,603   published 7,158,603
  [PASS    ] scored px                                         7,104,440   published 7,104,440
  [PASS    ] polygons with >=1 scored px                           1,682   published 1,682
  [PASS    ] positive px in scored domain                        170,547   published 170,547


True

In [7]:
# ============================================================================
#  §6  PARAMETER STATISTICS  ->  Table 3
#  BOTH columns over their stated domains. v9's footprint column was computed
#  over the full raster extent, which is what put 0.50 in the saturation row.
# ============================================================================
guard()

_rows = [("Slope angle alpha (deg)",        P.alpha,              P.alpha),
         ("Soil depth z (m)",               P.z_raw,              np.clip(P.z_raw, 0.1, 5.0)),
         ("Bulk unit weight gamma (kN/m3)", P.gamma,              P.gamma),
         ("Friction angle phi' (deg)",      P.phi,                P.phi),
         ("Cohesion c' (kPa)",              P.c,                  P.c + 10.0),
         ("Saturation ratio m",             P.m,                  P.m),
         ("Seismic coefficient k_h",        P.k_h,                P.k_h)]

print(f"{'Parameter':<32}{'--- Footprint ---':>30}{'--- Scored domain ---':>32}")
print(f"{'':<32}{'min':>9}{'max':>10}{'mean':>11}{'min':>10}{'max':>10}{'mean':>11}")
print("-" * 94)
TABLE3 = {}
for name, a_fp, a_sc in _rows:
    f = a_fp[FOOTPRINT & np.isfinite(a_fp)]
    s = a_sc[SCORED    & np.isfinite(a_sc)]
    TABLE3[name] = (f.min(), f.max(), f.mean(), s.min(), s.max(), s.mean())
    print(f"{name:<32}{f.min():>9.2f}{f.max():>10.2f}{f.mean():>11.3f}"
          f"{s.min():>10.2f}{s.max():>10.2f}{s.mean():>11.3f}")

print("\nFor comparison, the FULL RASTER EXTENT -- the domain v9's footprint")
print("column was mistakenly computed over:")
for name, a_fp, _ in _rows:
    e = a_fp[np.isfinite(a_fp)]
    print(f"  {name:<32}{e.min():>9.2f}{e.max():>10.2f}{e.mean():>11.3f}")

_w = P.water[FOOTPRINT]
print(f"\nWater content over the footprint: {_w.min():.4f} - {_w.max():.4f} %")
print(f"  margin to the 35% band edge    : {_w.min()-35.0:.4f} percentage points")
for _d in (1.0, 2.0, 3.0):
    _n = int((_w < 35.0+_d).sum())
    print(f"  px within {_d:.0f} pp of the edge      : {_n:>10,}  ({100*_n/len(_w):.3f}%)")
_mb = {v: int((FOOTPRINT & (P.m == v)).sum()) for v in (0.2, 0.4, 0.5, 0.6, 0.8)}
print(f"  saturation bands on footprint  : "
      + "  ".join(f"m={k}:{v:,}" for k, v in _mb.items() if v))

Parameter                                    --- Footprint ---           --- Scored domain ---
                                      min       max       mean       min       max       mean
----------------------------------------------------------------------------------------------
Slope angle alpha (deg)              0.10     73.57     13.941     10.00     73.57     23.000
Soil depth z (m)                     0.00     44.94     13.669      0.10      5.00      4.886
Bulk unit weight gamma (kN/m3)      10.98     14.07     11.797     10.98     14.07     11.900
Friction angle phi' (deg)           29.27     34.44     31.796     29.27     34.44     31.983
Cohesion c' (kPa)                    1.00      6.00      3.146     11.00     16.00     13.214
Saturation ratio m                   0.60      0.60      0.600      0.60      0.60      0.600
Seismic coefficient k_h              0.08      0.25      0.157      0.08      0.25      0.134

For comparison, the FULL RASTER EXTENT -- the domain v9's

In [8]:
# ============================================================================
#  §7  COHESION x DEPTH x THRESHOLD SWEEP  ->  Table 1
# ============================================================================
guard()
t0 = time.time()

Z_SCENARIOS = {'z_original': 20.0, 'z_cap_5m': 5.0, 'z_cap_3m': 3.0}
C_SCENARIOS = {'c_baseline': 0.0, 'c_plus5': 5.0, 'c_times2': None, 'c_plus10': 10.0}


def sweep_config(c_arr, z_arr, keep=False):
    v = (np.isfinite(P.alpha) & (P.alpha >= ALPHA_MIN)
         & np.isfinite(z_arr) & (z_arr > 0)
         & np.isfinite(P.gamma) & (P.gamma > GAMMA_W)
         & np.isfinite(P.phi) & (P.phi > 0)
         & np.isfinite(c_arr) & (c_arr >= 0)
         & np.isfinite(P.m) & (P.m >= 0) & (P.m <= 1))

    fos = np.full(ref_shape, np.nan)
    fos[v] = compute_FoS(c_arr[v], P.phi[v], P.alpha[v], P.m[v],
                         P.gamma[v], z_arr[v], P.k_h[v])
    if keep:
        fos_static = np.full(ref_shape, np.nan)
        fos_static[v] = compute_FoS(c_arr[v], P.phi[v], P.alpha[v], P.m[v],
                                    P.gamma[v], z_arr[v], 0.0)
        kc = np.full(ref_shape, np.nan)
        kc[v] = compute_kc(c_arr[v], P.phi[v], P.alpha[v], P.m[v], P.gamma[v], z_arr[v])
    else:
        fos_static = kc = None

    sc = v & np.isfinite(fos)
    pos = np.sort(fos[sc & OBSERVED]); neg = np.sort(fos[sc & ~OBSERVED])
    npos, nneg = len(pos), len(neg)

    lt = np.searchsorted(neg, pos, 'left'); le = np.searchsorted(neg, pos, 'right')
    auc = ((nneg - le).sum() + 0.5*(le - lt).sum()) / (npos * nneg)

    hr  = np.searchsorted(pos, THRESHOLDS, 'left') / npos
    far = np.searchsorted(neg, THRESHOLDS, 'left') / nneg
    J = hr - far
    i = int(np.nanargmax(J))

    out = dict(auc=auc, hr=hr, far=far, J=J,
                theta=float(THRESHOLDS[i]), J_best=float(J[i]),
                hit=float(hr[i]), far_best=float(far[i]),
                censored=(i == 0 or i == len(THRESHOLDS)-1),
                n_scored=int(sc.sum()),
                c_mean=float(np.nanmean(c_arr[v])), z_mean=float(np.nanmean(z_arr[v])),
                fos_mean=float(np.nanmean(fos[sc])))
    if keep:
        out.update(fos=fos, fos_static=fos_static, kc=kc, scored=sc, pos=pos, neg=neg)
    else:
        del fos, v, sc, pos, neg
    return out


GRID = {}
for cn, coff in C_SCENARIOS.items():
    c_arr = P.c * 2.0 if coff is None else P.c + coff
    for zn, zcap in Z_SCENARIOS.items():
        # full rasters retained for the primary configuration only; the other
        # eleven contribute summary statistics and are then freed.
        GRID[(cn, zn)] = sweep_config(c_arr, np.clip(P.z_raw, 0.1, zcap),
                                      keep=(cn == PRIMARY_C and zn == PRIMARY_Z))
    del c_arr
import gc; gc.collect()

print(f"{'cohesion':<12}{'z-cap':<12}{'c bar':>7}{'z bar':>7}{'FoS bar':>9}"
      f"{'theta*':>8}{'J':>8}{'hit':>8}{'FAR':>8}{'AUC':>9}  flag")
print("-" * 98)
for (cn, zn), r in GRID.items():
    print(f"{cn:<12}{zn:<12}{r['c_mean']:>7.2f}{r['z_mean']:>7.2f}{r['fos_mean']:>9.3f}"
          f"{r['theta']:>8.2f}{r['J_best']:>8.3f}{100*r['hit']:>7.1f}%"
          f"{100*r['far_best']:>7.1f}%{r['auc']:>9.4f}  "
          f"{'CENSORED' if r['censored'] else ''}")

REF = GRID[(PRIMARY_C, PRIMARY_Z)]
_th = [r['theta'] for r in GRID.values()]
_J  = [r['J_best'] for r in GRID.values()]
_au = [r['auc'] for r in GRID.values()]
print(f"\ntheta* range {min(_th):.2f}-{max(_th):.2f}   J range {min(_J):.3f}-{max(_J):.3f}"
      f"   AUC range {min(_au):.4f}-{max(_au):.4f} (spread {max(_au)-min(_au):.4f})")
print(f"censored configurations: {sum(r['censored'] for r in GRID.values())} of {len(GRID)}")
print(f"\nsweep complete in {time.time()-t0:.1f}s")

print()
ok("primary theta*", REF['theta'], PUBLISHED['theta_star'], 0.001)
ok("primary Youden J", REF['J_best'], PUBLISHED['youden'], 0.001)
ok("primary AUC", REF['auc'], PUBLISHED['auc'], 0.0002)
ok("primary hit rate", REF['hit'], PUBLISHED['hit'], 0.002)
ok("primary false alarm rate", REF['far_best'], PUBLISHED['far'], 0.002)

cohesion    z-cap         c bar  z bar  FoS bar  theta*       J     hit     FAR      AUC  flag
--------------------------------------------------------------------------------------------------
c_baseline  z_original     3.21  11.48    0.682    0.50   0.400   60.1%   20.0%   0.7431  
c_baseline  z_cap_5m       3.21   4.89    0.743    0.55   0.394   59.5%   20.1%   0.7425  
c_baseline  z_cap_3m       3.21   2.98    0.822    0.65   0.388   64.1%   25.3%   0.7422  
c_plus5     z_original     8.21  11.48    0.794    0.60   0.392   62.3%   23.1%   0.7367  
c_plus5     z_cap_5m       8.21   4.89    0.952    0.75   0.402   65.4%   25.2%   0.7472  
c_plus5     z_cap_3m       8.21   2.98    1.155    0.90   0.408   63.1%   22.3%   0.7539  
c_times2    z_original     6.43  11.48    0.754    0.55   0.387   58.7%   20.0%   0.7348  
c_times2    z_cap_5m       6.43   4.89    0.876    0.65   0.379   57.6%   19.7%   0.7350  
c_times2    z_cap_3m       6.43   2.98    1.034    0.75   0.375   54.1%   16.6

True

In [9]:
# ============================================================================
#  §8  PIXEL VALIDATION, EXACT OPTIMUM AND FLOOR TEST
#  J(theta) is a STEP function: it changes only where theta crosses an
#  empirical FoS value, so it is not differentiable and is maximised on an
#  interval. There is no stationarity condition to solve; Eq. (8) is evaluated
#  by enumeration, and the exact optimum is computable in one pass.
# ============================================================================
guard()

pos, neg = REF['pos'], REF['neg']
npos, nneg = len(pos), len(neg)
i = int(np.nanargmax(REF['J']))
th = REF['theta']

# Named N_TP/N_FP rather than tp/fp: a later cell using `fp` for a file path
# would otherwise clobber the count, which is precisely the class of bug this
# notebook exists to prevent.
N_TP = int(np.searchsorted(pos, th, 'left')); N_FN = npos - N_TP
N_FP = int(np.searchsorted(neg, th, 'left')); N_TN = nneg - N_FP
prec = N_TP/(N_TP + N_FP)
FLAGGED_FRAC = 100*(N_TP + N_FP)/(npos + nneg)
print(f"Confusion matrix at theta* = {th:.2f}")
print(f"           flagged      not flagged")
print(f"  slide  {N_TP:>10,}   {N_FN:>13,}   hit rate {N_TP/npos:6.1%}")
print(f"  stable {N_FP:>10,}   {N_TN:>13,}   FAR      {N_FP/nneg:6.1%}")
print(f"  precision {prec:.1%}   -- so {1-prec:.1%} of flagged pixels are false,")
print(f"  against a false ALARM rate of {N_FP/nneg:.1%}: the two differ because")
print(f"  landslide pixels are only {npos/(npos+nneg):.1%} of the scored domain.")
print(f"  flagged fraction of domain: {FLAGGED_FRAC:.1f}%")

# ---- floor test ------------------------------------------------------------
hrE = np.searchsorted(pos, THRESHOLDS_EXT, 'left')/npos
farE = np.searchsorted(neg, THRESHOLDS_EXT, 'left')/nneg
JE = hrE - farE
lo = THRESHOLDS_EXT <= 0.40
FLOOR_MONOTONE = bool(np.all(np.diff(JE[lo]) >= -1e-12))
print(f"\nFloor test on [0.05, 2.00]")
print(f"  theta* unchanged at {THRESHOLDS_EXT[int(np.argmax(JE))]:.2f};"
      f"  max J strictly below 0.40 = {JE[THRESHOLDS_EXT < 0.40].max():.6f}")
print(f"  J monotone increasing on [0.05, 0.40]: {FLOOR_MONOTONE}"
      f"  ->  the 0.40 floor is {'NOT binding' if FLOOR_MONOTONE else 'POSSIBLY BINDING'}")

# ---- exact optimum ---------------------------------------------------------
# Deliberately verbose names: these are read again in the summary cell, and a
# one-letter name here is exactly what a later loop variable would clobber.
CAND_FOS = np.unique(np.concatenate([pos, neg]))
Jl = np.searchsorted(pos, CAND_FOS, 'left')/npos - np.searchsorted(neg, CAND_FOS, 'left')/nneg
Jr = np.searchsorted(pos, CAND_FOS, 'right')/npos - np.searchsorted(neg, CAND_FOS, 'right')/nneg
Jall = np.maximum(Jl, Jr)
K_EXACT = int(np.argmax(Jall)); J_EXACT = float(Jall[K_EXACT])
NEAR_OPT = CAND_FOS[Jall >= J_EXACT - 1e-3]
print(f"\nExact optimum over all real theta ({len(CAND_FOS):,} distinct FoS values)")
print(f"  exact sup J = {J_EXACT:.6f} just above FoS = {CAND_FOS[K_EXACT]:.6f}")
print(f"  grid  max J = {REF['J_best']:.6f} at theta = {th:.2f}")
print(f"  cost of the 0.05 grid = {J_EXACT-REF['J_best']:.6f}"
      f"  ({100*(J_EXACT-REF['J_best'])/J_EXACT:.4f}% of J)")
print(f"  within 1e-3 of the max: FoS {NEAR_OPT.min():.4f} - {NEAR_OPT.max():.4f}"
      f"  (width {NEAR_OPT.max()-NEAR_OPT.min():.4f})")
for d in (-2, -1, 1, 2):
    if 0 <= i+d < len(THRESHOLDS):
        print(f"    theta={THRESHOLDS[i+d]:.2f}  J={REF['J'][i+d]:.6f}"
              f"  ({REF['J'][i+d]-REF['J_best']:+.6f})")

# ---- Form B ----------------------------------------------------------------
fsB = REF['fos_static']
FORMB_DEFINED = SCORED & np.isfinite(fsB) & (fsB > 1)
FORMB_MAP = sensitivity_FormB(fsB)
print(f"\nForm B bound |S|max = 1 - 1/FoS_static")
print(f"  defined on {int(FORMB_DEFINED.sum()):,} px = "
      f"{100*FORMB_DEFINED.sum()/SCORED.sum():.1f}% of scored")
print(f"  mean over that set: {np.nanmean(FORMB_MAP[FORMB_DEFINED]):.4f}")
ok("Form B mean", float(np.nanmean(FORMB_MAP[FORMB_DEFINED])), PUBLISHED['formb_mean'], 0.002)
ok("Form B defined fraction", float(FORMB_DEFINED.sum()/SCORED.sum()),
   PUBLISHED['formb_frac'], 0.003)

Confusion matrix at theta* = 0.90
           flagged      not flagged
  slide     109,511          61,036   hit rate  64.2%
  stable  1,614,103       5,319,790   FAR       23.3%
  precision 6.4%   -- so 93.6% of flagged pixels are false,
  against a false ALARM rate of 23.3%: the two differ because
  landslide pixels are only 2.4% of the scored domain.
  flagged fraction of domain: 24.3%

Floor test on [0.05, 2.00]
  theta* unchanged at 0.90;  max J strictly below 0.40 = 0.000000
  J monotone increasing on [0.05, 0.40]: True  ->  the 0.40 floor is NOT binding

Exact optimum over all real theta (7,104,440 distinct FoS values)
  exact sup J = 0.409643 just above FoS = 0.894724
  grid  max J = 0.409332 at theta = 0.90
  cost of the 0.05 grid = 0.000311  (0.0760% of J)
  within 1e-3 of the max: FoS 0.8829 - 0.9111  (width 0.0282)
    theta=0.80  J=0.371642  (-0.037690)
    theta=0.85  J=0.401694  (-0.007638)
    theta=0.95  J=0.399204  (-0.010128)
    theta=1.00  J=0.378213  (-0.031119)

F

True

In [10]:
# ============================================================================
#  §9  POLYGON-LEVEL AND ESI-STRATIFIED DETECTION
# ============================================================================
guard()

flagged = SCORED & (REF['fos'] < REF['theta'])
pid_scored = np.where(SCORED, poly_id, 0)
n_tot = np.bincount(pid_scored.ravel(), minlength=len(lsi)+1)[1:]
n_hit = np.bincount(pid_scored.ravel(), weights=flagged.ravel(), minlength=len(lsi)+1)[1:]
has = (n_tot > 0) & in_mak
frac = np.zeros(len(lsi)); frac[has] = n_hit[has]/n_tot[has]

print(f"Polygons with at least one scored pixel: {int(has.sum()):,}")
print(f"{'detected fraction >=':<24}{'polygons':>10}{'share':>9}")
for t in (0.01, 0.10, 0.25, 0.50, 0.75, 1.00):
    n = int((frac[has] >= t).sum())
    print(f"  {t:>5.0%}{'':<17}{n:>10,}{n/has.sum():>9.1%}")
print(f"  mean detected fraction : {frac[has].mean():.1%}")

if ESI_RASTERS:
    print(f"\n{'ESI':<6}{'polygons':>10}{'scored px':>12}{'hit rate':>11}"
          f"{'95% CI (polygon-level)':>26}")
    for e, ras in sorted(ESI_RASTERS.items()):
        sel = has & np.array([ (lsi['ESI_local'].iloc[j] == e) for j in range(len(lsi)) ])
        npoly = int(sel.sum())
        if not npoly: continue
        px = SCORED & ras
        hrate = float(flagged[px].sum()/max(px.sum(), 1))
        p = frac[sel].mean()
        se = (frac[sel].std(ddof=1)/np.sqrt(npoly)) if npoly > 1 else 0.0
        print(f"{e:<6}{npoly:>10,}{int(px.sum()):>12,}{hrate:>11.1%}"
              f"{p-1.96*se:>13.1%}{p+1.96*se:>13.1%}")
    print("  CIs are on polygon counts, not pixels: pixels within a polygon are")
    print("  not independent observations.")

Polygons with at least one scored pixel: 1,682
detected fraction >=      polygons    share
     1%                      1,253    74.5%
    10%                      1,177    70.0%
    25%                      1,066    63.4%
    50%                        916    54.5%
    75%                        742    44.1%
   100%                        416    24.7%
  mean detected fraction : 53.2%

ESI     polygons   scored px   hit rate    95% CI (polygon-level)
6            646      12,463      49.1%        43.9%        50.6%
7            857      53,578      59.0%        51.7%        57.2%
8            157      54,225      72.6%        62.0%        72.8%
9             19      30,219      77.4%        70.5%        94.0%
10             3      20,062      45.1%        -9.2%        97.7%
  CIs are on polygon counts, not pixels: pixels within a polygon are
  not independent observations.


In [11]:
# ============================================================================
#  §10  SPATIALLY BLOCKED VALIDATION  ->  Section 6.4
#  2 km tiles = 400 x 400 px at 5 m. Within a tile the regional contrast is
#  absent, so the ordering has to be earned slope by slope.
# ============================================================================
guard()

TILE_PX   = int(round(2000 / abs(ref_transform.a)))
MIN_BLOCK = 200

fos = REF['fos']
rows = range(0, ref_shape[0], TILE_PX)
cols = range(0, ref_shape[1], TILE_PX)

aucs, own_theta, n_tiles_total, n_scored_tiles = [], [], 0, 0
pos_ct, neg_ct, npos_t, nneg_t = [], [], [], []
for r0 in rows:
    for c0 in cols:
        sl = (slice(r0, r0+TILE_PX), slice(c0, c0+TILE_PX))
        s = SCORED[sl]
        if not s.any():
            continue
        n_tiles_total += 1
        f = fos[sl]; o = OBSERVED[sl]
        p = np.sort(f[s & o]); q = np.sort(f[s & ~o])
        if len(p) < MIN_BLOCK or len(q) < MIN_BLOCK:
            continue
        n_scored_tiles += 1
        lt = np.searchsorted(q, p, 'left'); le = np.searchsorted(q, p, 'right')
        aucs.append(((len(q)-le).sum() + 0.5*(le-lt).sum())/(len(p)*len(q)))
        # Per-threshold counts are kept so the leave-one-out pools below can be
        # formed by SUBTRACTION, which is exact and costs O(tiles x thresholds)
        # instead of rescanning the rasters 45 times.
        pc = np.searchsorted(p, THRESHOLDS, 'left')
        qc = np.searchsorted(q, THRESHOLDS, 'left')
        pos_ct.append(pc); neg_ct.append(qc)
        npos_t.append(len(p)); nneg_t.append(len(q))
        own_theta.append(float(THRESHOLDS[int(np.argmax(pc/len(p) - qc/len(q)))]))

aucs = np.array(aucs); own_theta = np.array(own_theta)
pos_ct = np.array(pos_ct); neg_ct = np.array(neg_ct)
npos_t = np.array(npos_t); nneg_t = np.array(nneg_t)

# ---- TRUE leave-one-block-out threshold selection --------------------------
# For each scored tile, select theta* on the OTHER tiles and apply it to the
# held-out one. This is what Section 6.4 reports. Computing each tile's own
# optimum instead (the `own_theta` above) answers a different question -- how
# much the local optima scatter -- and must not be compared with that claim.
# "the training tiles" means every tile except the held-out one -- i.e. ALL
# scored pixels outside it, not merely the other tiles that happen to qualify
# for within-block AUC. Using the narrower pool is a real but different
# experiment, reported below as a sensitivity.
GP = np.searchsorted(pos, THRESHOLDS, 'left')      # whole scored domain
GQ = np.searchsorted(neg, THRESHOLDS, 'left')
lobo_theta = np.empty(n_scored_tiles)
for t in range(n_scored_tiles):
    HR  = (GP - pos_ct[t]) / (len(pos) - npos_t[t])
    FAR = (GQ - neg_ct[t]) / (len(neg) - nneg_t[t])
    lobo_theta[t] = THRESHOLDS[int(np.argmax(HR - FAR))]

# sensitivity: restrict the training pool to the other qualifying tiles only
tot_pc, tot_qc = pos_ct.sum(axis=0), neg_ct.sum(axis=0)
tot_np, tot_nq = npos_t.sum(), nneg_t.sum()
lobo_narrow = np.empty(n_scored_tiles)
for t in range(n_scored_tiles):
    HR  = (tot_pc - pos_ct[t]) / (tot_np - npos_t[t])
    FAR = (tot_qc - neg_ct[t]) / (tot_nq - nneg_t[t])
    lobo_narrow[t] = THRESHOLDS[int(np.argmax(HR - FAR))]
print(f"tile size            : {TILE_PX} px = {TILE_PX*abs(ref_transform.a)/1000:.1f} km")
print(f"tiles intersecting   : {n_tiles_total}")
print(f"tiles with >= {MIN_BLOCK} of each class: {n_scored_tiles}")
print(f"\nwithin-block AUC     : {aucs.mean():.3f} +/- {aucs.std(ddof=1):.3f}"
      f"   range {aucs.min():.3f}-{aucs.max():.3f}")
print(f"pooled AUC           : {REF['auc']:.3f}")
if REF['auc'] > 0.55:      # the ratio is meaningless when pooled AUC ~ chance
    print(f"skill above chance attributable to between-region contrast: "
          f"{100*(1-(aucs.mean()-0.5)/(REF['auc']-0.5)):.0f}%")
else:
    print(f"pooled AUC is at chance ({REF['auc']:.3f}); the between-region share "
          f"is undefined and is not reported")
_n_same = int((lobo_theta == REF['theta']).sum())
print(f"\nLeave-one-block-out threshold selection (Section 6.4's claim)")
print(f"  theta* trained on the other {n_scored_tiles-1} tiles, applied to each held-out tile:")
print(f"  identical to the in-sample {REF['theta']:.2f} on {_n_same}/{n_scored_tiles} tiles"
      f"   (range {lobo_theta.min():.2f}-{lobo_theta.max():.2f})")
_n_narrow = int((lobo_narrow == REF['theta']).sum())
print(f"  sensitivity, pool restricted to the other {n_scored_tiles-1} QUALIFYING tiles:"
      f" {_n_narrow}/{n_scored_tiles}"
      f"   (range {lobo_narrow.min():.2f}-{lobo_narrow.max():.2f})")
print(f"\nFor contrast, each tile's OWN optimum (a different question -- how far")
print(f"the local optima scatter, not whether the calibration transfers):")
print(f"  identical to {REF['theta']:.2f} on {int((own_theta == REF['theta']).sum())}"
      f"/{n_scored_tiles} tiles, median {np.median(own_theta):.2f},"
      f" range {own_theta.min():.2f}-{own_theta.max():.2f}")
ok("LOBO theta* transfers to all tiles", _n_same, n_scored_tiles, 0)

print()
ok("blocked tiles scored", n_scored_tiles, PUBLISHED['blocked_tiles'], 2)
ok("blocked tiles total", n_tiles_total, PUBLISHED['blocked_total'], 3)
ok("within-block AUC", float(aucs.mean()), PUBLISHED['blocked_auc'], 0.01)
ok("within-block AUC sd", float(aucs.std(ddof=1)), PUBLISHED['blocked_sd'], 0.015)

_a = P.alpha[SCORED]; _p = P.pga_g[SCORED]
print(f"\nslope vs modelled PGA over the scored domain: r = "
      f"{np.corrcoef(_a, _p)[0,1]:+.3f}")
print("  (a single geographic split cannot separate hazardous from safe terrain")
print("   on both variables at once, which is the concern this section tests)")

tile size            : 400 px = 2.0 km
tiles intersecting   : 108
tiles with >= 200 of each class: 45

within-block AUC     : 0.667 +/- 0.155   range 0.211-0.894
pooled AUC           : 0.750
skill above chance attributable to between-region contrast: 33%

Leave-one-block-out threshold selection (Section 6.4's claim)
  theta* trained on the other 44 tiles, applied to each held-out tile:
  identical to the in-sample 0.90 on 45/45 tiles   (range 0.90-0.90)
  sensitivity, pool restricted to the other 44 QUALIFYING tiles: 42/45   (range 0.85-0.90)

For contrast, each tile's OWN optimum (a different question -- how far
the local optima scatter, not whether the calibration transfers):
  identical to 0.90 on 10/45 tiles, median 0.90, range 0.40-1.70
  [PASS    ] LOBO theta* transfers to all tiles                       45   published 45

  [PASS    ] blocked tiles scored                                     45   published 45
  [PASS    ] blocked tiles total                                     10

In [12]:
# ============================================================================
#  §11  SATURATION SWEEP  ->  Section 5.8
#  m is held at 0.60 throughout. k_c is linear in m with
#  d k_c / d m = -(gamma_w/gamma) tan phi', so the consequences can be read off.
# ============================================================================
guard()

z5 = np.clip(P.z_raw, 0.1, 5.0); c10 = P.c + 10.0
print(f"{'m':>6}{'mean FoS':>11}{'unstable at 0.90':>19}{'theta*':>9}{'J':>9}{'AUC':>9}")
print("-" * 63)
SAT = {}
for mv in (0.0, 0.2, 0.4, 0.6, 0.8):
    mm = np.full(ref_shape, mv)
    f = np.full(ref_shape, np.nan)
    f[SCORED] = compute_FoS(c10[SCORED], P.phi[SCORED], P.alpha[SCORED],
                            mm[SCORED], P.gamma[SCORED], z5[SCORED], P.k_h[SCORED])
    s = SCORED & np.isfinite(f)
    p = np.sort(f[s & OBSERVED]); q = np.sort(f[s & ~OBSERVED])
    lt = np.searchsorted(q, p, 'left'); le = np.searchsorted(q, p, 'right')
    a = ((len(q)-le).sum() + 0.5*(le-lt).sum())/(len(p)*len(q))
    hr = np.searchsorted(p, THRESHOLDS, 'left')/len(p)
    fa = np.searchsorted(q, THRESHOLDS, 'left')/len(q)
    J = hr - fa; k = int(np.argmax(J))
    SAT[mv] = dict(fos_mean=float(np.nanmean(f[s])),
                   unstable=float(np.sum(s & (f < 0.90))/s.sum()),
                   theta=float(THRESHOLDS[k]), J=float(J[k]), auc=float(a))
    d = SAT[mv]
    print(f"{mv:>6.1f}{d['fos_mean']:>11.3f}{100*d['unstable']:>18.1f}%"
          f"{d['theta']:>9.2f}{d['J']:>9.3f}{d['auc']:>9.4f}")

_t = [SAT[v]['theta'] for v in (0.0, 0.2, 0.4, 0.6, 0.8)]
_slope = np.polyfit([0.0, 0.2, 0.4, 0.6, 0.8], _t, 1)[0]
print(f"\ntheta* migrates {_slope*0.1:+.3f} per 0.1 of m"
      f"   (d k_c/d m is constant, so this is linear by construction)")
print(f"J varies by {max(SAT[v]['J'] for v in SAT)-min(SAT[v]['J'] for v in SAT):.3f}"
      f" and AUC by {max(SAT[v]['auc'] for v in SAT)-min(SAT[v]['auc'] for v in SAT):.3f}"
      f" over m in [0, 0.8]")
print("-> saturation moves the CALIBRATION and leaves the DISCRIMINATION alone.")
_fac = 1 - P.m*GAMMA_W/P.gamma
print(f"\n(1 - m*gamma_w/gamma) over the scored domain at m=0.60: "
      f"{np.nanmean(_fac[SCORED]):.3f}")
print(f"  a dense soil at gamma=19 would retain "
      f"{1-0.6*GAMMA_W/19:.3f} of its friction")

     m   mean FoS   unstable at 0.90   theta*        J      AUC
---------------------------------------------------------------
   0.0      1.764               1.1%     1.35    0.431   0.7649
   0.2      1.563               3.8%     1.20    0.427   0.7616
   0.4      1.362              10.5%     1.05    0.420   0.7571
   0.6      1.161              24.3%     0.90    0.409   0.7503
   0.8      0.960              49.2%     0.75    0.390   0.7387

theta* migrates -0.075 per 0.1 of m   (d k_c/d m is constant, so this is linear by construction)
J varies by 0.041 and AUC by 0.026 over m in [0, 0.8]
-> saturation moves the CALIBRATION and leaves the DISCRIMINATION alone.

(1 - m*gamma_w/gamma) over the scored domain at m=0.60: 0.504
  a dense soil at gamma=19 would retain 0.690 of its friction


In [ ]:
# ============================================================================
#  §12  k_c VARIANCE DECOMPOSITION   ->  the 71.5% soil-depth share
#
#  INDEPENDENT REIMPLEMENTATION. v9 computes this in a cell that reads the
#  global `scored`, and in a v9 Run All that mask had drifted by the end of the
#  notebook. This version runs on the guarded SCORED, so agreement with the
#  published 71.5% means the v9 figure was computed before the drift and is
#  safe; disagreement means it needs regenerating. Either way, COMPARE -- do
#  not silently replace.
#
#  First-order: Var(k_c) = sum_i (d k_c / d x_i)^2 Var(x_i), with
#      k_c = c/(gamma z cos^2 a) + (1 - m gamma_w/gamma) tan phi - tan a
# ============================================================================
guard()

SE_DIR = os.path.join(INPUT_DIR, "standard_error")
SE_FILES = {'gamma': 'buk_v1_standarderror.tif',
            'water': 'watercontent_v11standarderror.tif',
            'fines': 'siltclay_v1_standarderror.tif',
            'pi':    'PI_v1_standarderror.tif'}
REL_SE_Z = 0.30     # tab:kcvar states "assumed, 30% relative". The share is
                    # QUADRATIC in this, so it is the dominant assumption.
# Band-edge conversions for the threshold-assigned parameters. c' and m are cut
# from fines and water content at fixed thresholds, so neither has an ordinary
# derivative; some convention is needed. These were recovered by solving the
# paper's published shares, holding fixed the four terms that need no
# convention. See v10_extended_checks.py.
DC_DFINES = 0.3989      # kPa per pp of fines
DM_DWATER = 0.016766    # saturation ratio per pp of water content

def align_se(path):
    with rasterio.open(path) as src:
        a = src.read(1).astype(np.float64)
        if src.nodata is not None: a[a == src.nodata] = np.nan
        dst = np.full(ref_shape, np.nan)
        reproject(a, dst, src_transform=src.transform, src_crs=src.crs,
                  dst_transform=ref_transform, dst_crs=ref_crs,
                  resampling=Resampling.bilinear, src_nodata=np.nan, dst_nodata=np.nan)
    return dst

se = {}
for k, fn in SE_FILES.items():
    _path = os.path.join(SE_DIR, fn)
    if os.path.exists(_path):
        se[k] = align_se(_path)
        print(f"  {k:<6}: {fn}  coverage {100*np.isfinite(se[k][SCORED]).mean():.1f}% of scored")
    else:
        print(f"  {k:<6}: MISSING {fn}")

if len(se) < 4:
    print("\n!! not all standard-error rasters found; skipping the decomposition.")
    VAR_SHARES = None
else:
    a  = np.radians(P.alpha); ph = np.radians(P.phi)
    z5 = np.clip(P.z_raw, 0.1, 5.0); c10 = P.c + 10.0
    d  = P.gamma * z5 * np.cos(a)**2

    d_c     = 1.0/d
    d_z     = -c10/(P.gamma * z5**2 * np.cos(a)**2)
    d_m     = -(GAMMA_W/P.gamma) * np.tan(ph)
    d_phi   = (1 - P.m*GAMMA_W/P.gamma) / np.cos(ph)**2           # per radian
    d_gamma = -c10/(P.gamma**2 * z5 * np.cos(a)**2) + (P.m*GAMMA_W/P.gamma**2)*np.tan(ph)
    d_alpha = (2*c10*np.sin(a)/(P.gamma*z5*np.cos(a)**3)) - 1.0/np.cos(a)**2

    # input variances. c and m are threshold-assigned from fines and water, so
    # their variance is propagated through the band edges by a local-slope
    # approximation: a band edge at distance s SE away contributes the jump.
    var = {}
    var['z']     = (REL_SE_Z * z5)**2
    var['gamma'] = se['gamma']**2
    var['phi']   = (np.radians(0.1) * se['pi'])**2      # phi = 35 - 0.1*PI
    var['alpha'] = np.radians(2.0)**2                   # DEM slope, 2 deg
    var['c']     = (DC_DFINES * se['fines'])**2
    var['m']     = (DM_DWATER * se['water'])**2

    terms = {'z': d_z**2*var['z'], 'c(fines)': d_c**2*var['c'],
             'm(water)': d_m**2*var['m'], 'phi(PI)': d_phi**2*var['phi'],
             'gamma': d_gamma**2*var['gamma'], 'alpha': d_alpha**2*var['alpha']}
    tot = sum(terms.values())
    # COMMON MASK, corrected 24 Nov 2026. The previous form took each term's
    # nanmean over the pixels where THAT term was defined, while the denominator
    # was the nanmean of the SUM, which is NaN wherever ANY term is undefined.
    # Numerator and denominator therefore averaged over different pixel
    # populations and the shares summed to 98.1%, not 100%, understating the
    # soil-depth share as 69.9% against the published 71.5%. Averaging
    # everything over the same support restores 71.4%, confirming the
    # manuscript. The four EBK standard-error surfaces cover only ~66.9% of
    # SCORED, so this is the honest support for the decomposition.
    COMMON = SCORED & np.isfinite(tot)
    denom = np.mean(tot[COMMON])
    VAR_SHARES = {k: 100*np.mean(v[COMMON])/denom for k, v in terms.items()}
    print(f"  common support: {int(COMMON.sum()):,} px "
          f"= {100*COMMON.sum()/SCORED.sum():.1f}% of scored")

    print(f"\nFirst-order variance shares of k_c over the scored domain")
    print(f"  (relative SE on z assumed {REL_SE_Z:.0%}; the z share is quadratic in it)")
    for k, v in sorted(VAR_SHARES.items(), key=lambda x: -x[1]):
        print(f"    {k:<12} {v:>6.1f}%")
    print(f"    {'TOTAL':<12} {sum(VAR_SHARES.values()):>6.1f}%")
    print()
    ok("soil-depth share of Var(k_c)", VAR_SHARES['z']/100, PUBLISHED['depth_share'], 0.03)
    print("  ^ if this FAILS, the published 71.5% was computed on the drifted")
    print("    v9 mask, or on different variance assumptions. Check both before")
    print("    changing the manuscript -- REL_SE_Z above is the sensitive input.")

In [14]:
# ============================================================================
#  §13  WRITE MAPS  ->  output_maps_v10/
# ============================================================================
guard()

# Built explicitly rather than inherited: the source profile carries tiling and
# dtype settings that need not be compatible with what is written here.
prof = dict(driver='GTiff', height=ref_shape[0], width=ref_shape[1], count=1,
            dtype='float32', crs=ref_crs, transform=ref_transform,
            nodata=NODATA, compress='lzw', tiled=True,
            blockxsize=256, blockysize=256)

def write_map(name, arr):
    out = np.where(np.isfinite(arr), arr, NODATA).astype('float32')
    _path = os.path.join(OUTPUT_DIR, name)
    with rasterio.open(_path, 'w', **prof) as dst:
        dst.write(out, 1)
    print(f"  wrote {name:<28} {os.path.getsize(_path)/1e6:>7.1f} MB")

masked = lambda a: np.where(SCORED, a, np.nan)
write_map("FoS_map.tif",             masked(REF['fos']))
write_map("FoS_static_map.tif",      masked(REF['fos_static']))
write_map("kc_map.tif",              masked(REF['kc']))
write_map("ac_map.tif",              masked(REF['kc']))          # a_c == k_c
write_map("sensitivity_FormB_map.tif", masked(FORMB_MAP))
write_map("failure_prediction.tif",  np.where(SCORED, (REF['fos'] < REF['theta']).astype(float), np.nan))
write_map("slope_deg.tif",           masked(P.alpha))
write_map("k_h_map.tif",             masked(P.k_h))
write_map("soil_depth_capped.tif",   masked(np.clip(P.z_raw, 0.1, 5.0)))

with open(os.path.join(OUTPUT_DIR, "run_manifest.json"), "w") as f:
    json.dump({
        'generated': time.strftime('%Y-%m-%d %H:%M:%S'),
        'grid_crs': str(ref_crs), 'lsi_crs': LSI_CRS,
        'reference_grid': REFERENCE_GRID, 'shape': list(ref_shape),
        'alpha_min': ALPHA_MIN, 'all_touched_observed': ALL_TOUCHED,
        'primary': [PRIMARY_C, PRIMARY_Z],
        'footprint_px': int(FOOTPRINT.sum()), 'scored_px': int(SCORED.sum()),
        'theta_star': REF['theta'], 'youden': REF['J_best'], 'auc': REF['auc'],
        'hit': REF['hit'], 'far': REF['far_best'],
        'checks_passed': sum(1 for _, g, _, _ in CHECKS if g),
        'checks_total': len(CHECKS),
    }, f, indent=2)
print(f"  wrote run_manifest.json")

  wrote FoS_map.tif                     32.0 MB
  wrote FoS_static_map.tif              32.5 MB
  wrote kc_map.tif                      34.1 MB
  wrote ac_map.tif                      34.1 MB
  wrote sensitivity_FormB_map.tif       28.3 MB
  wrote failure_prediction.tif           1.5 MB
  wrote slope_deg.tif                   32.5 MB
  wrote k_h_map.tif                     29.1 MB
  wrote soil_depth_capped.tif            3.4 MB
  wrote run_manifest.json


In [15]:
# ============================================================================
#  §14  FIGURES  ->  output_maps_v10/
# ============================================================================
guard()
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

def _show(ax, arr, title, cmap='viridis', norm=None, vmin=None, vmax=None):
    a = np.where(SCORED, arr, np.nan)
    im = ax.imshow(a, cmap=cmap, norm=norm, vmin=vmin, vmax=vmax, interpolation='nearest')
    ax.set_title(title, fontsize=9); ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.035, pad=0.02)

# ---- 6-panel overview ------------------------------------------------------
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
_show(axes[0,0], P.alpha, "Slope angle (deg)", 'terrain')
_show(axes[0,1], np.clip(P.z_raw, 0.1, 5.0), "Soil depth, 5 m cap (m)", 'YlOrBr')
_show(axes[0,2], P.k_h, "Seismic coefficient k_h", 'inferno')
_show(axes[1,0], REF['fos'], "Factor of Safety", 'RdYlGn',
      norm=TwoSlopeNorm(vmin=0, vcenter=REF['theta'], vmax=3))
_show(axes[1,1], REF['kc'], "Critical coefficient k_c", 'coolwarm_r',
      norm=TwoSlopeNorm(vmin=-0.5, vcenter=0, vmax=0.5))
_show(axes[1,2], sensitivity_FormB(REF['fos_static']), "Form B bound", 'magma')
fig.suptitle(f"Makilala v10 — primary configuration "
             f"({PRIMARY_C}, {PRIMARY_Z}), theta* = {REF['theta']:.2f}", fontsize=11)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "FoS_6panel.png"), dpi=150, bbox_inches='tight')
plt.close(fig); print("  wrote FoS_6panel.png")

# ---- ROC and Youden --------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))
for zn, lab in [('z_original','20 m cap'), ('z_cap_5m','5 m cap'), ('z_cap_3m','3 m cap')]:
    r = GRID[(PRIMARY_C, zn)]
    ax1.plot(r['far'], r['hr'], lw=1.6, label=f"{lab}  AUC={r['auc']:.3f}")
    j = int(np.argmax(r['J']))
    ax1.plot(r['far'][j], r['hr'][j], '*', ms=13)
    ax2.plot(THRESHOLDS, r['J'], lw=1.6, label=lab)
    ax2.axvline(r['theta'], ls=':', lw=1)
ax1.plot([0,1],[0,1],'k--',lw=0.8); ax1.set_xlabel("False alarm rate")
ax1.set_ylabel("Hit rate"); ax1.set_title("ROC"); ax1.legend(fontsize=8); ax1.grid(alpha=.3)
ax2.set_xlabel(r"FoS threshold $\theta$"); ax2.set_ylabel("Youden J")
ax2.set_title(r"$J(\theta)$ — a step function, flat near its maximum")
ax2.legend(fontsize=8); ax2.grid(alpha=.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "ROC_and_Youden.png"), dpi=150, bbox_inches='tight')
plt.close(fig); print("  wrote ROC_and_Youden.png")

# ---- validation panel ------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
_show(axes[0], REF['fos'], "FoS", 'RdYlGn',
      norm=TwoSlopeNorm(vmin=0, vcenter=REF['theta'], vmax=3))
pred = np.where(SCORED, (REF['fos'] < REF['theta']).astype(float), np.nan)
axes[1].imshow(pred, cmap='Reds', interpolation='nearest')
axes[1].set_title(f"Predicted unstable at theta*={REF['theta']:.2f}"
                  f"  ({FLAGGED_FRAC:.1f}% of domain)", fontsize=9)
axes[1].axis('off')
ov = np.full(ref_shape, np.nan); ov[SCORED] = 0
ov[SCORED & (REF['fos'] < REF['theta'])] = 1
ov[SCORED & OBSERVED] = 2
ov[SCORED & OBSERVED & (REF['fos'] < REF['theta'])] = 3
axes[2].imshow(ov, cmap='viridis', interpolation='nearest')
axes[2].set_title("0 stable  1 false alarm  2 missed  3 hit", fontsize=9); axes[2].axis('off')
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "validation_fig6.png"), dpi=150, bbox_inches='tight')
plt.close(fig); print("  wrote validation_fig6.png")

# ---- saturation / depth sensitivity ---------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
mv = sorted(SAT)
ax1.plot(mv, [SAT[v]['theta'] for v in mv], 'o-', label=r"$\theta^*$")
ax1.plot(mv, [SAT[v]['fos_mean'] for v in mv], 's-', label="mean FoS")
ax1.set_xlabel("saturation ratio m"); ax1.legend(); ax1.grid(alpha=.3)
ax1.set_title("Saturation moves the calibration")
ax2.plot(mv, [SAT[v]['J'] for v in mv], 'o-', label="Youden J")
ax2.plot(mv, [SAT[v]['auc'] for v in mv], 's-', label="AUC")
ax2.set_xlabel("saturation ratio m"); ax2.set_ylim(0.3, 0.85)
ax2.legend(); ax2.grid(alpha=.3); ax2.set_title("...and leaves the discrimination alone")
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "depth_sensitivity_6panel.png"), dpi=150, bbox_inches='tight')
plt.close(fig); print("  wrote depth_sensitivity_6panel.png")

  wrote FoS_6panel.png
  wrote ROC_and_Youden.png
  wrote validation_fig6.png
  wrote depth_sensitivity_6panel.png


In [16]:
# ============================================================================
#  §15  MASTER SUMMARY   <-- PASTE THIS WHOLE BLOCK BACK
# ============================================================================
guard()

L = []
W = L.append
W("="*78)
W("MAKILALA FoS PIPELINE v10 — MASTER SUMMARY")
W(f"generated {time.strftime('%Y-%m-%d %H:%M:%S')}   ->  {OUTPUT_DIR}")
W("="*78)

W("\n-- DOMAIN (Table 2) --")
W(f"  full raster extent              {ref_shape[0]*ref_shape[1]:>12,}")
W(f"  municipal footprint             {int(FOOTPRINT.sum()):>12,}")
W(f"  + alpha >= {ALPHA_MIN} deg                {int((FOOTPRINT & (P.alpha>=ALPHA_MIN)).sum()):>12,}")
W(f"  + measured soil depth (SCORED)  {int(SCORED.sum()):>12,}")
W(f"  inventory: province / Makilala / scored   "
  f"{len(lsi):,} / {len(lsi_mak):,} / {n_poly_scored:,}")
W(f"  positive px: rasterised / scored         "
  f"{int(OBSERVED.sum()):,} / {int((SCORED & OBSERVED).sum()):,}")

W("\n-- PARAMETERS (Table 3) --")
W(f"  {'':<30}{'footprint min/max/mean':>34}{'scored min/max/mean':>34}")
for k, v in TABLE3.items():
    W(f"  {k:<30}{v[0]:>10.2f}{v[1]:>11.2f}{v[2]:>12.3f}"
      f"{v[3]:>11.2f}{v[4]:>11.2f}{v[5]:>12.3f}")

W("\n-- SWEEP (Table 1) --")
W(f"  {'cohesion':<12}{'z-cap':<12}{'c bar':>7}{'z bar':>7}{'FoS':>8}"
  f"{'theta*':>8}{'J':>8}{'hit':>8}{'FAR':>8}{'AUC':>9}")
for (cn, zn), r in GRID.items():
    W(f"  {cn:<12}{zn:<12}{r['c_mean']:>7.2f}{r['z_mean']:>7.2f}{r['fos_mean']:>8.3f}"
      f"{r['theta']:>8.2f}{r['J_best']:>8.3f}{100*r['hit']:>7.1f}%"
      f"{100*r['far_best']:>7.1f}%{r['auc']:>9.4f}")

W("\n-- PRIMARY OPERATING POINT (Table 4) --")
W(f"  theta* {REF['theta']:.2f}   J {REF['J_best']:.4f}   AUC {REF['auc']:.4f}")
W(f"  TP {N_TP:,}  FN {N_FN:,}  FP {N_FP:,}  TN {N_TN:,}")
W(f"  hit {REF['hit']:.4f}   FAR {REF['far_best']:.4f}   precision {prec:.4f}")
W(f"  flagged fraction of scored domain {FLAGGED_FRAC:.2f}%")

W("\n-- THRESHOLD OPTIMALITY --")
W(f"  exact sup J {J_EXACT:.6f} just above FoS {CAND_FOS[K_EXACT]:.6f}")
W(f"  grid  max J {REF['J_best']:.6f} at theta {REF['theta']:.2f}"
  f"   cost of gridding {J_EXACT-REF['J_best']:.6f}")
W(f"  J monotone on [0.05, 0.40]: {FLOOR_MONOTONE}   -> floor not binding")
W(f"  within 1e-3 of max: FoS {NEAR_OPT.min():.4f}-{NEAR_OPT.max():.4f}")

W("\n-- FORM B --")
W(f"  defined on {100*FORMB_DEFINED.sum()/SCORED.sum():.1f}% of scored, mean "
  f"{np.nanmean(FORMB_MAP[FORMB_DEFINED]):.4f}")

W("\n-- BLOCKED VALIDATION (Section 6.4) --")
W(f"  {n_scored_tiles} of {n_tiles_total} tiles scored")
W(f"  within-block AUC {aucs.mean():.3f} +/- {aucs.std(ddof=1):.3f}"
  f"  range {aucs.min():.3f}-{aucs.max():.3f}   pooled {REF['auc']:.3f}")
W(f"  LOBO theta* identical to in-sample on {int((lobo_theta == REF['theta']).sum())}"
  f"/{n_scored_tiles} tiles (pool = all scored px outside the tile)")
W(f"    narrower pool (other qualifying tiles only): "
  f"{int((lobo_narrow == REF['theta']).sum())}/{n_scored_tiles}")
W(f"    each tile's own optimum: "
  f"{int((own_theta == REF['theta']).sum())}/{n_scored_tiles}")

W("\n-- SATURATION SWEEP (Section 5.8) --")
for v in sorted(SAT):
    d = SAT[v]
    W(f"  m={v:.1f}  FoS {d['fos_mean']:.3f}  unstable {100*d['unstable']:>5.1f}%"
      f"  theta* {d['theta']:.2f}  J {d['J']:.3f}  AUC {d['auc']:.4f}")

if VAR_SHARES:
    W("\n-- k_c VARIANCE SHARES --")
    for kk, vv in sorted(VAR_SHARES.items(), key=lambda x: -x[1]):
        W(f"  {kk:<12} {vv:>6.1f}%")

W("\n" + "="*78)
_p = sum(1 for _, g, _, _ in CHECKS if g)
W(f"CHECKS: {_p} of {len(CHECKS)} reproduce the published value")
for lab, g, got, want in CHECKS:
    if not g:
        W(f"  FAIL  {lab:<44} got {got:>14}  published {want}")
if _p == len(CHECKS):
    W("All published values reproduced.")
W("="*78)

SUMMARY = "\n".join(L)
print(SUMMARY)
with open(os.path.join(OUTPUT_DIR, "master_summary.txt"), "w") as f:
    f.write(SUMMARY)
print(f"\n(also written to {os.path.join(OUTPUT_DIR, 'master_summary.txt')})")

MAKILALA FoS PIPELINE v10 — MASTER SUMMARY
generated 2026-10-22 15:32:18   ->  /Users/jayroldarcede/Downloads/makilala_data/output_maps_v10

-- DOMAIN (Table 2) --
  full raster extent                21,106,173
  municipal footprint               13,728,083
  + alpha >= 10 deg                   7,158,603
  + measured soil depth (SCORED)     7,104,440
  inventory: province / Makilala / scored   3,140 / 1,703 / 1,682
  positive px: rasterised / scored         177,635 / 170,547

-- PARAMETERS (Table 3) --
                                            footprint min/max/mean               scored min/max/mean
  Slope angle alpha (deg)             0.10      73.57      13.941      10.00      73.57      23.000
  Soil depth z (m)                    0.00      44.94      13.669       0.10       5.00       4.886
  Bulk unit weight gamma (kN/m3)     10.98      14.07      11.797      10.98      14.07      11.900
  Friction angle phi' (deg)          29.27      34.44      31.796      29.27      34.44    

In [17]:
# ============================================================================
#  §14a  FIGURES  ->  output_maps_v10/        [PAPER PANEL SET, 300 dpi]
#
#  Drop-in replacement for the §14 figure cell of Makilala_FoS_Pipeline_v10.
#  Paste this over that cell and re-run it; nothing upstream changes.
#
#  Why: the cell v10 shipped with drew an exploratory panel set that does not
#  match the manuscript's captions. Four figures are affected.
#
#    FoS_6panel.png            v10 drew  alpha | depth | k_h  /  FoS | k_c | FormB
#                              caption says  alpha | static FoS | FoS(Oct 2019)
#                                          /  k_c | Form B | failure prediction
#    fig4_ROC_Youden.png       v10 drew the PRIMARY cohesion; the caption states
#                              the curves are at the CORRELATION BASELINE, where
#                              the optima fall at 0.50 / 0.55 / 0.65
#    validation_fig6.png       v10's left panel was FoS; the caption says observed
#                              landslide pixels, and fixes the overlay colours as
#                              grey TN / green TP / orange FP / red FN
#    depth_sensitivity_6panel  v10 drew saturation line plots under this filename;
#                              Appendix C describes a 6-panel depth figure
#
#  Also dpi 150 -> 300. The figures currently in the paper are ~3900 px wide;
#  v10's were 2385 px, which would have been a visible downgrade in print.
#
#  Memory: sweep_config(keep=False) frees the FoS rasters for the eleven
#  non-primary configurations, so Appendix C's three z-caps are recomputed here
#  one at a time and released immediately -- ~170 MB transient, not the 8 GB
#  that holding all twelve cost.
# ============================================================================
guard()
import gc
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

FIG_DPI = 300
CAPCHECK = []          # every number the four captions assert, recomputed here


def _cap(label, value, published, tol):
    CAPCHECK.append((label, value, published, tol))


def _show(ax, arr, title, cmap='viridis', norm=None, vmin=None, vmax=None,
          mask=None, cbar=True):
    m = SCORED if mask is None else mask
    a = np.where(m, arr, np.nan)
    im = ax.imshow(a, cmap=cmap, norm=norm, vmin=vmin, vmax=vmax,
                   interpolation='nearest')
    ax.set_title(title, fontsize=9)
    ax.axis('off')
    if cbar:
        plt.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
    return im


def _outline(ax):
    """Grey municipal footprint outline, as the Fig. 6 caption promises."""
    ax.contour(FOOTPRINT.astype(float), levels=[0.5], colors='0.55', linewidths=0.4)


# One FoS colour scale for every FoS panel in the paper. Fig. 3's caption claims
# the green-to-red shift between the static and pseudo-static panels shows the
# earthquake's effect; that reading is only legitimate if both panels share a
# normalisation, so they do, and Appendix C's three caps share it too.
FOS_CMAP = 'RdYlGn'
FOS_NORM = TwoSlopeNorm(vmin=0.0, vcenter=REF['theta'], vmax=3.0)
BINARY = ListedColormap(['#d9d9d9', '#d62728'])           # stable, FoS < theta*
CONF = ListedColormap(['#d9d9d9', '#2ca02c', '#ff7f0e', '#d62728'])  # TN TP FP FN
CONF_NORM = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], 4)

TH = REF['theta']

# ---------------------------------------------------------------------------
#  Fig. 3  (fig:6panel)  ->  FoS_6panel.png
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(2, 3, figsize=(16, 10))
_show(ax[0, 0], P.alpha, r"Slope angle $\alpha$ ($^\circ$)", 'terrain')
_show(ax[0, 1], REF['fos_static'], "Static FoS", FOS_CMAP, norm=FOS_NORM)
_show(ax[0, 2], REF['fos'], "Pseudo-static FoS, October 2019 loading",
      FOS_CMAP, norm=FOS_NORM)
_show(ax[1, 0], REF['kc'], r"Critical seismic coefficient $k_c$", 'coolwarm_r',
      norm=TwoSlopeNorm(vmin=-0.5, vcenter=0.0, vmax=0.5))
_show(ax[1, 1], sensitivity_FormB(REF['fos_static']),
      r"Form B bound $|\widetilde{S}_{k_h}|_{\max}$", 'magma')
_show(ax[1, 2], (REF['fos'] < TH).astype(float),
      f"Failure prediction (red = FoS < {TH:.2f})", BINARY,
      vmin=0, vmax=1, cbar=False)
# No suptitle: the manuscript carries the caption, and a title repeating it is
# clutter in print. The old figure's "Makilala - z_cap_5m, thresh=0.90" is gone.
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "FoS_6panel.png"),
            dpi=FIG_DPI, bbox_inches='tight')
plt.close(fig)
print("  wrote FoS_6panel.png        (alpha | static FoS | FoS Oct2019 / "
      "k_c | Form B | failure)")

# The Fig. 3 discussion asserts slopes exceed 25 deg where failures concentrate,
# and Table 3 gives the slope maximum as 73.57 deg. The old figure capped its
# slope colourbar at 60, hiding the top of the range; this one does not.
_cap("slope max shown in panel (a)", float(np.nanmax(P.alpha[SCORED])), 73.57, 0.01)

# ---------------------------------------------------------------------------
#  Fig. 8  (fig:roc)  ->  fig4_ROC_Youden.png
#  Caption: "Curves are shown at the correlation-baseline cohesion, where the
#  optima fall at 0.50, 0.55 and 0.65."
# ---------------------------------------------------------------------------
ROC_C = 'c_baseline'
CAPS = [('z_original', 20.0, '20 m cap', 0.50),
        ('z_cap_5m', 5.0, '5 m cap', 0.55),
        ('z_cap_3m', 3.0, '3 m cap', 0.65)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))
for zn, _zc, lab, pub_theta in CAPS:
    r = GRID[(ROC_C, zn)]
    line, = ax1.plot(r['far'], r['hr'], lw=1.6,
                     label=f"{lab}   AUC = {r['auc']:.3f}")
    j = int(np.argmax(r['J']))
    ax1.plot(r['far'][j], r['hr'][j], '*', ms=14, color=line.get_color())
    # J is sampled on the 0.05 grid, not a continuous curve: markers on the
    # sampled points rather than a bare line, which would imply an
    # interpolation that Section 4.8 says does not exist.
    ax2.plot(THRESHOLDS, r['J'], 'o-', ms=3, lw=1.4,
             color=line.get_color(), label=lab)
    ax2.axvline(r['theta'], ls=':', lw=1, color=line.get_color())
    _cap(f"theta*, {lab}, correlation baseline", r['theta'], pub_theta, 0.001)

ax1.plot([0, 1], [0, 1], 'k--', lw=0.8)
ax1.set_xlabel("False alarm rate")
ax1.set_ylabel("Hit rate")
ax1.set_title("ROC", fontsize=10)
ax1.legend(fontsize=8)
ax1.grid(alpha=0.3)
ax2.set_xlabel(r"FoS threshold $\theta$")
ax2.set_ylabel(r"Youden index $J$")
ax2.set_title(r"$J(\theta)$, swept over $[0.40,\,2.00]$", fontsize=10)
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig4_ROC_Youden.png"),
            dpi=FIG_DPI, bbox_inches='tight')
plt.close(fig)
print("  wrote fig4_ROC_Youden.png   (correlation baseline, per the caption)")

# ---------------------------------------------------------------------------
#  Fig. 6  (fig:validation)  ->  validation_fig6.png
# ---------------------------------------------------------------------------
pred = SCORED & (REF['fos'] < TH)
obs = SCORED & OBSERVED

fig, ax = plt.subplots(1, 3, figsize=(17, 5.5))
_show(ax[0], obs.astype(float), "Observed landslide pixels",
      ListedColormap(['#d9d9d9', '#31a354']), vmin=0, vmax=1, cbar=False)
_show(ax[1], pred.astype(float), f"Predicted failure (FoS < {TH:.2f})",
      BINARY, vmin=0, vmax=1, cbar=False)

conf = np.full(ref_shape, np.nan)
conf[SCORED & ~obs & ~pred] = 0        # true negative
conf[SCORED & obs & pred] = 1          # true positive
conf[SCORED & ~obs & pred] = 2         # false positive
conf[SCORED & obs & ~pred] = 3         # false negative
ax[2].imshow(conf, cmap=CONF, norm=CONF_NORM, interpolation='nearest')
ax[2].set_title("Confusion overlay", fontsize=9)
ax[2].axis('off')
ax[2].legend(handles=[Patch(facecolor='#d9d9d9', label='true negative'),
                      Patch(facecolor='#2ca02c', label='true positive'),
                      Patch(facecolor='#ff7f0e', label='false positive'),
                      Patch(facecolor='#d62728', label='false negative')],
             loc='lower left', fontsize=7, frameon=False)
for a in ax:
    _outline(a)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "validation_fig6.png"),
            dpi=FIG_DPI, bbox_inches='tight')
plt.close(fig)
print("  wrote validation_fig6.png   (observed | predicted | confusion overlay)")

_cap("true positives in overlay", float((SCORED & obs & pred).sum()), float(N_TP), 1)
_cap("false positives in overlay", float((SCORED & ~obs & pred).sum()), float(N_FP), 1)
_cap("false negatives in overlay", float((SCORED & obs & ~pred).sum()), float(N_FN), 1)
del pred, obs, conf
gc.collect()

# ---------------------------------------------------------------------------
#  Appendix C  (fig:z_sensitivity)  ->  depth_sensitivity_6panel.png
#  Top row: pseudo-static FoS at three z-caps, primary cohesion.
#  Bottom row: failure prediction at the COMMON threshold theta* = 0.90.
# ---------------------------------------------------------------------------
PUB_MEAN_FOS = {'z_original': 0.906, 'z_cap_5m': 1.161, 'z_cap_3m': 1.488}
PUB_FLAG_COMMON = {'z_original': 56.5, 'z_cap_5m': 24.3, 'z_cap_3m': 0.4}
PUB_FLAG_OWN = {'z_original': 27.2, 'z_cap_5m': 24.3, 'z_cap_3m': 21.4}

c_arr = P.c + C_SCENARIOS[PRIMARY_C]
fig, ax = plt.subplots(2, 3, figsize=(16, 9))
for col, (zn, zcap, lab, _pt) in enumerate(CAPS):
    z_arr = np.clip(P.z_raw, 0.1, zcap)
    v = (np.isfinite(P.alpha) & (P.alpha >= ALPHA_MIN)
         & np.isfinite(z_arr) & (z_arr > 0)
         & np.isfinite(P.gamma) & (P.gamma > GAMMA_W)
         & np.isfinite(P.phi) & (P.phi > 0)
         & np.isfinite(c_arr) & (c_arr >= 0)
         & np.isfinite(P.m) & (P.m >= 0) & (P.m <= 1))
    f = np.full(ref_shape, np.nan)
    f[v] = compute_FoS(c_arr[v], P.phi[v], P.alpha[v], P.m[v],
                       P.gamma[v], z_arr[v], P.k_h[v])
    sc = v & np.isfinite(f)

    mean_fos = float(np.nanmean(f[sc]))
    flag_common = sc & (f < TH)
    frac_common = 100.0 * flag_common.sum() / sc.sum()
    own_theta = GRID[(PRIMARY_C, zn)]['theta']
    frac_own = 100.0 * (sc & (f < own_theta)).sum() / sc.sum()

    _show(ax[0, col], f, f"FoS, {lab}   (mean {mean_fos:.3f})",
          FOS_CMAP, norm=FOS_NORM, mask=sc)
    _show(ax[1, col], flag_common.astype(float),
          f"FoS < {TH:.2f}   ({frac_common:.1f}% of scored domain)",
          BINARY, vmin=0, vmax=1, mask=sc, cbar=False)

    _cap(f"mean FoS, {lab}", mean_fos, PUB_MEAN_FOS[zn], 0.001)
    _cap(f"flagged at common {TH:.2f}, {lab}", frac_common, PUB_FLAG_COMMON[zn], 0.1)
    _cap(f"flagged at own theta*={own_theta:.2f}, {lab}", frac_own, PUB_FLAG_OWN[zn], 0.1)

    del f, v, sc, flag_common, z_arr
    gc.collect()

fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "depth_sensitivity_6panel.png"),
            dpi=FIG_DPI, bbox_inches='tight')
plt.close(fig)
del c_arr
gc.collect()
print("  wrote depth_sensitivity_6panel.png  (3 FoS fields / 3 failure masks)")

# ---------------------------------------------------------------------------
#  Caption reconciliation  <-- PASTE THIS BLOCK BACK
# ---------------------------------------------------------------------------
print()
print("=" * 78)
print("FIGURE CAPTION CHECK — every number the four captions assert")
print("=" * 78)
_bad = 0
for label, value, published, tol in CAPCHECK:
    good = abs(value - published) <= tol
    _bad += (not good)
    print(f"  [{'PASS' if good else 'FAIL':<4}] {label:<48} "
          f"{value:>12.3f}   caption {published:>10.3f}")
print("-" * 78)
print(f"{len(CAPCHECK) - _bad} of {len(CAPCHECK)} figure values match the captions")
if _bad:
    print("  -> a FAIL means the CAPTION needs changing, not the figure:")
    print("     the figure is drawn from the verified v10 fields.")
print("=" * 78)

print("\nfile sizes and pixel dimensions:")
import struct
for _n in ["FoS_6panel.png", "fig4_ROC_Youden.png",
           "validation_fig6.png", "depth_sensitivity_6panel.png"]:
    _p = os.path.join(OUTPUT_DIR, _n)
    _d = open(_p, 'rb').read(33)
    _w, _h = struct.unpack('>II', _d[16:24])
    print(f"  {_n:<34} {_w:>5} x {_h:<5}  {os.path.getsize(_p)/1e6:>6.2f} MB")

  wrote FoS_6panel.png        (alpha | static FoS | FoS Oct2019 / k_c | Form B | failure)
  wrote fig4_ROC_Youden.png   (correlation baseline, per the caption)
  wrote validation_fig6.png   (observed | predicted | confusion overlay)
  wrote depth_sensitivity_6panel.png  (3 FoS fields / 3 failure masks)

FIGURE CAPTION CHECK — every number the four captions assert
  [PASS] slope max shown in panel (a)                           73.567   caption     73.570
  [PASS] theta*, 20 m cap, correlation baseline                  0.500   caption      0.500
  [PASS] theta*, 5 m cap, correlation baseline                   0.550   caption      0.550
  [PASS] theta*, 3 m cap, correlation baseline                   0.650   caption      0.650
  [PASS] true positives in overlay                          109511.000   caption 109511.000
  [PASS] false positives in overlay                        1614103.000   caption 1614103.000
  [PASS] false negatives in overlay                          61036.000   caption 

In [18]:
# ============================================================================
#  §16  FLAGGED FRACTIONS  ->  backs the six percentages in Section 5.4
#
#  Section 5.4 asserts that each depth cap at its own optimal threshold flags
#  27.2 / 24.3 / 21.4% of the scored domain, and that at a common theta = 0.90
#  the same three caps flag 56.5 / 24.3 / 0.4%. None of those six numbers is in
#  any table. This cell recovers them -- and the same pair for all twelve
#  configurations -- from what GRID already holds:
#
#      flagged = (hit * n_pos + far * n_neg) / n_scored
#
#  No sweep is re-run and no raster is rebuilt: hit/far at the optimum are
#  stored per configuration, and the full hr/far arrays are stored over
#  THRESHOLDS, of which 0.90 is a grid point.
#
#  Paste the whole output back.
# ============================================================================
guard()

N_POS = len(REF['pos'])
N_NEG = len(REF['neg'])

# The validity mask v in sweep_config does not depend on which cap or cohesion
# scenario is in play (z is clipped to a floor of 0.1, so z > 0 always holds
# wherever z_raw is finite). Assert that rather than assume it, because the
# whole calculation below reuses one n_pos/n_neg across all twelve.
_ns = {k: r['n_scored'] for k, r in GRID.items()}
assert len(set(_ns.values())) == 1, f"scored counts differ across configs: {_ns}"
N_SC = N_POS + N_NEG
assert N_SC == REF['n_scored'], f"{N_SC} != {REF['n_scored']}"

I90 = int(np.argmin(np.abs(THRESHOLDS - 0.90)))
assert abs(THRESHOLDS[I90] - 0.90) < 1e-9, "0.90 is not on the threshold grid"


def _flagged(hit, far):
    return 100.0 * (hit * N_POS + far * N_NEG) / N_SC


print(f"scored {N_SC:,}   positives {N_POS:,}   negatives {N_NEG:,}")
print(f"common threshold index {I90} -> theta = {THRESHOLDS[I90]:.2f}\n")

print(f"{'cohesion':<12}{'z-cap':<12}{'theta*':>8}{'flag@own':>10}{'flag@0.90':>11}")
print("-" * 53)
ROWS = {}
for (cn, zn), r in GRID.items():
    own = _flagged(r['hit'], r['far_best'])
    common = _flagged(r['hr'][I90], r['far'][I90])
    ROWS[(cn, zn)] = (r['theta'], own, common)
    print(f"{cn:<12}{zn:<12}{r['theta']:>8.2f}{own:>9.1f}%{common:>10.1f}%")

# -- the two triples Section 5.4 quotes, at the primary cohesion --------------
print(f"\nSection 5.4, primary cohesion ({PRIMARY_C}):")
PUB_OWN = {'z_original': 27.2, 'z_cap_5m': 24.3, 'z_cap_3m': 21.4}
PUB_COMMON = {'z_original': 56.5, 'z_cap_5m': 24.3, 'z_cap_3m': 0.4}
_bad = 0
for zn, lab in [('z_original', '20 m'), ('z_cap_5m', '5 m'), ('z_cap_3m', '3 m')]:
    _th, own, common = ROWS[(PRIMARY_C, zn)]
    for what, got, pub in [("at own theta*", own, PUB_OWN[zn]),
                           ("at theta=0.90", common, PUB_COMMON[zn])]:
        good = abs(got - pub) <= 0.1
        _bad += (not good)
        print(f"  [{'PASS' if good else 'FAIL':<4}] {lab:>5} cap {what:<15}"
              f"{got:>7.1f}%   paper {pub:>6.1f}%")
print(f"\n{6 - _bad} of 6 match Section 5.4")

# -- ready-to-paste LaTeX column for Table 4 (tab:sweep) ---------------------
print("\n" + "=" * 60)
print("Flagged column for Table 4, in table row order:")
print("=" * 60)
for cn in ['c_baseline', 'c_times2', 'c_plus5', 'c_plus10']:
    for zn in ['z_original', 'z_cap_5m', 'z_cap_3m']:
        _th, own, common = ROWS[(cn, zn)]
        bold = (cn == PRIMARY_C and zn == PRIMARY_Z)
        cell = f"\\textbf{{{own:.1f}\\%}}" if bold else f"{own:.1f}\\%"
        print(f"  {cn:<12}{zn:<12} & {cell}")
print("=" * 60)

scored 7,104,440   positives 170,547   negatives 6,933,893
common threshold index 10 -> theta = 0.90

cohesion    z-cap         theta*  flag@own  flag@0.90
-----------------------------------------------------
c_baseline  z_original      0.50     21.0%      85.6%
c_baseline  z_cap_5m        0.55     21.0%      77.2%
c_baseline  z_cap_3m        0.65     26.2%      65.2%
c_plus5     z_original      0.60     24.0%      71.3%
c_plus5     z_cap_5m        0.75     26.2%      47.9%
c_plus5     z_cap_3m        0.90     23.3%      23.3%
c_times2    z_original      0.55     20.9%      76.9%
c_times2    z_cap_5m        0.65     20.6%      58.2%
c_times2    z_cap_3m        0.75     17.5%      36.9%
c_plus10    z_original      0.70     27.2%      56.5%
c_plus10    z_cap_5m        0.90     24.3%      24.3%
c_plus10    z_cap_3m        1.15     21.4%       0.4%

Section 5.4, primary cohesion (c_plus10):
  [PASS]  20 m cap at own theta*     27.2%   paper   27.2%
  [PASS]  20 m cap at theta=0.90     56.

In [19]:
# ============================================================================
#  §17  FORM B ZONE STATISTICS  ->  every number in Section 5.3
#
#  Section 5.3 quotes fifteen figures about the Form B bound
#  |S~_kh|max = 1 - 1/FoS_static, of which only two (83.1% defined, mean 0.387)
#  appear in master_summary.txt. This cell recomputes the lot.
#
#  Two conventions need stating, because the section's wording is ambiguous and
#  the answer differs:
#    * the bound EXISTS only where FoS_static > 1. Zone statistics are therefore
#      reported over the pixels of that zone where the bound is defined, which
#      is what the section's closing paragraph says is intended.
#    * the "gentle lowlands (alpha < 15 deg)" band is bounded below by the
#      scored domain itself at alpha >= 10 deg, so it is really 10-15 deg.
#
#  Paste the whole output back.
# ============================================================================
guard()

FS = REF['fos_static']
FB = sensitivity_FormB(FS)                 # 1 - 1/FoS_static, NaN where <= 1
DEF = SCORED & np.isfinite(FB)             # bound defined
A = P.alpha

n_sc = int(SCORED.sum())
n_def = int(DEF.sum())

print("=" * 76)
print("FORM B — Section 5.3")
print("=" * 76)


def chk(label, got, pub, tol, fmt="{:.4g}"):
    ok = abs(got - pub) <= tol
    print(f"  [{'PASS' if ok else 'FAIL':<4}] {label:<46}"
          f"{fmt.format(got):>12}   paper {fmt.format(pub):>10}")
    return 0 if ok else 1


bad = 0
print("\n-- domain --")
print(f"  scored {n_sc:,}   bound defined {n_def:,}")
bad += chk("fraction of scored where bound is defined", 100*n_def/n_sc, 83.1, 0.1)
bad += chk("defined pixels (millions)", n_def/1e6, 5.90, 0.01)
bad += chk("mean bound over defined", float(np.nanmean(FB[DEF])), 0.387, 0.001)
bad += chk("max bound over municipality", float(np.nanmax(FB[DEF])), 0.985, 0.001)

print("\n-- three zones (over pixels where the bound is defined) --")
ZONES = [("alpha > 30",        (A > 30),              0.09, None),
         ("15 < alpha < 30",   (A > 15) & (A < 30),   0.32, (0.21, 0.43)),
         ("alpha < 15  (>=10)", (A < 15),             0.60, (0.55, 0.64))]
for name, sel, pub_mean, pub_iqr in ZONES:
    z = DEF & sel
    v = FB[z]
    q1, q3 = np.percentile(v, [25, 75])
    print(f"  {name:<20} n={int(z.sum()):>10,}  mean {np.mean(v):.4f}"
          f"  IQR {q1:.3f}-{q3:.3f}")
    bad += chk(f"   mean, {name}", float(np.mean(v)), pub_mean, 0.005, "{:.3f}")
    if pub_iqr:
        bad += chk(f"   IQR low, {name}", float(q1), pub_iqr[0], 0.005, "{:.3f}")
        bad += chk(f"   IQR high, {name}", float(q3), pub_iqr[1], 0.005, "{:.3f}")

print("\n-- steep terrain --")
steep = SCORED & (A > 30)
steep_def = DEF & (A > 30)
bad += chk("scored pixels steeper than 30 (millions)",
           int(steep.sum())/1e6, 1.59, 0.01)
bad += chk("of those, fraction where bound is defined",
           100*int(steep_def.sum())/int(steep.sum()), 24.2, 0.1)
bad += chk("bound below 0.3 on steep defined pixels (%)",
           100*float(np.mean(FB[steep_def] < 0.3)), 91.8, 0.1)
below30 = SCORED & (A <= 30)
print(f"  below 30 deg: defined on "
      f"{100*int((DEF & (A <= 30)).sum())/int(below30.sum()):.2f}% "
      f"(paper: 'essentially every pixel')")

print("\n-- FoS_static over the defined subdomain --")
fsd = FS[DEF]
bad += chk("median FoS_static", float(np.median(fsd)), 1.67, 0.005, "{:.3f}")
bad += chk("90th percentile FoS_static",
           float(np.percentile(fsd, 90)), 2.68, 0.005, "{:.3f}")

print("\n-- threshold equivalence, Eq. (threshold) --")
# bound > 1/3 <=> FoS_static > 1.5 ; > 0.5 <=> > 2 ; > 0.75 <=> > 4
for pub, cut, fscut in [(61.1, 1/3, 1.5), (33.4, 0.5, 2.0), (0.48, 0.75, 4.0)]:
    frac = 100*float(np.mean(FB[DEF] > cut))
    frac_fs = 100*float(np.mean(fsd > fscut))
    agree = "" if abs(frac - frac_fs) < 1e-6 else f"  (via FoS: {frac_fs:.3f})"
    bad += chk(f"bound above {cut:.3f}  (FoS_static > {fscut})", frac, pub, 0.05,
               "{:.2f}")
    if agree:
        print(f"          {agree}")

print("\n-- closed-form ladder in the last paragraph --")
# Section 5.3 illustrates the crossover with domain-typical values. Report both
# the set the text currently prints and the Table 3 footprint means.
def fos_static_scalar(c, phi, alpha, m, gamma, z):
    p, a = np.radians(phi), np.radians(alpha)
    return ((c + (gamma - m*GAMMA_W)*z*np.cos(a)**2*np.tan(p))
            / (gamma*z*np.sin(a)*np.cos(a)))


def crossover(c, phi, m, gamma, z):
    lo, hi = 10.0, 60.0
    for _ in range(90):
        mid = 0.5*(lo + hi)
        if fos_static_scalar(c, phi, mid, m, gamma, z) > 1.0:
            lo = mid
        else:
            hi = mid
    return lo


FP_C = float(np.nanmean((P.c + 10.0)[FOOTPRINT]))
FP_G = float(np.nanmean(P.gamma[FOOTPRINT]))
FP_P = float(np.nanmean(P.phi[FOOTPRINT]))
print(f"  Table 3 footprint means: c'+10 {FP_C:.2f}  gamma {FP_G:.2f}  phi' {FP_P:.2f}")
for lab, (c, phi, g) in [
        ("as printed in 5.3 (13.0, 11.6, 31.2)", (13.0, 31.2, 11.6)),
        ("Table 3 footprint", (FP_C, FP_P, FP_G))]:
    lad = [fos_static_scalar(c, phi, a, 0.60, g, 5.0) for a in (28, 31, 32)]
    print(f"  {lab:<38} {lad[0]:.3f} / {lad[1]:.3f} / {lad[2]:.3f}"
          f"   crossover {crossover(c, phi, 0.60, g, 5.0):.2f} deg")

print("\n" + "=" * 76)
print(f"{'ALL PASS' if bad == 0 else str(bad) + ' MISMATCH(ES)'}"
      f" — a FAIL means Section 5.3 needs the new value")
print("=" * 76)

FORM B — Section 5.3

-- domain --
  scored 7,104,440   bound defined 5,906,214
  [PASS] fraction of scored where bound is defined            83.13   paper       83.1
  [PASS] defined pixels (millions)                            5.906   paper        5.9
  [PASS] mean bound over defined                             0.3865   paper      0.387
  [PASS] max bound over municipality                         0.9849   paper      0.985

-- three zones (over pixels where the bound is defined) --
  alpha > 30           n=   385,118  mean 0.0878  IQR 0.020-0.093
  [PASS]    mean, alpha > 30                                  0.088   paper      0.090
  15 < alpha < 30      n= 3,774,678  mean 0.3199  IQR 0.214-0.428
  [PASS]    mean, 15 < alpha < 30                             0.320   paper      0.320
  [PASS]    IQR low, 15 < alpha < 30                          0.214   paper      0.210
  [PASS]    IQR high, 15 < alpha < 30                         0.428   paper      0.430
  alpha < 15  (>=10)   n= 1,746,

In [20]:
# ============================================================================
#  §20  EBK STANDARD ERRORS — WHICH RASTER GENERATION, WHICH DOMAIN?
#
#  Table 5 states four EBK prediction standard errors:
#      fines 4.63 pp, water 4.40 pp, unit weight 1.10 kN/m3, PI 6.36
#
#  Averaged over SCORED, the v1 rasters in standard_error/ give 4.418, 4.143,
#  1.037 and 6.087 -- every one 4-6% below the published figure. Too uniform to
#  be four separate slips.
#
#  There are THREE generations of these rasters in the data folder:
#      standard_error/  (v1)      rasters_v2/       rasters_v3/
#  and the earlier extended-checks pass read v1. So the question is which
#  generation and which averaging domain reproduce Table 5. This cell sweeps
#  both and reports the combination that matches.
#
#  (The earlier pass reported these four as PASS at a tolerance of 0.35 -- wide
#  enough to swallow the entire discrepancy. That was my error; this replaces
#  it with a real test.)
#
#  Paste the whole output back.
# ============================================================================
guard()

import os
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

CLAIM = {'fines': 4.63, 'water': 4.40, 'gamma': 1.10, 'pi': 6.36}
LABEL = {'fines': 'fines', 'water': 'water', 'gamma': 'unit wt', 'pi': 'PI'}
KEYS = ['fines', 'water', 'gamma', 'pi']

ROOT = os.path.dirname(INPUT_DIR.rstrip('/')) if os.path.basename(
    INPUT_DIR.rstrip('/')) in ('input_rasters',) else INPUT_DIR

GENERATIONS = {
    'v1  (standard_error/)': {
        'fines': 'standard_error/siltclay_v1_standarderror.tif',
        'water': 'standard_error/watercontent_v11standarderror.tif',
        'gamma': 'standard_error/buk_v1_standarderror.tif',
        'pi':    'standard_error/PI_v1_standarderror.tif'},
    'v2  (rasters_v2/)': {
        'fines': 'rasters_v2/Silt_Clay__v2_standarderror.tif',
        'water': 'rasters_v2/water-content_v2_standarderror.tif',
        'gamma': 'rasters_v2/buk_v2_standarderror.tif',
        'pi':    'rasters_v2/PI_v2_standarderror.tif'},
    'v3  (rasters_v3/)': {
        'fines': 'rasters_v3/Silt_Clay__v3_standarderror.tif',
        'water': 'rasters_v3/water-content_v3_standarderror.tif',
        'gamma': 'rasters_v3/buk_v3_standarderror.tif',
        'pi':    'rasters_v3/PI_v3_standarderror.tif'},
}


def _align_se(p):
    with rasterio.open(p) as src:
        a = src.read(1).astype(np.float64)
        if src.nodata is not None:
            a[a == src.nodata] = np.nan
        if a.shape == ref_shape and src.transform.almost_equals(ref_transform,
                                                                precision=1e-6):
            return a
        d = np.full(ref_shape, np.nan)
        reproject(a, d, src_transform=src.transform, src_crs=src.crs,
                  dst_transform=ref_transform, dst_crs=ref_crs,
                  resampling=Resampling.bilinear,
                  src_nodata=np.nan, dst_nodata=np.nan)
        return d


DOMAINS = [("full raster", np.ones(ref_shape, bool)),
           ("footprint", FOOTPRINT),
           ("footprint & a>=10", FOOTPRINT & (P.alpha >= ALPHA_MIN)),
           ("SCORED", SCORED)]

print("=" * 92)
print("EBK standard errors: generation x domain, against Table 5")
print("=" * 92)
print(f"Table 5 claims:  " + "   ".join(f"{LABEL[k]} {CLAIM[k]}" for k in KEYS))
print("-" * 92)

results = []
for gen, files in GENERATIONS.items():
    paths = {k: os.path.join(ROOT, v) for k, v in files.items()}
    missing = [k for k, p in paths.items() if not os.path.exists(p)]
    if missing:
        print(f"\n{gen}:  SKIPPED, missing {missing}")
        continue
    se = {k: _align_se(p) for k, p in paths.items()}
    valid = np.ones(ref_shape, bool)
    for k in KEYS:
        valid &= np.isfinite(se[k])
    doms = DOMAINS + [("SE-valid & SCORED", SCORED & valid)]

    print(f"\n{gen}")
    print(f"  {'domain':<22}{'n px':>12}" + "".join(f"{LABEL[k]:>11}" for k in KEYS)
          + f"{'mean gap':>11}")
    for name, dom in doms:
        n = int(dom.sum())
        if n == 0:
            continue
        row, gaps = [], []
        for k in KEYS:
            v = se[k][dom]
            v = v[np.isfinite(v)]
            mu = float(v.mean()) if v.size else float('nan')
            row.append(mu)
            if np.isfinite(mu):
                gaps.append(abs(mu - CLAIM[k]) / CLAIM[k])
        g = 100 * float(np.mean(gaps)) if gaps else float('nan')
        results.append((g, gen, name, row))
        print(f"  {name:<22}{n:>12,}" + "".join(f"{x:>11.3f}" for x in row)
              + f"{g:>10.2f}%")

print("\n" + "=" * 92)
if results:
    results.sort(key=lambda r: r[0])
    g, gen, dom, row = results[0]
    print(f"closest: {gen}  over  {dom}")
    print(f"         " + "   ".join(f"{LABEL[k]} {row[i]:.3f}" for i, k in enumerate(KEYS)))
    print(f"         mean relative gap {g:.2f}%")
    if g < 0.6:
        print("\n  -> this combination reproduces Table 5. The table should name the")
        print("     raster generation and the averaging domain explicitly.")
    else:
        print("\n  -> NO combination reproduces Table 5 to better than 0.6%.")
        print("     The four sigma values need recomputing from whichever rasters")
        print("     the propagation actually used.")
print("=" * 92)
print("\nNote: only the four sigma values in Table 5's 'Uncertainty source'")
print("column are in question. The variance SHARES were recomputed on the")
print("SE-valid domain and matched the published values exactly.")

EBK standard errors: generation x domain, against Table 5
Table 5 claims:  fines 4.63   water 4.4   unit wt 1.1   PI 6.36
--------------------------------------------------------------------------------------------

v1  (standard_error/)
  domain                        n px      fines      water    unit wt         PI   mean gap
  full raster             21,106,173      4.628      4.333      1.100      6.355      0.41%
  footprint               13,728,083      4.587      4.324      1.089      6.313      1.10%
  footprint & a>=10        7,158,603      4.418      4.142      1.037      6.087      5.11%
  SCORED                   7,104,440      4.418      4.143      1.037      6.087      5.11%
  SE-valid & SCORED        4,754,653      4.418      4.215      1.037      6.087      4.70%

v2  (rasters_v2/)
  domain                        n px      fines      water    unit wt         PI   mean gap
  full raster             21,106,173      3.687      4.083      1.153      6.612      9.09%
  footp

In [21]:
# ============================================================================
#  §21  SECTION 6 — ERROR STRUCTURE AND BLOCKED PERFORMANCE
#
#  Covers the figures in Sections 6.2 and 6.4 that master_summary.txt does not:
#
#    6.2  TP / FN median slope; FN median FoS and the fraction within 0.05,
#         0.10 and 0.25 of theta*; FP vs TP median FoS and interquartile range
#    6.4  slope-PGA correlation; out-of-sample hit rate, FAR and Youden index
#         with their tile-to-tile spreads
#
#  The blocked-validation cell must have run first in this kernel: this cell
#  reuses its per-tile threshold counts rather than re-tiling the rasters.
#
#  Paste the whole output back.
# ============================================================================
guard()

_need = ['pos_ct', 'neg_ct', 'npos_t', 'nneg_t', 'lobo_theta', 'n_scored_tiles']
_missing = [v for v in _need if v not in globals()]
if _missing:
    raise RuntimeError(
        f"missing {_missing} -- run the blocked-validation cell (Section 6.4) "
        "in this kernel first; this cell reuses its per-tile counts")

TH = REF['theta']
FOS = REF['fos']
obs = SCORED & OBSERVED
pred = SCORED & (FOS < TH)

TPm = obs & pred
FNm = obs & ~pred
FPm = SCORED & ~obs & pred

bad = 0


def chk(label, got, pub, tol, fmt="{:.3f}"):
    global bad
    ok_ = abs(got - pub) <= tol
    bad += (not ok_)
    print(f"  [{'PASS' if ok_ else 'FAIL':<4}] {label:<44}"
          f"{fmt.format(got):>10}   paper {fmt.format(pub):>9}")


print("=" * 78)
print("SECTION 6.2 — error structure")
print("=" * 78)
print(f"  TP {int(TPm.sum()):,}   FN {int(FNm.sum()):,}   FP {int(FPm.sum()):,}\n")

chk("median slope, true positives (deg)",
    float(np.median(P.alpha[TPm])), 41.0, 0.05, "{:.1f}")
chk("median slope, false negatives (deg)",
    float(np.median(P.alpha[FNm])), 23.3, 0.05, "{:.1f}")

fn_fos = FOS[FNm]
chk("median FoS, false negatives", float(np.median(fn_fos)), 1.128, 0.001)
for d, pub in [(0.05, 14.7), (0.10, 26.7), (0.25, 53.1)]:
    chk(f"  FN within {d:.2f} of theta* (%)",
        100 * float(np.mean(np.abs(fn_fos - TH) <= d)), pub, 0.1, "{:.1f}")

fp_fos, tp_fos = FOS[FPm], FOS[TPm]
chk("median FoS, false positives", float(np.median(fp_fos)), 0.798, 0.001)
chk("median FoS, true positives", float(np.median(tp_fos)), 0.730, 0.001)
for lab, arr, lo_p, hi_p in [("false positives", fp_fos, 0.735, 0.851),
                             ("true positives", tp_fos, 0.683, 0.795)]:
    lo, hi = np.percentile(arr, [25, 75])
    chk(f"  IQR low, {lab}", float(lo), lo_p, 0.001)
    chk(f"  IQR high, {lab}", float(hi), hi_p, 0.001)

print()
print("=" * 78)
print("SECTION 6.4 — blocked performance")
print("=" * 78)

# --- slope vs modelled PGA over the scored domain ---------------------------
_pga = None
for _n in ('pga_g', 'k_h'):
    if hasattr(P, _n):
        _pga = getattr(P, _n)
        _pga_name = _n
        break
if _pga is None:
    print("  [SKIP] no PGA field on P; slope-PGA correlation not checked")
else:
    a = P.alpha[SCORED]
    g = _pga[SCORED]
    m = np.isfinite(a) & np.isfinite(g)
    r = float(np.corrcoef(a[m], g[m])[0, 1])
    print(f"  (correlating alpha against P.{_pga_name})")
    chk("slope-PGA correlation over scored domain", r, -0.42, 0.005, "{:+.2f}")

# --- out-of-sample performance on each held-out tile -------------------------
# For tile t the threshold was chosen on every scored pixel OUTSIDE it
# (lobo_theta[t]); performance is then read off tile t's own counts.
idx = {round(float(v), 4): i for i, v in enumerate(THRESHOLDS)}
hit_o, far_o, J_o = [], [], []
for t in range(n_scored_tiles):
    i = idx[round(float(lobo_theta[t]), 4)]
    h = pos_ct[t][i] / npos_t[t]
    f = neg_ct[t][i] / nneg_t[t]
    hit_o.append(h)
    far_o.append(f)
    J_o.append(h - f)
hit_o, far_o, J_o = map(np.array, (hit_o, far_o, J_o))

print(f"\n  evaluated on {n_scored_tiles} held-out tiles")
chk("out-of-sample hit rate, mean (%)", 100 * hit_o.mean(), 51.1, 0.1, "{:.1f}")
chk("  its standard deviation (pts)", 100 * hit_o.std(ddof=1), 28.7, 0.1, "{:.1f}")
chk("out-of-sample FAR, mean (%)", 100 * far_o.mean(), 27.0, 0.1, "{:.1f}")
chk("  its standard deviation (pts)", 100 * far_o.std(ddof=1), 18.5, 0.1, "{:.1f}")
chk("out-of-sample Youden J, mean", float(J_o.mean()), 0.241, 0.001)
chk("  its standard deviation", float(J_o.std(ddof=1)), 0.205, 0.001)

print(f"\n  in-sample J for comparison: {REF['J_best']:.3f}")
print(f"  J falls by {100*(1 - J_o.mean()/REF['J_best']):.0f}%"
      f"   (paper: 'about 40%')")

print("\n" + "=" * 78)
print(f"{'ALL PASS' if bad == 0 else str(bad) + ' MISMATCH(ES)'}"
      f" — a FAIL means Section 6 needs the new value")
print("=" * 78)

SECTION 6.2 — error structure
  TP 109,511   FN 61,036   FP 1,614,103

  [PASS] median slope, true positives (deg)                41.0   paper      41.0
  [PASS] median slope, false negatives (deg)               23.3   paper      23.3
  [PASS] median FoS, false negatives                      1.128   paper     1.128
  [PASS]   FN within 0.05 of theta* (%)                    14.7   paper      14.7
  [PASS]   FN within 0.10 of theta* (%)                    26.7   paper      26.7
  [PASS]   FN within 0.25 of theta* (%)                    53.1   paper      53.1
  [PASS] median FoS, false positives                      0.798   paper     0.798
  [PASS] median FoS, true positives                       0.730   paper     0.730
  [PASS]   IQR low, false positives                       0.735   paper     0.735
  [PASS]   IQR high, false positives                      0.851   paper     0.851
  [PASS]   IQR low, true positives                        0.683   paper     0.683
  [PASS]   IQR high, true p

In [22]:
# ============================================================================
#  §22  POINT-LEVEL VALIDATION — Section 6.5, rebuilt on the v10 pipeline
#
#  Section 6.5 reports roughly twenty-five figures and NONE of them can be
#  traced to a surviving artifact. Makilala_Point_Validation.ipynb (1 Aug) is
#  not their source: it uses a FoS threshold of 0.8 against the paper's 0.90,
#  carries 40 samples against 43, reads Slope_PRSTM51N.tif and the raw Mak_*
#  rasters with no registration correction, and refers to "v5 results". Its
#  numbers appear nowhere in the paper. Whatever produced Section 6.5 is gone.
#
#  This cell recomputes the section from the field spreadsheet against the
#  corrected, aligned rasters, reusing the pipeline's own correlation
#  functions so that the "measured properties" scoring tests the correlations
#  and not a second implementation of them.
#
#  Paste the whole output back.
# ============================================================================
guard()

import os
import numpy as np
import pandas as pd
from rasterio.warp import transform as warp_transform


def _nearest_observed_m(r, c, mask, px, max_m=600.0):
    """Distance in metres from (r, c) to the nearest True cell of `mask`.

    A windowed search rather than a full distance transform: there are 43
    points, and an exact EDT over 21 million pixels to serve them would be
    absurd. Returns np.inf if nothing is found inside the window.
    """
    rad = int(np.ceil(max_m / px))
    r0, r1 = max(0, r - rad), min(mask.shape[0], r + rad + 1)
    c0, c1 = max(0, c - rad), min(mask.shape[1], c + rad + 1)
    sub = mask[r0:r1, c0:c1]
    if not sub.any():
        return np.inf
    rs, cs = np.nonzero(sub)
    d = np.hypot((rs + r0) - r, (cs + c0) - c) * px
    return float(d.min())


def _spearman(a, b):
    """Rank correlation, average ranks for ties. Avoids a scipy dependency."""
    def rank(x):
        order = np.argsort(x, kind='mergesort')
        rk = np.empty(len(x), float)
        rk[order] = np.arange(1, len(x) + 1)
        _, inv, cnt = np.unique(x, return_inverse=True, return_counts=True)
        if (cnt > 1).any():                      # average tied ranks
            sums = np.zeros(len(cnt))
            np.add.at(sums, inv, rk)
            rk = (sums / cnt)[inv]
        return rk
    ra, rb = rank(np.asarray(a, float)), rank(np.asarray(b, float))
    return float(np.corrcoef(ra, rb)[0, 1])

XLSX = os.path.join(INPUT_DIR, "Makilala Geotechnical Data.xlsx")
PX = abs(ref_transform.a)                      # metres per pixel
TH = REF['theta']
FOS = REF['fos']

df = pd.read_excel(XLSX, sheet_name=0, header=1)
df.columns = [str(c).strip() for c in df.columns]


def col(*cands):
    for c in df.columns:
        u = c.upper()
        for cand in cands:
            if cand in u:
                return c
    raise KeyError(cands)


C_LAT, C_LON = col("LATITUDE"), col("LONGITUDE")
C_TYPE = col("SITE TYPE")
C_W, C_G = col("WATER CONTENT"), col("BULK UNIT WEIG")
C_SAND, C_FINES, C_PI = col("SAND %"), col("SILT/CLAY"), col("PI")

df = df[pd.to_numeric(df[C_LAT], errors='coerce').notna()].copy()
for c in (C_LAT, C_LON, C_W, C_G, C_SAND, C_FINES, C_PI):
    df[c] = pd.to_numeric(df[c], errors='coerce')
is_ls = df[C_TYPE].astype(str).str.upper().str.startswith('LANDSLIDE').values

print("=" * 76)
print("SECTION 6.5 — POINT-LEVEL VALIDATION")
print("=" * 76)
print(f"  spreadsheet rows with coordinates : {len(df)}   (paper: 43 sites)")
print(f"    landslide / non-landslide       : {is_ls.sum()} / {(~is_ls).sum()}")

# --- locate every site on the reference grid --------------------------------
xs, ys = warp_transform("EPSG:4326", ref_crs,
                        df[C_LON].tolist(), df[C_LAT].tolist())
inv = ~ref_transform
cc, rr = [], []
for x, y in zip(xs, ys):
    c, r = inv * (x, y)
    cc.append(int(np.floor(c)))
    rr.append(int(np.floor(r)))
rr, cc = np.array(rr), np.array(cc)
on_grid = (rr >= 0) & (rr < ref_shape[0]) & (cc >= 0) & (cc < ref_shape[1])
print(f"  sites falling on the reference grid: {int(on_grid.sum())}")

R, C = rr[on_grid], cc[on_grid]
LS = is_ls[on_grid]
sub = df[on_grid].reset_index(drop=True)

in_dom = SCORED[R, C]
print(f"  sites inside the scored domain    : {int(in_dom.sum())}   (paper: 36 of 43)")
out = ~in_dom
if out.any():
    a_out = P.alpha[R[out], C[out]]
    print(f"    the {int(out.sum())} excluded: mean slope "
          f"{np.nanmean(a_out[LS[out]]):.1f} deg landslide / "
          f"{np.nanmean(a_out[~LS[out]]):.1f} deg non-landslide"
          f"   (paper: 5.4 / 6.4)")

# --- co-location of landslide sites with the inventory ----------------------
print("\n-- co-location with the mapped inventory --")
d = np.array([_nearest_observed_m(r, c, OBSERVED, PX)
              for r, c in zip(R, C)])
d_ls = d[LS]
print(f"  landslide sites considered        : {len(d_ls)}   (paper: 25)")
print(f"  inside a mapped polygon (d = 0)   : {int((d_ls == 0).sum())}   (paper: 3)")
print(f"  within 25 m                       : {int((d_ls <= 25).sum())}   (paper: 15)")
print(f"  within 50 m                       : {int((d_ls <= 50).sum())}   (paper: 19)")
print(f"  median separation                 : {np.median(d_ls):.1f} m   (paper: 25)")

# --- slope contrast within the evaluated domain -----------------------------
a_in = P.alpha[R[in_dom], C[in_dom]]
l_in = LS[in_dom]
print(f"\n  slope, landslide sites in domain  : {np.nanmean(a_in[l_in]):.1f} deg"
      f"   (paper: 25.1)")
print(f"  slope, non-landslide in domain    : {np.nanmean(a_in[~l_in]):.1f} deg"
      f"   (paper: 18.9)")


def score(fos_vals, truth, label, pub=None):
    ok_ = np.isfinite(fos_vals)
    f, t = fos_vals[ok_], truth[ok_]
    pred = f < TH
    TPn = int((pred & t).sum()); FNn = int((~pred & t).sum())
    FPn = int((pred & ~t).sum()); TNn = int((~pred & ~t).sum())
    hit = TPn / max(TPn + FNn, 1)
    far = FPn / max(FPn + TNn, 1)
    prec = TPn / max(TPn + FPn, 1)
    print(f"\n  {label}  (n = {len(f)}: {int(t.sum())} positive)")
    print(f"    TP {TPn}  FN {FNn}  FP {FPn}  TN {TNn}")
    print(f"    hit {100*hit:.1f}%   FAR {100*far:.1f}%   J {hit-far:.3f}"
          f"   precision {100*prec:.1f}%")
    print(f"    mean FoS  landslide {np.mean(f[t]):.3f}  non-landslide {np.mean(f[~t]):.3f}")
    if pub:
        print(f"    paper: {pub}")
    return f, t, ok_


# --- scoring 1: the map as deployed -----------------------------------------
print("\n-- scoring the FoS raster at each site --")
fos_map = np.full(len(R), np.nan)
fos_map[in_dom] = FOS[R[in_dom], C[in_dom]]
f_map, t_map, ok_map = score(fos_map, LS, "FoS sampled from the map",
                             "hit 40.9%, FAR 14.3%, J 0.266, precision 81.8%, "
                             "mean FoS 1.179 / 1.261")

# --- scoring 2: the site's own measured index properties --------------------
print("\n-- scoring from measured index properties (same correlations) --")
phi_m = estimate_phi(sub[C_PI].values, sub[C_SAND].values, sub[C_FINES].values)
# The map being compared against is the PRIMARY configuration, whose cohesion
# field carries the +10 kPa offset. Scoring measured properties without it
# compares two different models, not two ways of obtaining the same one: the
# missing 10 kPa costs about 10/(gamma z sin a cos a) ~ 0.4 of FoS at a typical
# site, which is the whole of the discrepancy it produces.
_C_OFF = C_SCENARIOS[PRIMARY_C]
if _C_OFF is None:
    raise RuntimeError("primary scenario is multiplicative; adjust this line")
c_m = estimate_c(sub[C_FINES].values, sub[C_SAND].values) + _C_OFF
print(f"  (measured cohesion carries the primary offset, +{_C_OFF:.0f} kPa)")
m_m = estimate_m(sub[C_W].values)
g_m = sub[C_G].values

fos_meas = np.full(len(R), np.nan)
sel = in_dom & np.isfinite(phi_m) & np.isfinite(c_m) & np.isfinite(m_m) & np.isfinite(g_m)
if sel.any():
    fos_meas[sel] = compute_FoS(c_m[sel], phi_m[sel],
                                P.alpha[R[sel], C[sel]], m_m[sel], g_m[sel],
                                np.clip(P.z_raw[R[sel], C[sel]], 0.1, 5.0),
                                P.k_h[R[sel], C[sel]])
score(fos_meas, LS, "FoS from measured properties", "J 0.097")

# --- the sites scorable both ways -------------------------------------------
both = np.isfinite(fos_map) & np.isfinite(fos_meas)
print(f"\n-- the {int(both.sum())} sites scorable both ways   (paper: 27) --")
if both.sum() > 2:
    a, b = fos_meas[both], fos_map[both]
    diff = a - b
    rho = _spearman(a, b)
    nd = int(((a < TH) != (b < TH)).sum())
    print(f"    FoS difference (measured - map): mean {diff.mean():+.3f}"
          f"   sd {diff.std(ddof=1):.3f}   (paper: +0.037, sd 0.137)")
    print(f"    Spearman rho                   : {rho:.3f}   (paper: 0.935)")
    print(f"    classified differently         : {nd} of {int(both.sum())}"
          f"   (paper: 3 of 27)")

    print("\n    measured vs interpolated at those sites:")
    # P keeps cohesion and saturation, not the fines and sand percentages they
    # were derived from -- those rasters are dropped after use to save memory.
    # Reporting P.c against measured fines would compare kPa with per cent.
    for lab, meas, ras, pub in [
            ("fines %", sub[C_FINES].values, None, "11.35 vs 10.96"),
            ("sand %", sub[C_SAND].values, None, "84.99 vs 85.13"),
            ("water %", sub[C_W].values, P.water, "40.47 vs 40.61"),
            ("unit weight", sub[C_G].values, P.gamma, "12.69 vs 12.47")]:
        if ras is None:
            print(f"      {lab:<12} measured {np.nanmean(meas[both]):.2f}"
                  f"   (raster not retained; paper {pub})")
            continue
        iv = ras[R[both], C[both]]
        mv = meas[both]
        good = np.isfinite(iv) & np.isfinite(mv)
        print(f"      {lab:<12} measured {mv[good].mean():.2f}"
              f"   interpolated {iv[good].mean():.2f}"
              f"   MAD {np.abs(mv[good]-iv[good]).mean():.1f}   (paper {pub})")

print("\n" + "=" * 76)
print("Compare each line against Section 6.5. Because the section's original")
print("computation cannot be located, treat DISAGREEMENTS as the paper needing")
print("the new value, not as this cell being wrong -- but check the site counts")
print("first, since those determine everything downstream.")
print("=" * 76)

SECTION 6.5 — POINT-LEVEL VALIDATION
  spreadsheet rows with coordinates : 43   (paper: 43 sites)
    landslide / non-landslide       : 25 / 18
  sites falling on the reference grid: 43
  sites inside the scored domain    : 36   (paper: 36 of 43)
    the 7 excluded: mean slope 5.4 deg landslide / 6.4 deg non-landslide   (paper: 5.4 / 6.4)

-- co-location with the mapped inventory --
  landslide sites considered        : 25   (paper: 25)
  inside a mapped polygon (d = 0)   : 3   (paper: 3)
  within 25 m                       : 14   (paper: 15)
  within 50 m                       : 18   (paper: 19)
  median separation                 : 25.0 m   (paper: 25)

  slope, landslide sites in domain  : 25.1 deg   (paper: 25.1)
  slope, non-landslide in domain    : 18.9 deg   (paper: 18.9)

-- scoring the FoS raster at each site --

  FoS sampled from the map  (n = 36: 22 positive)
    TP 9  FN 13  FP 2  TN 12
    hit 40.9%   FAR 14.3%   J 0.266   precision 81.8%
    mean FoS  landslide 1.179  no

In [23]:
# ============================================================================
#  §22a  POINT-LEVEL VALIDATION — Section 6.5, rebuilt on the v10 pipeline
#
#  Section 6.5 reports roughly twenty-five figures and NONE of them can be
#  traced to a surviving artifact. Makilala_Point_Validation.ipynb (1 Aug) is
#  not their source: it uses a FoS threshold of 0.8 against the paper's 0.90,
#  carries 40 samples against 43, reads Slope_PRSTM51N.tif and the raw Mak_*
#  rasters with no registration correction, and refers to "v5 results". Its
#  numbers appear nowhere in the paper. Whatever produced Section 6.5 is gone.
#
#  This cell recomputes the section from the field spreadsheet against the
#  corrected, aligned rasters, reusing the pipeline's own correlation
#  functions so that the "measured properties" scoring tests the correlations
#  and not a second implementation of them.
#
#  Paste the whole output back.
# ============================================================================
guard()

import os
import numpy as np
import pandas as pd
from rasterio.warp import transform as warp_transform


def _nearest_observed_m(r, c, mask, px, max_m=600.0):
    """Distance in metres from (r, c) to the nearest True cell of `mask`.

    A windowed search rather than a full distance transform: there are 43
    points, and an exact EDT over 21 million pixels to serve them would be
    absurd. Returns np.inf if nothing is found inside the window.
    """
    rad = int(np.ceil(max_m / px))
    r0, r1 = max(0, r - rad), min(mask.shape[0], r + rad + 1)
    c0, c1 = max(0, c - rad), min(mask.shape[1], c + rad + 1)
    sub = mask[r0:r1, c0:c1]
    if not sub.any():
        return np.inf
    rs, cs = np.nonzero(sub)
    d = np.hypot((rs + r0) - r, (cs + c0) - c) * px
    return float(d.min())


def _spearman(a, b):
    """Rank correlation, average ranks for ties. Avoids a scipy dependency."""
    def rank(x):
        order = np.argsort(x, kind='mergesort')
        rk = np.empty(len(x), float)
        rk[order] = np.arange(1, len(x) + 1)
        _, inv, cnt = np.unique(x, return_inverse=True, return_counts=True)
        if (cnt > 1).any():                      # average tied ranks
            sums = np.zeros(len(cnt))
            np.add.at(sums, inv, rk)
            rk = (sums / cnt)[inv]
        return rk
    ra, rb = rank(np.asarray(a, float)), rank(np.asarray(b, float))
    return float(np.corrcoef(ra, rb)[0, 1])

XLSX = os.path.join(INPUT_DIR, "Makilala Geotechnical Data.xlsx")
PX = abs(ref_transform.a)                      # metres per pixel
TH = REF['theta']
FOS = REF['fos']

df = pd.read_excel(XLSX, sheet_name=0, header=1)
df.columns = [str(c).strip() for c in df.columns]


def col(*cands):
    for c in df.columns:
        u = c.upper()
        for cand in cands:
            if cand in u:
                return c
    raise KeyError(cands)


C_LAT, C_LON = col("LATITUDE"), col("LONGITUDE")
C_TYPE = col("SITE TYPE")
C_W, C_G = col("WATER CONTENT"), col("BULK UNIT WEIG")
C_SAND, C_FINES, C_PI = col("SAND %"), col("SILT/CLAY"), col("PI")

df = df[pd.to_numeric(df[C_LAT], errors='coerce').notna()].copy()
for c in (C_LAT, C_LON, C_W, C_G, C_SAND, C_FINES, C_PI):
    df[c] = pd.to_numeric(df[c], errors='coerce')
is_ls = df[C_TYPE].astype(str).str.upper().str.startswith('LANDSLIDE').values

print("=" * 76)
print("SECTION 6.5 — POINT-LEVEL VALIDATION")
print("=" * 76)
print(f"  spreadsheet rows with coordinates : {len(df)}   (paper: 43 sites)")
print(f"    landslide / non-landslide       : {is_ls.sum()} / {(~is_ls).sum()}")

# --- locate every site on the reference grid --------------------------------
xs, ys = warp_transform("EPSG:4326", ref_crs,
                        df[C_LON].tolist(), df[C_LAT].tolist())
inv = ~ref_transform
cc, rr = [], []
for x, y in zip(xs, ys):
    c, r = inv * (x, y)
    cc.append(int(np.floor(c)))
    rr.append(int(np.floor(r)))
rr, cc = np.array(rr), np.array(cc)
on_grid = (rr >= 0) & (rr < ref_shape[0]) & (cc >= 0) & (cc < ref_shape[1])
print(f"  sites falling on the reference grid: {int(on_grid.sum())}")

R, C = rr[on_grid], cc[on_grid]
LS = is_ls[on_grid]
sub = df[on_grid].reset_index(drop=True)

in_dom = SCORED[R, C]
print(f"  sites inside the scored domain    : {int(in_dom.sum())}   (paper: 36 of 43)")
out = ~in_dom
if out.any():
    a_out = P.alpha[R[out], C[out]]
    print(f"    the {int(out.sum())} excluded: mean slope "
          f"{np.nanmean(a_out[LS[out]]):.1f} deg landslide / "
          f"{np.nanmean(a_out[~LS[out]]):.1f} deg non-landslide"
          f"   (paper: 5.4 / 6.4)")

# --- co-location of landslide sites with the inventory ----------------------
print("\n-- co-location with the mapped inventory --")
d = np.array([_nearest_observed_m(r, c, OBSERVED, PX)
              for r, c in zip(R, C)])
d_ls = d[LS]
print(f"  landslide sites considered        : {len(d_ls)}   (paper: 25)")
print(f"  inside a mapped polygon (d = 0)   : {int((d_ls == 0).sum())}   (paper: 3)")
print(f"  within 25 m                       : {int((d_ls <= 25).sum())}   (paper: 15)")
print(f"  within 50 m                       : {int((d_ls <= 50).sum())}   (paper: 19)")
print(f"  median separation                 : {np.median(d_ls):.1f} m   (paper: 25)")

# --- slope contrast within the evaluated domain -----------------------------
a_in = P.alpha[R[in_dom], C[in_dom]]
l_in = LS[in_dom]
print(f"\n  slope, landslide sites in domain  : {np.nanmean(a_in[l_in]):.1f} deg"
      f"   (paper: 25.1)")
print(f"  slope, non-landslide in domain    : {np.nanmean(a_in[~l_in]):.1f} deg"
      f"   (paper: 18.9)")


def score(fos_vals, truth, label, pub=None):
    ok_ = np.isfinite(fos_vals)
    f, t = fos_vals[ok_], truth[ok_]
    pred = f < TH
    TPn = int((pred & t).sum()); FNn = int((~pred & t).sum())
    FPn = int((pred & ~t).sum()); TNn = int((~pred & ~t).sum())
    hit = TPn / max(TPn + FNn, 1)
    far = FPn / max(FPn + TNn, 1)
    prec = TPn / max(TPn + FPn, 1)
    print(f"\n  {label}  (n = {len(f)}: {int(t.sum())} positive)")
    print(f"    TP {TPn}  FN {FNn}  FP {FPn}  TN {TNn}")
    print(f"    hit {100*hit:.1f}%   FAR {100*far:.1f}%   J {hit-far:.3f}"
          f"   precision {100*prec:.1f}%")
    print(f"    mean FoS  landslide {np.mean(f[t]):.3f}  non-landslide {np.mean(f[~t]):.3f}")
    if pub:
        print(f"    paper: {pub}")
    return f, t, ok_


# --- scoring 1: the map as deployed -----------------------------------------
print("\n-- scoring the FoS raster at each site --")
fos_map = np.full(len(R), np.nan)
fos_map[in_dom] = FOS[R[in_dom], C[in_dom]]
f_map, t_map, ok_map = score(fos_map, LS, "FoS sampled from the map",
                             "hit 40.9%, FAR 14.3%, J 0.266, precision 81.8%, "
                             "mean FoS 1.179 / 1.261")

# --- scoring 2: the site's own measured index properties --------------------
print("\n-- scoring from measured index properties (same correlations) --")
phi_m = estimate_phi(sub[C_PI].values, sub[C_SAND].values, sub[C_FINES].values)
# The map being compared against is the PRIMARY configuration, whose cohesion
# field carries the +10 kPa offset. Scoring measured properties without it
# compares two different models, not two ways of obtaining the same one: the
# missing 10 kPa costs about 10/(gamma z sin a cos a) ~ 0.4 of FoS at a typical
# site, which is the whole of the discrepancy it produces.
_C_OFF = C_SCENARIOS[PRIMARY_C]
if _C_OFF is None:
    raise RuntimeError("primary scenario is multiplicative; adjust this line")
c_m = estimate_c(sub[C_FINES].values, sub[C_SAND].values) + _C_OFF
print(f"  (measured cohesion carries the primary offset, +{_C_OFF:.0f} kPa)")
m_m = estimate_m(sub[C_W].values)
g_m = sub[C_G].values

fos_meas = np.full(len(R), np.nan)
# Screen on the MEASURED COLUMNS, not on the estimator outputs. estimate_phi,
# estimate_c and estimate_m all substitute a default when their input is
# missing (30 deg, 5 kPa, m = 0.50) and never return NaN, so testing their
# outputs for finiteness screens nothing and silently admits sites scored on
# defaults rather than on laboratory data. This is Section 6.5's "laboratory
# completeness screen".
_have = {"PI": np.isfinite(sub[C_PI].values),
         "sand": np.isfinite(sub[C_SAND].values),
         "fines": np.isfinite(sub[C_FINES].values),
         "water": np.isfinite(sub[C_W].values),
         "unit weight": np.isfinite(sub[C_G].values)}
print("  measured values present, of the %d sites in the scored domain:"
      % int(in_dom.sum()))
for k, v in _have.items():
    print(f"    {k:<12} {int((v & in_dom).sum()):>3}")
complete = np.ones(len(R), bool)
for v in _have.values():
    complete &= v
sel = in_dom & complete
print(f"  complete on all five                {int(sel.sum()):>3}   (paper: 27)")
if sel.any():
    fos_meas[sel] = compute_FoS(c_m[sel], phi_m[sel],
                                P.alpha[R[sel], C[sel]], m_m[sel], g_m[sel],
                                np.clip(P.z_raw[R[sel], C[sel]], 0.1, 5.0),
                                P.k_h[R[sel], C[sel]])
score(fos_meas, LS, "FoS from measured properties", "J 0.097")

# --- the sites scorable both ways -------------------------------------------
both = np.isfinite(fos_map) & np.isfinite(fos_meas)
print(f"\n-- the {int(both.sum())} sites scorable both ways   (paper: 27) --")
if both.sum() > 2:
    a, b = fos_meas[both], fos_map[both]
    diff = a - b
    rho = _spearman(a, b)
    nd = int(((a < TH) != (b < TH)).sum())
    print(f"    FoS difference (measured - map): mean {diff.mean():+.3f}"
          f"   sd {diff.std(ddof=1):.3f}   (paper: +0.037, sd 0.137)")
    print(f"    Spearman rho                   : {rho:.3f}   (paper: 0.935)")
    print(f"    classified differently         : {nd} of {int(both.sum())}"
          f"   (paper: 3 of 27)")

    print("\n    measured vs interpolated at those sites:")
    # P keeps cohesion and saturation, not the fines and sand percentages they
    # were derived from -- those rasters are dropped after use to save memory.
    # Reporting P.c against measured fines would compare kPa with per cent.
    for lab, meas, ras, pub in [
            ("fines %", sub[C_FINES].values, None, "11.35 vs 10.96"),
            ("sand %", sub[C_SAND].values, None, "84.99 vs 85.13"),
            ("water %", sub[C_W].values, P.water, "40.47 vs 40.61"),
            ("unit weight", sub[C_G].values, P.gamma, "12.69 vs 12.47")]:
        if ras is None:
            print(f"      {lab:<12} measured {np.nanmean(meas[both]):.2f}"
                  f"   (raster not retained; paper {pub})")
            continue
        iv = ras[R[both], C[both]]
        mv = meas[both]
        good = np.isfinite(iv) & np.isfinite(mv)
        print(f"      {lab:<12} measured {mv[good].mean():.2f}"
              f"   interpolated {iv[good].mean():.2f}"
              f"   MAD {np.abs(mv[good]-iv[good]).mean():.1f}   (paper {pub})")

print("\n" + "=" * 76)
print("Compare each line against Section 6.5. Because the section's original")
print("computation cannot be located, treat DISAGREEMENTS as the paper needing")
print("the new value, not as this cell being wrong -- but check the site counts")
print("first, since those determine everything downstream.")
print("=" * 76)

SECTION 6.5 — POINT-LEVEL VALIDATION
  spreadsheet rows with coordinates : 43   (paper: 43 sites)
    landslide / non-landslide       : 25 / 18
  sites falling on the reference grid: 43
  sites inside the scored domain    : 36   (paper: 36 of 43)
    the 7 excluded: mean slope 5.4 deg landslide / 6.4 deg non-landslide   (paper: 5.4 / 6.4)

-- co-location with the mapped inventory --
  landslide sites considered        : 25   (paper: 25)
  inside a mapped polygon (d = 0)   : 3   (paper: 3)
  within 25 m                       : 14   (paper: 15)
  within 50 m                       : 18   (paper: 19)
  median separation                 : 25.0 m   (paper: 25)

  slope, landslide sites in domain  : 25.1 deg   (paper: 25.1)
  slope, non-landslide in domain    : 18.9 deg   (paper: 18.9)

-- scoring the FoS raster at each site --

  FoS sampled from the map  (n = 36: 22 positive)
    TP 9  FN 13  FP 2  TN 12
    hit 40.9%   FAR 14.3%   J 0.266   precision 81.8%
    mean FoS  landslide 1.179  no

In [24]:
# ============================================================================
#  §22b  POINT-LEVEL VALIDATION — Section 6.5, rebuilt on the v10 pipeline
#
#  Section 6.5 reports roughly twenty-five figures and NONE of them can be
#  traced to a surviving artifact. Makilala_Point_Validation.ipynb (1 Aug) is
#  not their source: it uses a FoS threshold of 0.8 against the paper's 0.90,
#  carries 40 samples against 43, reads Slope_PRSTM51N.tif and the raw Mak_*
#  rasters with no registration correction, and refers to "v5 results". Its
#  numbers appear nowhere in the paper. Whatever produced Section 6.5 is gone.
#
#  This cell recomputes the section from the field spreadsheet against the
#  corrected, aligned rasters, reusing the pipeline's own correlation
#  functions so that the "measured properties" scoring tests the correlations
#  and not a second implementation of them.
#
#  Paste the whole output back.
# ============================================================================
guard()

import os
import numpy as np
import pandas as pd
from rasterio.warp import transform as warp_transform


def _nearest_observed_m(r, c, mask, px, max_m=600.0):
    """Distance in metres from (r, c) to the nearest True cell of `mask`.

    A windowed search rather than a full distance transform: there are 43
    points, and an exact EDT over 21 million pixels to serve them would be
    absurd. Returns np.inf if nothing is found inside the window.
    """
    rad = int(np.ceil(max_m / px))
    r0, r1 = max(0, r - rad), min(mask.shape[0], r + rad + 1)
    c0, c1 = max(0, c - rad), min(mask.shape[1], c + rad + 1)
    sub = mask[r0:r1, c0:c1]
    if not sub.any():
        return np.inf
    rs, cs = np.nonzero(sub)
    d = np.hypot((rs + r0) - r, (cs + c0) - c) * px
    return float(d.min())


def _spearman(a, b):
    """Rank correlation, average ranks for ties. Avoids a scipy dependency."""
    def rank(x):
        order = np.argsort(x, kind='mergesort')
        rk = np.empty(len(x), float)
        rk[order] = np.arange(1, len(x) + 1)
        _, inv, cnt = np.unique(x, return_inverse=True, return_counts=True)
        if (cnt > 1).any():                      # average tied ranks
            sums = np.zeros(len(cnt))
            np.add.at(sums, inv, rk)
            rk = (sums / cnt)[inv]
        return rk
    ra, rb = rank(np.asarray(a, float)), rank(np.asarray(b, float))
    return float(np.corrcoef(ra, rb)[0, 1])

XLSX = os.path.join(INPUT_DIR, "Makilala Geotechnical Data.xlsx")
PX = abs(ref_transform.a)                      # metres per pixel
TH = REF['theta']
FOS = REF['fos']

df = pd.read_excel(XLSX, sheet_name=0, header=1)
df.columns = [str(c).strip() for c in df.columns]


def col(*cands):
    for c in df.columns:
        u = c.upper()
        for cand in cands:
            if cand in u:
                return c
    raise KeyError(cands)


C_LAT, C_LON = col("LATITUDE"), col("LONGITUDE")
C_TYPE = col("SITE TYPE")
C_W, C_G = col("WATER CONTENT"), col("BULK UNIT WEIG")
C_SAND, C_FINES, C_PI = col("SAND %"), col("SILT/CLAY"), col("PI")

df = df[pd.to_numeric(df[C_LAT], errors='coerce').notna()].copy()
for c in (C_LAT, C_LON, C_W, C_G, C_SAND, C_FINES, C_PI):
    df[c] = pd.to_numeric(df[c], errors='coerce')
is_ls = df[C_TYPE].astype(str).str.upper().str.startswith('LANDSLIDE').values

print("=" * 76)
print("SECTION 6.5 — POINT-LEVEL VALIDATION")
print("=" * 76)
print(f"  spreadsheet rows with coordinates : {len(df)}   (paper: 43 sites)")
print(f"    landslide / non-landslide       : {is_ls.sum()} / {(~is_ls).sum()}")

# --- locate every site on the reference grid --------------------------------
xs, ys = warp_transform("EPSG:4326", ref_crs,
                        df[C_LON].tolist(), df[C_LAT].tolist())
inv = ~ref_transform
cc, rr = [], []
for x, y in zip(xs, ys):
    c, r = inv * (x, y)
    cc.append(int(np.floor(c)))
    rr.append(int(np.floor(r)))
rr, cc = np.array(rr), np.array(cc)
on_grid = (rr >= 0) & (rr < ref_shape[0]) & (cc >= 0) & (cc < ref_shape[1])
print(f"  sites falling on the reference grid: {int(on_grid.sum())}")

R, C = rr[on_grid], cc[on_grid]
LS = is_ls[on_grid]
sub = df[on_grid].reset_index(drop=True)

in_dom = SCORED[R, C]
print(f"  sites inside the scored domain    : {int(in_dom.sum())}   (paper: 36 of 43)")
out = ~in_dom
if out.any():
    a_out = P.alpha[R[out], C[out]]
    print(f"    the {int(out.sum())} excluded: mean slope "
          f"{np.nanmean(a_out[LS[out]]):.1f} deg landslide / "
          f"{np.nanmean(a_out[~LS[out]]):.1f} deg non-landslide"
          f"   (paper: 5.4 / 6.4)")

# --- co-location of landslide sites with the inventory ----------------------
print("\n-- co-location with the mapped inventory --")
d = np.array([_nearest_observed_m(r, c, OBSERVED, PX)
              for r, c in zip(R, C)])
d_ls = d[LS]
print(f"  landslide sites considered        : {len(d_ls)}   (paper: 25)")
print(f"  inside a mapped polygon (d = 0)   : {int((d_ls == 0).sum())}   (paper: 3)")
print(f"  within 25 m                       : {int((d_ls <= 25).sum())}   (paper: 15)")
print(f"  within 50 m                       : {int((d_ls <= 50).sum())}   (paper: 19)")
print(f"  median separation                 : {np.median(d_ls):.1f} m   (paper: 25)")

# --- slope contrast within the evaluated domain -----------------------------
a_in = P.alpha[R[in_dom], C[in_dom]]
l_in = LS[in_dom]
print(f"\n  slope, landslide sites in domain  : {np.nanmean(a_in[l_in]):.1f} deg"
      f"   (paper: 25.1)")
print(f"  slope, non-landslide in domain    : {np.nanmean(a_in[~l_in]):.1f} deg"
      f"   (paper: 18.9)")


def score(fos_vals, truth, label, pub=None):
    ok_ = np.isfinite(fos_vals)
    f, t = fos_vals[ok_], truth[ok_]
    pred = f < TH
    TPn = int((pred & t).sum()); FNn = int((~pred & t).sum())
    FPn = int((pred & ~t).sum()); TNn = int((~pred & ~t).sum())
    hit = TPn / max(TPn + FNn, 1)
    far = FPn / max(FPn + TNn, 1)
    prec = TPn / max(TPn + FPn, 1)
    print(f"\n  {label}  (n = {len(f)}: {int(t.sum())} positive)")
    print(f"    TP {TPn}  FN {FNn}  FP {FPn}  TN {TNn}")
    print(f"    hit {100*hit:.1f}%   FAR {100*far:.1f}%   J {hit-far:.3f}"
          f"   precision {100*prec:.1f}%")
    print(f"    mean FoS  landslide {np.mean(f[t]):.3f}  non-landslide {np.mean(f[~t]):.3f}")
    if pub:
        print(f"    paper: {pub}")
    return f, t, ok_


# --- scoring 1: the map as deployed -----------------------------------------
print("\n-- scoring the FoS raster at each site --")
fos_map = np.full(len(R), np.nan)
fos_map[in_dom] = FOS[R[in_dom], C[in_dom]]
f_map, t_map, ok_map = score(fos_map, LS, "FoS sampled from the map",
                             "hit 40.9%, FAR 14.3%, J 0.266, precision 81.8%, "
                             "mean FoS 1.179 / 1.261")

# --- scoring 2: the site's own measured index properties --------------------
print("\n-- scoring from measured index properties (same correlations) --")
phi_m = estimate_phi(sub[C_PI].values, sub[C_SAND].values, sub[C_FINES].values)
# The map being compared against is the PRIMARY configuration, whose cohesion
# field carries the +10 kPa offset. Scoring measured properties without it
# compares two different models, not two ways of obtaining the same one: the
# missing 10 kPa costs about 10/(gamma z sin a cos a) ~ 0.4 of FoS at a typical
# site, which is the whole of the discrepancy it produces.
_C_OFF = C_SCENARIOS[PRIMARY_C]
if _C_OFF is None:
    raise RuntimeError("primary scenario is multiplicative; adjust this line")
c_m = estimate_c(sub[C_FINES].values, sub[C_SAND].values) + _C_OFF
print(f"  (measured cohesion carries the primary offset, +{_C_OFF:.0f} kPa)")
m_m = estimate_m(sub[C_W].values)
g_m = sub[C_G].values

fos_meas = np.full(len(R), np.nan)
# Screen on the MEASURED COLUMNS, not on the estimator outputs. estimate_phi,
# estimate_c and estimate_m all substitute a default when their input is
# missing (30 deg, 5 kPa, m = 0.50) and never return NaN, so testing their
# outputs for finiteness screens nothing and silently admits sites scored on
# defaults rather than on laboratory data. This is Section 6.5's "laboratory
# completeness screen".
_have = {"PI": np.isfinite(sub[C_PI].values),
         "sand": np.isfinite(sub[C_SAND].values),
         "fines": np.isfinite(sub[C_FINES].values),
         "water": np.isfinite(sub[C_W].values),
         "unit weight": np.isfinite(sub[C_G].values)}
print("  measured values present, of the %d sites in the scored domain:"
      % int(in_dom.sum()))
for k, v in _have.items():
    print(f"    {k:<12} {int((v & in_dom).sum()):>3}")
complete = np.ones(len(R), bool)
for v in _have.values():
    complete &= v

# Section 6.5 says the laboratory completeness screen "reduced the
# interpolation set to 37" -- a named subset, which the spreadsheet's
# Availability column should mark. Try the candidate screens and report which
# one yields the 27 the paper reports.
try:
    C_AV = col("AVAILABILITY")
    av = sub[C_AV].astype(str).str.strip().str.upper().str.startswith('Y').values
    print(f"  Availability = YES, over all {len(sub)} sites: {int(av.sum())}"
          f"   (Section 6.5 implies 37)")
except KeyError:
    av = None
    print("  (no Availability column found)")

print("  candidate screens:")
cands = [("in domain", in_dom),
         ("in domain & complete on five", in_dom & complete)]
if av is not None:
    cands += [("in domain & Availability", in_dom & av),
              ("in domain & Availability & complete", in_dom & av & complete)]
for lab, msk in cands:
    flag = "  <-- matches the paper's 27" if int(msk.sum()) == 27 else ""
    print(f"    {lab:<38}{int(msk.sum()):>3}{flag}")

sel = in_dom & complete if av is None else in_dom & av & complete
print(f"  using: {int(sel.sum())} sites   (paper: 27)")
if sel.any():
    fos_meas[sel] = compute_FoS(c_m[sel], phi_m[sel],
                                P.alpha[R[sel], C[sel]], m_m[sel], g_m[sel],
                                np.clip(P.z_raw[R[sel], C[sel]], 0.1, 5.0),
                                P.k_h[R[sel], C[sel]])
score(fos_meas, LS, "FoS from measured properties", "J 0.097")

# --- the sites scorable both ways -------------------------------------------
both = np.isfinite(fos_map) & np.isfinite(fos_meas)
print(f"\n-- the {int(both.sum())} sites scorable both ways   (paper: 27) --")
if both.sum() > 2:
    a, b = fos_meas[both], fos_map[both]
    diff = a - b
    rho = _spearman(a, b)
    nd = int(((a < TH) != (b < TH)).sum())
    print(f"    FoS difference (measured - map): mean {diff.mean():+.3f}"
          f"   sd {diff.std(ddof=1):.3f}   (paper: +0.037, sd 0.137)")
    print(f"    Spearman rho                   : {rho:.3f}   (paper: 0.935)")
    print(f"    classified differently         : {nd} of {int(both.sum())}"
          f"   (paper: 3 of 27)")

    print("\n    measured vs interpolated at those sites:")
    # P keeps cohesion and saturation, not the fines and sand percentages they
    # were derived from -- those rasters are dropped after use to save memory.
    # Reporting P.c against measured fines would compare kPa with per cent.
    for lab, meas, ras, pub in [
            ("fines %", sub[C_FINES].values, None, "11.35 vs 10.96"),
            ("sand %", sub[C_SAND].values, None, "84.99 vs 85.13"),
            ("water %", sub[C_W].values, P.water, "40.47 vs 40.61"),
            ("unit weight", sub[C_G].values, P.gamma, "12.69 vs 12.47")]:
        if ras is None:
            print(f"      {lab:<12} measured {np.nanmean(meas[both]):.2f}"
                  f"   (raster not retained; paper {pub})")
            continue
        iv = ras[R[both], C[both]]
        mv = meas[both]
        good = np.isfinite(iv) & np.isfinite(mv)
        print(f"      {lab:<12} measured {mv[good].mean():.2f}"
              f"   interpolated {iv[good].mean():.2f}"
              f"   MAD {np.abs(mv[good]-iv[good]).mean():.1f}   (paper {pub})")

print("\n" + "=" * 76)
print("Compare each line against Section 6.5. Because the section's original")
print("computation cannot be located, treat DISAGREEMENTS as the paper needing")
print("the new value, not as this cell being wrong -- but check the site counts")
print("first, since those determine everything downstream.")
print("=" * 76)

SECTION 6.5 — POINT-LEVEL VALIDATION
  spreadsheet rows with coordinates : 43   (paper: 43 sites)
    landslide / non-landslide       : 25 / 18
  sites falling on the reference grid: 43
  sites inside the scored domain    : 36   (paper: 36 of 43)
    the 7 excluded: mean slope 5.4 deg landslide / 6.4 deg non-landslide   (paper: 5.4 / 6.4)

-- co-location with the mapped inventory --
  landslide sites considered        : 25   (paper: 25)
  inside a mapped polygon (d = 0)   : 3   (paper: 3)
  within 25 m                       : 14   (paper: 15)
  within 50 m                       : 18   (paper: 19)
  median separation                 : 25.0 m   (paper: 25)

  slope, landslide sites in domain  : 25.1 deg   (paper: 25.1)
  slope, non-landslide in domain    : 18.9 deg   (paper: 18.9)

-- scoring the FoS raster at each site --

  FoS sampled from the map  (n = 36: 22 positive)
    TP 9  FN 13  FP 2  TN 12
    hit 40.9%   FAR 14.3%   J 0.266   precision 81.8%
    mean FoS  landslide 1.179  no

In [ ]:
# ============================================================================
#  §22c  POINT-LEVEL VALIDATION — Section 6.5, rebuilt on the v10 pipeline
#
#  Section 6.5 reports roughly twenty-five figures and NONE of them can be
#  traced to a surviving artifact. Makilala_Point_Validation.ipynb (1 Aug) is
#  not their source: it uses a FoS threshold of 0.8 against the paper's 0.90,
#  carries 40 samples against 43, reads Slope_PRSTM51N.tif and the raw Mak_*
#  rasters with no registration correction, and refers to "v5 results". Its
#  numbers appear nowhere in the paper. Whatever produced Section 6.5 is gone.
#
#  This cell recomputes the section from the field spreadsheet against the
#  corrected, aligned rasters, reusing the pipeline's own correlation
#  functions so that the "measured properties" scoring tests the correlations
#  and not a second implementation of them.
#
#  Paste the whole output back.
# ============================================================================
#  NB 24 Nov 2026: the '(paper: ...)' literals below are COMPARISON TARGETS and
#  they go stale when the manuscript is revised. They are now synchronised with
#  the submitted Section 6.5, which adopted this cell's own rebuilt 31-site
#  figures. An external audit read the previous, stale literals and reported
#  them as the manuscript's values; update them whenever Section 6.5 changes.

guard()

import os
import numpy as np
import pandas as pd
from rasterio.warp import transform as warp_transform


def _nearest_observed_m(r, c, mask, px, max_m=600.0):
    """Distance in metres from (r, c) to the nearest True cell of `mask`.

    A windowed search rather than a full distance transform: there are 43
    points, and an exact EDT over 21 million pixels to serve them would be
    absurd. Returns np.inf if nothing is found inside the window.
    """
    rad = int(np.ceil(max_m / px))
    r0, r1 = max(0, r - rad), min(mask.shape[0], r + rad + 1)
    c0, c1 = max(0, c - rad), min(mask.shape[1], c + rad + 1)
    sub = mask[r0:r1, c0:c1]
    if not sub.any():
        return np.inf
    rs, cs = np.nonzero(sub)
    d = np.hypot((rs + r0) - r, (cs + c0) - c) * px
    return float(d.min())


def _spearman(a, b):
    """Rank correlation, average ranks for ties. Avoids a scipy dependency."""
    def rank(x):
        order = np.argsort(x, kind='mergesort')
        rk = np.empty(len(x), float)
        rk[order] = np.arange(1, len(x) + 1)
        _, inv, cnt = np.unique(x, return_inverse=True, return_counts=True)
        if (cnt > 1).any():                      # average tied ranks
            sums = np.zeros(len(cnt))
            np.add.at(sums, inv, rk)
            rk = (sums / cnt)[inv]
        return rk
    ra, rb = rank(np.asarray(a, float)), rank(np.asarray(b, float))
    return float(np.corrcoef(ra, rb)[0, 1])

XLSX = os.path.join(INPUT_DIR, "Makilala Geotechnical Data.xlsx")
PX = abs(ref_transform.a)                      # metres per pixel
TH = REF['theta']
FOS = REF['fos']

df = pd.read_excel(XLSX, sheet_name=0, header=1)
df.columns = [str(c).strip() for c in df.columns]


def col(*cands):
    for c in df.columns:
        u = c.upper()
        for cand in cands:
            if cand in u:
                return c
    raise KeyError(cands)


C_LAT, C_LON = col("LATITUDE"), col("LONGITUDE")
C_TYPE = col("SITE TYPE")
C_W, C_G = col("WATER CONTENT"), col("BULK UNIT WEIG")
C_SAND, C_FINES, C_PI = col("SAND %"), col("SILT/CLAY"), col("PI")

df = df[pd.to_numeric(df[C_LAT], errors='coerce').notna()].copy()
for c in (C_LAT, C_LON, C_W, C_G, C_SAND, C_FINES, C_PI):
    df[c] = pd.to_numeric(df[c], errors='coerce')
is_ls = df[C_TYPE].astype(str).str.upper().str.startswith('LANDSLIDE').values

print("=" * 76)
print("SECTION 6.5 — POINT-LEVEL VALIDATION")
print("=" * 76)
print(f"  spreadsheet rows with coordinates : {len(df)}   (paper: 43 sites)")
print(f"    landslide / non-landslide       : {is_ls.sum()} / {(~is_ls).sum()}")

# --- locate every site on the reference grid --------------------------------
xs, ys = warp_transform("EPSG:4326", ref_crs,
                        df[C_LON].tolist(), df[C_LAT].tolist())
inv = ~ref_transform
cc, rr = [], []
for x, y in zip(xs, ys):
    c, r = inv * (x, y)
    cc.append(int(np.floor(c)))
    rr.append(int(np.floor(r)))
rr, cc = np.array(rr), np.array(cc)
on_grid = (rr >= 0) & (rr < ref_shape[0]) & (cc >= 0) & (cc < ref_shape[1])
print(f"  sites falling on the reference grid: {int(on_grid.sum())}")

R, C = rr[on_grid], cc[on_grid]
LS = is_ls[on_grid]
sub = df[on_grid].reset_index(drop=True)

in_dom = SCORED[R, C]
print(f"  sites inside the scored domain    : {int(in_dom.sum())}   (paper: 36 of 43)")
out = ~in_dom
if out.any():
    a_out = P.alpha[R[out], C[out]]
    print(f"    the {int(out.sum())} excluded: mean slope "
          f"{np.nanmean(a_out[LS[out]]):.1f} deg landslide / "
          f"{np.nanmean(a_out[~LS[out]]):.1f} deg non-landslide"
          f"   (paper: 5.4 / 6.4)")

# --- co-location of landslide sites with the inventory ----------------------
print("\n-- co-location with the mapped inventory --")
d = np.array([_nearest_observed_m(r, c, OBSERVED, PX)
              for r, c in zip(R, C)])
d_ls = d[LS]
print(f"  landslide sites considered        : {len(d_ls)}   (paper: 25)")
print(f"  inside a mapped polygon (d = 0)   : {int((d_ls == 0).sum())}   (paper: 3)")
print(f"  within 25 m                       : {int((d_ls <= 25).sum())}   (paper: 14)")
print(f"  within 50 m                       : {int((d_ls <= 50).sum())}   (paper: 18)")
print(f"  median separation                 : {np.median(d_ls):.1f} m   (paper: 25)")

# --- slope contrast within the evaluated domain -----------------------------
a_in = P.alpha[R[in_dom], C[in_dom]]
l_in = LS[in_dom]
print(f"\n  slope, landslide sites in domain  : {np.nanmean(a_in[l_in]):.1f} deg"
      f"   (paper: 25.1)")
print(f"  slope, non-landslide in domain    : {np.nanmean(a_in[~l_in]):.1f} deg"
      f"   (paper: 18.9)")


def score(fos_vals, truth, label, pub=None):
    ok_ = np.isfinite(fos_vals)
    f, t = fos_vals[ok_], truth[ok_]
    pred = f < TH
    TPn = int((pred & t).sum()); FNn = int((~pred & t).sum())
    FPn = int((pred & ~t).sum()); TNn = int((~pred & ~t).sum())
    hit = TPn / max(TPn + FNn, 1)
    far = FPn / max(FPn + TNn, 1)
    prec = TPn / max(TPn + FPn, 1)
    print(f"\n  {label}  (n = {len(f)}: {int(t.sum())} positive)")
    print(f"    TP {TPn}  FN {FNn}  FP {FPn}  TN {TNn}")
    print(f"    hit {100*hit:.1f}%   FAR {100*far:.1f}%   J {hit-far:.3f}"
          f"   precision {100*prec:.1f}%")
    print(f"    mean FoS  landslide {np.mean(f[t]):.3f}  non-landslide {np.mean(f[~t]):.3f}")
    if pub:
        print(f"    paper: {pub}")
    return f, t, ok_


# --- scoring 1: the map as deployed -----------------------------------------
print("\n-- scoring the FoS raster at each site --")
fos_map = np.full(len(R), np.nan)
fos_map[in_dom] = FOS[R[in_dom], C[in_dom]]
f_map, t_map, ok_map = score(fos_map, LS, "FoS sampled from the map",
                             "hit 40.9%, FAR 14.3%, J 0.266, precision 81.8%, "
                             "mean FoS 1.179 / 1.261")

# --- scoring 2: the site's own measured index properties --------------------
print("\n-- scoring from measured index properties (same correlations) --")
phi_m = estimate_phi(sub[C_PI].values, sub[C_SAND].values, sub[C_FINES].values)
# The map being compared against is the PRIMARY configuration, whose cohesion
# field carries the +10 kPa offset. Scoring measured properties without it
# compares two different models, not two ways of obtaining the same one: the
# missing 10 kPa costs about 10/(gamma z sin a cos a) ~ 0.4 of FoS at a typical
# site, which is the whole of the discrepancy it produces.
_C_OFF = C_SCENARIOS[PRIMARY_C]
if _C_OFF is None:
    raise RuntimeError("primary scenario is multiplicative; adjust this line")
c_m = estimate_c(sub[C_FINES].values, sub[C_SAND].values) + _C_OFF
print(f"  (measured cohesion carries the primary offset, +{_C_OFF:.0f} kPa)")
m_m = estimate_m(sub[C_W].values)
g_m = sub[C_G].values

fos_meas = np.full(len(R), np.nan)
# Screen on the MEASURED COLUMNS, not on the estimator outputs. estimate_phi,
# estimate_c and estimate_m all substitute a default when their input is
# missing (30 deg, 5 kPa, m = 0.50) and never return NaN, so testing their
# outputs for finiteness screens nothing and silently admits sites scored on
# defaults rather than on laboratory data. This is Section 6.5's "laboratory
# completeness screen".
_have = {"PI": np.isfinite(sub[C_PI].values),
         "sand": np.isfinite(sub[C_SAND].values),
         "fines": np.isfinite(sub[C_FINES].values),
         "water": np.isfinite(sub[C_W].values),
         "unit weight": np.isfinite(sub[C_G].values)}
print("  measured values present, of the %d sites in the scored domain:"
      % int(in_dom.sum()))
for k, v in _have.items():
    print(f"    {k:<12} {int((v & in_dom).sum()):>3}")
complete = np.ones(len(R), bool)
for v in _have.values():
    complete &= v

# Section 6.5 says the laboratory completeness screen "reduced the
# interpolation set to 37" -- a named subset, which the spreadsheet's
# Availability column should mark. Try the candidate screens and report which
# one yields the 27 the paper reports.
try:
    C_AV = col("AVAILABILITY")
    av = sub[C_AV].astype(str).str.strip().str.upper().str.startswith('Y').values
    print(f"  Availability = YES, over all {len(sub)} sites: {int(av.sum())}"
          f"   (Section 6.5 implies 37)")
except KeyError:
    av = None
    print("  (no Availability column found)")

# The five columns above are the ones the FoS needs. The campaign also recorded
# LL, PL and gravel; if the original screen demanded a COMPLETE laboratory
# record rather than merely the inputs, that would exclude more sites.
_extra = {}
for nm, pat in [("LL", "LL"), ("PL", "PL"), ("gravel", "GRAVEL")]:
    try:
        cc_ = col(pat)
        _extra[nm] = np.isfinite(pd.to_numeric(sub[cc_], errors='coerce').values)
    except KeyError:
        pass
if _extra:
    print("  also recorded, of the 36 in domain:")
    for k, v in _extra.items():
        print(f"    {k:<12} {int((v & in_dom).sum()):>3}")
full_lab = complete.copy()
for v in _extra.values():
    full_lab &= v

print("  candidate screens:")
cands = [("in domain", in_dom),
         ("in domain & complete on five", in_dom & complete),
         ("in domain & full laboratory record", in_dom & full_lab)]
if av is not None:
    cands += [("in domain & Availability & complete", in_dom & av & complete),
              ("in domain & Availability & full lab", in_dom & av & full_lab)]
hit27 = None
for lab, msk in cands:
    n_ = int(msk.sum())
    flag = ""
    if n_ == 27 and hit27 is None:
        flag = "  <-- matches the paper's 27"
        hit27 = msk
    print(f"    {lab:<38}{n_:>3}{flag}")

sel = hit27 if hit27 is not None else (in_dom & complete)
print(f"  using: {int(sel.sum())} sites"
      f"   {'(the screen that reproduces 27)' if hit27 is not None else '(paper: 31)'}")
if sel.any():
    fos_meas[sel] = compute_FoS(c_m[sel], phi_m[sel],
                                P.alpha[R[sel], C[sel]], m_m[sel], g_m[sel],
                                np.clip(P.z_raw[R[sel], C[sel]], 0.1, 5.0),
                                P.k_h[R[sel], C[sel]])
score(fos_meas, LS, "FoS from measured properties", "J 0.097")

# --- the sites scorable both ways -------------------------------------------
both = np.isfinite(fos_map) & np.isfinite(fos_meas)
print(f"\n-- the {int(both.sum())} sites scorable both ways   (paper: 31) --")
if both.sum() > 2:
    a, b = fos_meas[both], fos_map[both]
    diff = a - b
    rho = _spearman(a, b)
    nd = int(((a < TH) != (b < TH)).sum())
    print(f"    FoS difference (measured - map): mean {diff.mean():+.3f}"
          f"   sd {diff.std(ddof=1):.3f}   (paper: +0.036, sd 0.138)")
    print(f"    Spearman rho                   : {rho:.3f}   (paper: 0.934)")
    print(f"    classified differently         : {nd} of {int(both.sum())}"
          f"   (paper: 4 of 31)")

    print("\n    measured vs interpolated at those sites:")
    # P keeps cohesion and saturation, not the fines and sand percentages they
    # were derived from. Recover them from `aligned` if it survived, else read
    # them off disk -- reporting P.c against measured fines would compare kPa
    # with per cent.
    _fines_r = _sand_r = None
    if 'aligned' in globals() and isinstance(aligned, dict):
        _fines_r, _sand_r = aligned.get('fines'), aligned.get('sand')
    if _fines_r is None or _sand_r is None:
        try:
            import rasterio
            from rasterio.warp import reproject, Resampling

            def _ld(fn):
                with rasterio.open(os.path.join(INPUT_DIR, fn)) as s_:
                    a_ = s_.read(1).astype(np.float64)
                    if s_.nodata is not None:
                        a_[a_ == s_.nodata] = np.nan
                    if a_.shape == ref_shape and s_.transform.almost_equals(
                            ref_transform, precision=1e-6):
                        return a_
                    d_ = np.full(ref_shape, np.nan)
                    reproject(a_, d_, src_transform=s_.transform, src_crs=s_.crs,
                              dst_transform=ref_transform, dst_crs=ref_crs,
                              resampling=Resampling.bilinear,
                              src_nodata=np.nan, dst_nodata=np.nan)
                    return d_
            _fines_r, _sand_r = _ld(FILE_FINES), _ld(FILE_SAND)
        except Exception as e:                       # noqa: BLE001
            print(f"      (could not load fines/sand rasters: {e})")

    for lab, meas, ras, pub in [
            ("fines %", sub[C_FINES].values, _fines_r, "11.35 vs 10.96"),
            ("sand %", sub[C_SAND].values, _sand_r, "84.99 vs 85.13"),
            ("water %", sub[C_W].values, P.water, "40.47 vs 40.61"),
            ("unit weight", sub[C_G].values, P.gamma, "12.69 vs 12.47")]:
        if ras is None:
            print(f"      {lab:<12} measured {np.nanmean(meas[both]):.2f}"
                  f"   (raster not retained; paper {pub})")
            continue
        iv = ras[R[both], C[both]]
        mv = meas[both]
        good = np.isfinite(iv) & np.isfinite(mv)
        print(f"      {lab:<12} measured {mv[good].mean():.2f}"
              f"   interpolated {iv[good].mean():.2f}"
              f"   MAD {np.abs(mv[good]-iv[good]).mean():.1f}   (paper {pub})")

print("\n" + "=" * 76)
print("Compare each line against Section 6.5. Because the section's original")
print("computation cannot be located, treat DISAGREEMENTS as the paper needing")
print("the new value, not as this cell being wrong -- but check the site counts")
print("first, since those determine everything downstream.")
print("=" * 76)

In [26]:
# ============================================================================
#  §24  DEPTH SCALING OF THE COHESION-THRESHOLD DERIVATIVE  ->  Section 7.2
#  (v2 -- corrects two errors in v1; see NOTE ON v1 at the foot of this header)
#
#  *** THIS IS A NOTEBOOK CELL, NOT A SCRIPT. ***
#  Paste it as a new cell at the end of Makilala_FoS_Pipeline_v10.ipynb and run
#  it in the SAME KERNEL that already holds P, SCORED and guard().
#
#  WHAT THIS SETTLES
#  Section 7.2 states that theta* responds to cohesion more strongly at shallow
#  depth caps than at deep ones. Regressing theta* on mean cohesion across the
#  four scenarios of Table 4 gives
#
#      dtheta*/dc'  =  0.0204, 0.0355, 0.0516  kPa^-1   at 20, 5 and 3 m
#
#  a ratio of 2.53 between the extreme caps. The closed form should predict it.
#  theta* is a threshold on the PSEUDO-STATIC FoS, which the pipeline computes as
#
#      FoS = [c' + (gamma - m*gamma_w) z cos^2(a) tan(phi)] / D
#      D   = gamma z sin(a) cos(a) + k_h gamma z cos^2(a)
#          = gamma z cos(a) [ sin(a) + k_h cos(a) ]
#
#  so   dFoS/dc' = 1/D   [per kPa],
#
#  and the depth cap enters only through z. The ratio between two caps is
#  therefore the ratio of the DOMAIN AVERAGE of 1/D. Averaging 1/z is not the
#  same as inverting the average z -- Jensen makes the first strictly larger --
#  and the gap is widest at the 20 m cap, where the depth distribution is broad.
#  That is precisely why a ratio computed from mean depths alone (which gives
#  3.9) is the wrong quantity, and why this cell exists.
#
#  WHAT TO COMPARE
#  The RATIOS between caps are what Section 7.2 claims. The absolute values are
#  reported too, but theta* tracks the Youden optimum, which need not move by
#  the full distributional shift, so treat any absolute agreement as a bonus
#  rather than as the test.
#
#  SELF-CHECK
#  Before reporting anything, the cell verifies 1/D against a finite difference
#  of the pipeline's own compute_FoS at c' and c'+1 kPa. If the analytic
#  derivative and the pipeline disagree, the cell stops. This is the step that
#  would have caught the v1 error below.
#
#  NOTE ON v1, both mine:
#    (1) it assumed P.alpha was in radians; it is in DEGREES (ALPHA_MIN = 10,
#        and compute_FoS calls np.radians internally). The unit guard caught it.
#    (2) more seriously, it used the STATIC denominator gamma z sin(a) cos(a),
#        omitting the k_h term. theta* is a threshold on the pseudo-static FoS,
#        so the seismic term belongs in D. That one would NOT have raised an
#        error -- it would have printed plausible wrong numbers.
#
#  Paste the whole output back.
# ============================================================================
guard()

import numpy as np

MEASURED = {20.0: 0.0204, 5.0: 0.0355, 3.0: 0.0516}   # regression on Table 4
CAPS = [20.0, 5.0, 3.0]

_a_deg = P.alpha[SCORED]
_g = P.gamma[SCORED]
_zr = P.z_raw[SCORED]
_kh = P.k_h[SCORED]

_ok = (np.isfinite(_a_deg) & np.isfinite(_g) & np.isfinite(_zr)
       & np.isfinite(_kh) & (_g > 0) & (_zr > 0))
a_deg, g, zr, kh = _a_deg[_ok], _g[_ok], _zr[_ok], _kh[_ok]

if np.nanmax(a_deg) < 1.6:
    raise RuntimeError("P.alpha looks like radians, not degrees -- check units")
if np.nanmin(a_deg) < ALPHA_MIN - 1e-6:
    raise RuntimeError(f"alpha below ALPHA_MIN={ALPHA_MIN} inside SCORED")

a = np.radians(a_deg)
ca, sa = np.cos(a), np.sin(a)


def dFoS_dc(z):
    """Analytic dFoS/dc' per kPa, pseudo-static denominator."""
    return 1.0 / (g * z * ca * (sa + kh * ca))


# ---- self-check against the pipeline's own compute_FoS ----------------------
_zt = np.clip(zr, 0.1, 5.0)
_c0 = P.c[SCORED][_ok]
_f0 = compute_FoS(_c0, P.phi[SCORED][_ok], a_deg, P.m[SCORED][_ok], g, _zt, kh)
_f1 = compute_FoS(_c0 + 1.0, P.phi[SCORED][_ok], a_deg, P.m[SCORED][_ok], g, _zt, kh)
_fd = _f1 - _f0                       # exact: FoS is linear in c'
_an = dFoS_dc(_zt)
_fin = np.isfinite(_fd) & np.isfinite(_an)
_err = np.abs(_fd[_fin] - _an[_fin]) / np.abs(_an[_fin])
print(f"self-check  max relative error {_err.max():.2e}  on {_fin.sum():,} px")
if _err.max() > 1e-9:
    raise RuntimeError("analytic dFoS/dc' disagrees with compute_FoS -- stop")
print("            analytic derivative agrees with the pipeline\n")

print("=" * 82)
print("DEPTH SCALING OF dFoS/dc'  (Section 7.2)")
print("=" * 82)
print(f"  scored pixels used : {a.size:,}")
print(f"  mean gamma {g.mean():7.3f} kN/m3   mean alpha {a_deg.mean():6.2f} deg"
      f"   mean k_h {kh.mean():6.4f}")
print()
print(f"  {'cap':>5} {'mean z':>8} {'mean 1/z':>9} {'1/mean z':>9}"
      f" {'mean dF/dc':>11} {'median dF/dc':>13} {'measured':>9}")
print(f"  {'(m)':>5} {'(m)':>8} {'(1/m)':>9} {'(1/m)':>9}"
      f" {'(1/kPa)':>11} {'(1/kPa)':>13} {'(1/kPa)':>9}")
print("  " + "-" * 72)

res = {}
for cap in CAPS:
    z = np.clip(zr, 0.1, cap)
    D = dFoS_dc(z)
    res[cap] = dict(mz=z.mean(), m_inv_z=(1.0 / z).mean(), inv_mz=1.0 / z.mean(),
                    mD=float(D.mean()), medD=float(np.median(D)))
    r = res[cap]
    print(f"  {cap:5.0f} {r['mz']:8.3f} {r['m_inv_z']:9.4f} {r['inv_mz']:9.4f}"
          f" {r['mD']:11.4f} {r['medD']:13.4f} {MEASURED[cap]:9.4f}")

print()
print("=" * 82)
print("PREDICTED vs MEASURED RATIOS")
print("=" * 82)
print(f"  {'pair':>12} {'measured':>9} {'mean dF/dc':>12} {'median dF/dc':>14}"
      f" {'mean-z only':>13}")
print("  " + "-" * 64)
for lo, hi in [(3.0, 20.0), (5.0, 20.0), (3.0, 5.0)]:
    print(f"  {f'{lo:.0f}m / {hi:.0f}m':>12} {MEASURED[lo]/MEASURED[hi]:9.2f}"
          f" {res[lo]['mD']/res[hi]['mD']:12.2f}"
          f" {res[lo]['medD']/res[hi]['medD']:14.2f}"
          f" {res[hi]['mz']/res[lo]['mz']:13.2f}")

print()
print("  'mean-z only' is the naive prediction the paper currently quotes as 3.9.")
print("  It is the column NOT to use; it ignores the spread of z at the 20 m cap.")
for lab, key in [("mean", 'mD'), ("median", 'medD')]:
    pr = res[3.0][key] / res[20.0][key]
    print(f"\n  extreme-cap ratio, {lab:6} : predicted {pr:5.2f}   measured 2.53"
          f"   gap {100*abs(pr-2.53)/2.53:4.0f}%")
print("=" * 82)

self-check  max relative error 6.07e-15  on 7,104,440 px
            analytic derivative agrees with the pipeline

DEPTH SCALING OF dFoS/dc'  (Section 7.2)
  scored pixels used : 7,104,440
  mean gamma  11.900 kN/m3   mean alpha  23.00 deg   mean k_h 0.1338

    cap   mean z  mean 1/z  1/mean z  mean dF/dc  median dF/dc  measured
    (m)      (m)     (1/m)     (1/m)     (1/kPa)       (1/kPa)   (1/kPa)
  ------------------------------------------------------------------------
     20   11.484    0.1180    0.0871      0.0224        0.0174    0.0204
      5    4.886    0.2164    0.2047      0.0418        0.0379    0.0355
      3    2.975    0.3443    0.3361      0.0666        0.0624    0.0516

PREDICTED vs MEASURED RATIOS
          pair  measured   mean dF/dc   median dF/dc   mean-z only
  ----------------------------------------------------------------
      3m / 20m      2.53         2.97           3.59          3.86
      5m / 20m      1.74         1.86           2.18          2.35
   

In [27]:
# ============================================================================
#  §25   THRESHOLD TRANSFER: IS THE "ZERO SPREAD" REAL, OR IS IT THE GRID?
#
#  Paste as a new cell in Makilala_FoS_Pipeline_v10.ipynb, AFTER cell §10
#  (spatially blocked validation). Uses only objects that cell already defines:
#  SCORED, OBSERVED, REF, THRESHOLDS, ref_shape, ref_transform, pos, neg.
#  Mutates nothing.
#
#  WHY THIS EXISTS
#  ---------------
#  Section 6.4 claims theta* is "identical on every one of the forty-five
#  held-out tiles, with zero spread", and calls that evidence "as strong as
#  this design can produce". Section 4.8 says of the same quantity that the
#  Youden maximum is flat, that every threshold in 0.883-0.911 is within 0.001
#  of the optimum, and that thresholds "are best read as representatives of
#  that plateau rather than as sharply determined values".
#
#  Both cannot stand. THRESHOLDS has a step of 0.05 and the plateau is ~0.028
#  wide -- narrower than the step. Any tile whose true optimum lies anywhere in
#  roughly 0.875-0.925 reports exactly 0.90. So "zero spread" is consistent
#  with genuine stability AND with a coarse grid over a flat maximum.
#
#  There is a second, independent weakness. Leave-one-block-out as implemented
#  selects theta* on every scored pixel outside the held-out tile. One 2 km
#  tile is a small share of 7.1 M pixels, so the training pool is almost
#  unchanged from fold to fold. A statistic that does not move when 2% of the
#  data is removed has not been tested very hard.
#
#  This cell answers both. Three diagnostics:
#    A  each tile's own optimum at CONTINUOUS resolution, and its plateau width
#    B  spatial K-fold selection, where the training pool really does change
#    C  the grid-vs-continuous comparison stated explicitly
#
#  HOW TO READ THE RESULT
#  ----------------------
#  If the 45 continuous maximisers cluster tightly (say IQR < 0.05) and the
#  K-fold thresholds hold at larger K, the claim is REAL and currently
#  understated: report the distribution and the title's first clause is earned.
#  If they scatter across the plateau, the claim must weaken to "no tile
#  selects a different grid point", and the title needs revisiting.
#  Either outcome is publishable. The present ambiguity is not.
# ============================================================================
import numpy as np

guard()

_TILE_PX   = int(round(2000 / abs(ref_transform.a)))
_MIN_BLOCK = 200
_fos = REF['fos']


def _exact_youden(p, q, tol=(0.001, 0.01)):
    """Exact Youden optimum over all real thresholds, from the sorted values.

    Mirrors the pooled calculation in Section 4.8: J changes only where theta
    crosses an observed FoS value, so evaluating immediately above each
    distinct value is exhaustive, not an approximation.

    Returns (theta_hat, J_max, {tol: (lo, hi, width)}).
    """
    cand = np.unique(np.concatenate([p, q]))
    hr = np.searchsorted(p, cand, 'right') / len(p)
    far = np.searchsorted(q, cand, 'right') / len(q)
    J = hr - far
    k = int(np.argmax(J))
    out = {}
    for t in tol:
        m = J >= J[k] - t
        out[t] = (float(cand[m].min()), float(cand[m].max()),
                  float(cand[m].max() - cand[m].min()))
    return float(cand[k]), float(J[k]), out


# ---------------------------------------------------------------- collect ---
_tiles = []                                   # (r0, c0, p, q)
for r0 in range(0, ref_shape[0], _TILE_PX):
    for c0 in range(0, ref_shape[1], _TILE_PX):
        sl = (slice(r0, r0 + _TILE_PX), slice(c0, c0 + _TILE_PX))
        s = SCORED[sl]
        if not s.any():
            continue
        f, o = _fos[sl], OBSERVED[sl]
        p = np.sort(f[s & o]); q = np.sort(f[s & ~o])
        if len(p) < _MIN_BLOCK or len(q) < _MIN_BLOCK:
            continue
        _tiles.append((r0, c0, p, q))

N = len(_tiles)
print("=" * 78)
print("A.  PER-TILE OPTIMUM AT CONTINUOUS RESOLUTION")
print("=" * 78)
print(f"scored tiles: {N}   (grid step of THRESHOLDS = "
      f"{THRESHOLDS[1]-THRESHOLDS[0]:.2f})\n")

_exact, _w001, _w010, _grid = [], [], [], []
for r0, c0, p, q in _tiles:
    th, Jm, wid = _exact_youden(p, q)
    _exact.append(th); _w001.append(wid[0.001][2]); _w010.append(wid[0.01][2])
    hr = np.searchsorted(p, THRESHOLDS, 'left') / len(p)
    far = np.searchsorted(q, THRESHOLDS, 'left') / len(q)
    _grid.append(THRESHOLDS[int(np.argmax(hr - far))])
_exact = np.array(_exact); _grid = np.array(_grid)
_w001 = np.array(_w001); _w010 = np.array(_w010)

_th = REF['theta']
_half = (THRESHOLDS[1] - THRESHOLDS[0]) / 2
print(f"  exact maximiser : median {np.median(_exact):.4f}   "
      f"IQR {np.percentile(_exact,25):.4f}-{np.percentile(_exact,75):.4f}")
print(f"                    range  {_exact.min():.4f}-{_exact.max():.4f}   "
      f"sd {_exact.std(ddof=1):.4f}")
print(f"  plateau width   : J within 0.001 of max -> median "
      f"{np.median(_w001):.4f}, max {_w001.max():.4f}")
print(f"                    J within 0.01  of max -> median "
      f"{np.median(_w010):.4f}, max {_w010.max():.4f}")
print(f"\n  tiles whose GRID optimum is {_th:.2f}            : "
      f"{int((_grid == _th).sum())}/{N}")
print(f"  tiles whose EXACT optimum is within +/-{_half:.3f} of {_th:.2f}: "
      f"{int((np.abs(_exact - _th) <= _half).sum())}/{N}")
print(f"  distinct grid optima across tiles           : "
      f"{sorted(set(np.round(_grid, 2)))}")
print("\n  READ: if the second count is much lower than the first, the zero")
print("  spread is the grid, not the physics.")

# ------------------------------------------------------------- K-fold ------
print("\n" + "=" * 78)
print("B.  SPATIAL K-FOLD SELECTION  (the training pool actually changes)")
print("=" * 78)
print("Tiles are sorted west to east and cut into K contiguous bands. theta*")
print("is selected on all scored pixels OUTSIDE a band and applied to it.")
print("K = 45 is leave-one-tile-out, i.e. what Section 6.4 currently reports.\n")

_order = np.argsort([c0 for _, c0, _, _ in _tiles])
_P = [_tiles[i][2] for i in _order]
_Q = [_tiles[i][3] for i in _order]
_gp = np.searchsorted(pos, THRESHOLDS, 'left')
_gq = np.searchsorted(neg, THRESHOLDS, 'left')

print(f"{'K':>4}  {'left out':>9}  {'theta* per fold':<34}  {'moved':>6}")
print("-" * 62)
for K in (2, 3, 5, 9, 15, N):
    edges = np.linspace(0, N, K + 1).astype(int)
    ths, frac = [], []
    for a, b in zip(edges[:-1], edges[1:]):
        if b <= a:
            continue
        hp = np.concatenate(_P[a:b]); hq = np.concatenate(_Q[a:b])
        cp = np.searchsorted(np.sort(hp), THRESHOLDS, 'left')
        cq = np.searchsorted(np.sort(hq), THRESHOLDS, 'left')
        HR = (_gp - cp) / (len(pos) - len(hp))
        FAR = (_gq - cq) / (len(neg) - len(hq))
        ths.append(THRESHOLDS[int(np.argmax(HR - FAR))])
        frac.append((len(hp) + len(hq)) / (len(pos) + len(neg)))
    ths = np.array(ths)
    shown = sorted(set(np.round(ths, 2)))
    label = f"{np.mean(frac)*100:.0f}%"
    print(f"{K:>4}  {label:>9}  {str(shown):<34}  "
          f"{int((ths != _th).sum())}/{len(ths)}")

print("\n  READ: theta* holding at K = 2 or 3, where a third to a half of the")
print("  domain is withheld, is a far stronger statement than it holding at")
print("  K = 45, where each fold withholds about 2%.")

# ------------------------------------------------------------ verdict ------
print("\n" + "=" * 78)
print("C.  VERDICT")
print("=" * 78)
_tight = float(np.percentile(_exact, 75) - np.percentile(_exact, 25)) < 0.05
_gridbound = _w001.max() > (THRESHOLDS[1] - THRESHOLDS[0])
print(f"  exact-optimum IQR < one grid step : {_tight}")
print(f"  some tile's 0.001-plateau exceeds a grid step: {_gridbound}")
print("""
  If the first is True, report the exact-maximiser distribution in Section 6.4
  and state the claim in terms of it. The grid then becomes a reporting
  resolution, exactly as Section 4.8 says, and the transfer claim is earned.

  If the first is False, replace "identical on every one of the forty-five
  held-out tiles, with zero spread" with a statement about grid points, move
  the Section 4.8 plateau caveat into 6.4 where the claim is made, and
  reconsider the first clause of the title.
""")

A.  PER-TILE OPTIMUM AT CONTINUOUS RESOLUTION
scored tiles: 45   (grid step of THRESHOLDS = 0.05)

  exact maximiser : median 0.9104   IQR 0.8260-0.9895
                    range  0.7053-6.0605   sd 0.7876
  plateau width   : J within 0.001 of max -> median 0.0056, max 3.8305
                    J within 0.01  of max -> median 0.0620, max 29.7098

  tiles whose GRID optimum is 0.90            : 10/45
  tiles whose EXACT optimum is within +/-0.025 of 0.90: 8/45
  distinct grid optima across tiles           : [0.4, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15, 1.2, 1.3, 1.55, 1.65, 1.7]

  READ: if the second count is much lower than the first, the zero
  spread is the grid, not the physics.

B.  SPATIAL K-FOLD SELECTION  (the training pool actually changes)
Tiles are sorted west to east and cut into K contiguous bands. theta*
is selected on all scored pixels OUTSIDE a band and applied to it.
K = 45 is leave-one-tile-out, i.e. what Section 6.4 currently reports.

   K   left out

In [28]:
# ============================================================================
#  §26   TILE SELECTION: WHAT THE 45 SCORED TILES REPRESENT   ->  Section 6.4
#
#  Paste as a new cell in Makilala_FoS_Pipeline_v10.ipynb, AFTER cell §10.
#  Uses only SCORED, OBSERVED, REF, ref_shape, ref_transform, pos, neg.
#  Mutates nothing.
#
#  WHY THIS EXISTS
#  ---------------
#  Section 6.4 says "Forty-five of the 108 tiles contain at least 200 pixels of
#  each class and are scored", states the rule once, and never defends it.
#  A referee's objection writes itself: requiring 200 pixels OF EACH CLASS
#  selects landslide-rich tiles, so the blocked estimate is conditioned on
#  landslide presence and is not an estimate over the municipality. The
#  headline result rests on that subset.
#
#  The objection is reasonable and has to be answered with numbers, not prose.
#  This cell produces them:
#
#    A  what the 45 tiles carry -- share of scored pixels, share of the
#       170,547 landslide pixels, and landslide prevalence inside vs outside
#    B  why each of the 63 excluded tiles is excluded, separated into
#       "no positives at all", "too few positives", "too few negatives"
#    C  THE DECISIVE TEST: pooled AUC recomputed on the union of the 45 tiles
#       alone. If that is close to the whole-domain 0.750, the tile subset is
#       not a systematically easier or harder subdomain, and the blocked
#       estimate is comparable with the pooled one. If it is far from 0.750,
#       the comparison in Section 6.4 is between two different populations and
#       has to be reframed.
#    D  sensitivity of every blocked number to the inclusion rule, at
#       MIN_BLOCK = 50, 100, 200, 500, and under a positives-only rule
#
#  HOW TO READ IT
#  --------------
#  The result to watch is C. B and D are supporting. If the blocked AUC is
#  stable across D, the 200 threshold is a reporting choice rather than a
#  result-determining one, and Section 6.4 can say so and show the table.
# ============================================================================
import numpy as np

guard()

_TILE_PX = int(round(2000 / abs(ref_transform.a)))
_fos = REF['fos']


def _auc(p, q):
    """Exact Mann-Whitney AUC from sorted arrays, ties at 0.5 (as in cell 10)."""
    if len(p) == 0 or len(q) == 0:
        return np.nan
    lt = np.searchsorted(q, p, 'left')
    le = np.searchsorted(q, p, 'right')
    return ((len(q) - le).sum() + 0.5 * (le - lt).sum()) / (len(p) * len(q))


# ----------------------------------------------------------- collect tiles --
_T = []                                    # (r0, c0, n_pos, n_neg, p, q)
for r0 in range(0, ref_shape[0], _TILE_PX):
    for c0 in range(0, ref_shape[1], _TILE_PX):
        sl = (slice(r0, r0 + _TILE_PX), slice(c0, c0 + _TILE_PX))
        s = SCORED[sl]
        if not s.any():
            continue
        f, o = _fos[sl], OBSERVED[sl]
        p = np.sort(f[s & o]); q = np.sort(f[s & ~o])
        _T.append((r0, c0, len(p), len(q), p, q))

NPOS = np.array([t[2] for t in _T])
NNEG = np.array([t[3] for t in _T])
TOT_POS, TOT_NEG = len(pos), len(neg)

print("=" * 78)
print("A.  WHAT THE SCORED TILES CARRY")
print("=" * 78)
keep = (NPOS >= 200) & (NNEG >= 200)
kp, kq = NPOS[keep].sum(), NNEG[keep].sum()
print(f"  tiles intersecting the scored domain : {len(_T)}")
print(f"  tiles with >= 200 of each class      : {int(keep.sum())}")
print(f"\n  scored pixels in those tiles         : {kp+kq:,} of {TOT_POS+TOT_NEG:,}"
      f"   ({100*(kp+kq)/(TOT_POS+TOT_NEG):.1f}%)")
print(f"  landslide pixels in those tiles      : {kp:,} of {TOT_POS:,}"
      f"   ({100*kp/TOT_POS:.1f}%)")
print(f"\n  landslide prevalence, scored domain  : {100*TOT_POS/(TOT_POS+TOT_NEG):.2f}%")
print(f"  landslide prevalence, the 45 tiles   : {100*kp/(kp+kq):.2f}%")
print(f"  landslide prevalence, excluded tiles : "
      f"{100*NPOS[~keep].sum()/max(1,(NPOS[~keep].sum()+NNEG[~keep].sum())):.2f}%")
print("\n  READ: the ratio of the second prevalence to the first is exactly how")
print("  landslide-rich the scored tiles are. Report it rather than let a")
print("  referee compute it.")

print("\n" + "=" * 78)
print("B.  WHY EACH EXCLUDED TILE IS EXCLUDED")
print("=" * 78)
zero_pos = (NPOS == 0)
few_pos = (NPOS > 0) & (NPOS < 200)
few_neg = (NPOS >= 200) & (NNEG < 200)
for lbl, m in (("no landslide pixels at all", zero_pos),
               ("1-199 landslide pixels", few_pos),
               ("enough positives, <200 negatives", few_neg)):
    sh = (NPOS[m].sum() + NNEG[m].sum()) / (TOT_POS + TOT_NEG)
    print(f"  {lbl:<34} {int(m.sum()):>4} tiles   {100*sh:5.1f}% of scored pixels")
print(f"\n  positives per excluded tile: median {np.median(NPOS[~keep]):.0f}, "
      f"max {NPOS[~keep].max()}")
print("  READ: if most exclusions are 'no landslide pixels at all', the rule is")
print("  not discarding evidence -- a tile with no positives has no AUC to")
print("  compute. That is the honest form of the defence.")

print("\n" + "=" * 78)
print("C.  IS THE 45-TILE UNION A DIFFERENT POPULATION?   [the decisive test]")
print("=" * 78)
P45 = np.sort(np.concatenate([t[4] for t, k in zip(_T, keep) if k]))
Q45 = np.sort(np.concatenate([t[5] for t, k in zip(_T, keep) if k]))
auc45 = _auc(P45, Q45)
print(f"  pooled AUC, whole scored domain      : {REF['auc']:.4f}")
print(f"  pooled AUC, union of the 45 tiles    : {auc45:.4f}")
print(f"  difference                           : {auc45 - REF['auc']:+.4f}")
print("""
  READ: these two are computed the same way on different pixel sets. If they
  agree to a couple of thousandths, the tile subset is not an easier or harder
  subdomain and the pooled-vs-blocked comparison in Section 6.4 is between two
  ways of PARTITIONING the same population, which is the claim the section
  needs. If they differ materially, say so and compare the blocked mean against
  THIS number instead of against the whole-domain 0.750.""")

print("\n" + "=" * 78)
print("D.  SENSITIVITY TO THE INCLUSION RULE")
print("=" * 78)
print(f"{'rule':<26} {'tiles':>6} {'blocked AUC':>22} {'pooled on union':>16}")
print("-" * 74)
for lbl, m in ([(f"both classes >= {n}", (NPOS >= n) & (NNEG >= n))
                for n in (50, 100, 200, 500)]
               + [("positives >= 200 only", NPOS >= 200),
                  ("positives >= 50 only", NPOS >= 50)]):
    if m.sum() < 2:
        print(f"{lbl:<26} {int(m.sum()):>6}   (too few tiles)"); continue
    a = np.array([_auc(t[4], t[5]) for t, k in zip(_T, m) if k])
    a = a[np.isfinite(a)]
    u = _auc(np.sort(np.concatenate([t[4] for t, k in zip(_T, m) if k])),
             np.sort(np.concatenate([t[5] for t, k in zip(_T, m) if k])))
    print(f"{lbl:<26} {int(m.sum()):>6}   {a.mean():.3f} +/- {a.std(ddof=1):.3f}"
          f"   ({a.min():.3f}-{a.max():.3f})   {u:.4f}")
print("""
  READ: if the blocked mean moves little from 50 to 500, the 200 threshold is a
  reporting choice and Section 6.4 should show this table and say so. If it
  climbs steadily with the threshold, the blocked estimate depends on how
  landslide-rich the retained tiles are, and that dependence has to be stated
  as a limitation rather than buried in a rule.
""")


A.  WHAT THE SCORED TILES CARRY
  tiles intersecting the scored domain : 108
  tiles with >= 200 of each class      : 45

  scored pixels in those tiles         : 5,203,449 of 7,104,440   (73.2%)
  landslide pixels in those tiles      : 169,580 of 170,547   (99.4%)

  landslide prevalence, scored domain  : 2.40%
  landslide prevalence, the 45 tiles   : 3.26%
  landslide prevalence, excluded tiles : 0.05%

  READ: the ratio of the second prevalence to the first is exactly how
  landslide-rich the scored tiles are. Report it rather than let a
  referee compute it.

B.  WHY EACH EXCLUDED TILE IS EXCLUDED
  no landslide pixels at all           51 tiles    18.9% of scored pixels
  1-199 landslide pixels               12 tiles     7.9% of scored pixels
  enough positives, <200 negatives      0 tiles     0.0% of scored pixels

  positives per excluded tile: median 0, max 178
  READ: if most exclusions are 'no landslide pixels at all', the rule is
  not discarding evidence -- a tile with no po

In [29]:
# ============================================================================
#  §27   BLOCK SIZE: WHERE 2 km COMES FROM   ->  Section 6.4
#
#  Paste as a new cell in Makilala_FoS_Pipeline_v10.ipynb, AFTER cell §10.
#  Uses only SCORED, OBSERVED, REF, THRESHOLDS, ref_shape, ref_transform,
#  pos, neg. Mutates nothing. Runtime is dominated by part A (~1-2 min).
#
#  WHY THIS EXISTS
#  ---------------
#  Section 6.4 says "we partitioned the scored domain into 2 km square tiles"
#  and never says why 2 km. As written, a referee is entitled to read it as the
#  number that produced 45 tiles. Standard practice in spatial cross-validation
#  sets the block size from the distance over which the quantity being
#  predicted -- or better, the model's own error -- stays correlated: blocks
#  smaller than that range leak information across the partition, blocks much
#  larger than it throw away data for nothing.
#
#  So the defensible procedure is: measure the range, then show the result over
#  a sweep of sizes around it.
#
#    A  empirical variograms, by random pair sampling, of
#         (i)  the landslide indicator  -- how far landsliding itself clusters
#         (ii) the model residual       -- how far the model's ERRORS cluster,
#              residual = 1{landslide} - 1{FoS < theta*}, which is the
#              quantity spatial CV is actually trying to decorrelate
#       Reported as binned semivariance against separation, with the practical
#       range: the first distance at which semivariance reaches 95% of its sill.
#
#    B  block-size sweep at 0.5, 1, 2, 4 and 8 km: tiles, scored tiles, share
#       of landslide pixels retained, within-block AUC, and pooled AUC on the
#       union of the scored tiles
#
#    C  does theta* still transfer at every block size? (two-fold spatial
#       split at each size, the hard version of the test from cell 27)
#
#  HOW TO READ IT
#  --------------
#  The number to quote in the paper is the RESIDUAL range from A(ii). If 2 km
#  is at or above it, the choice is justified and should be stated that way.
#  If the range is materially larger than 2 km, say so and report the sweep:
#  the honest position is then that 2 km is a lower bound on the block size and
#  the blocked estimate is, if anything, optimistic.
#
#  In B, watch whether within-block AUC falls monotonically with block size.
#  It should: larger blocks retain more internal regional contrast to exploit.
#  A monotone decay toward an asymptote is the strongest possible form of the
#  paper's argument -- it shows the pooled-to-blocked loss is a continuous
#  function of support, not an artifact of one arbitrary choice.
# ============================================================================
import numpy as np

guard()

PX_M = abs(ref_transform.a)                 # metres per pixel (5 m)
_fos = REF['fos']
_th = REF['theta']
rng = np.random.default_rng(20261022)


def _auc(p, q):
    if len(p) == 0 or len(q) == 0:
        return np.nan
    lt = np.searchsorted(q, p, 'left'); le = np.searchsorted(q, p, 'right')
    return ((len(q) - le).sum() + 0.5 * (le - lt).sum()) / (len(p) * len(q))


# ============================== A. VARIOGRAMS ==============================
print("=" * 78)
print("A.  HOW FAR DOES THE SIGNAL -- AND THE ERROR -- STAY CORRELATED?")
print("=" * 78)

R, C = np.nonzero(SCORED)
N = len(R)
M = min(60000, N)
sel = rng.choice(N, M, replace=False)
rs, cs = R[sel], C[sel]
ind = OBSERVED[rs, cs].astype(float)                  # 1 = landslide
pred = (_fos[rs, cs] < _th).astype(float)             # the model's call
resid = ind - pred                                    # -1, 0 or +1

MAXLAG_M = 12000.0
EDGES = np.concatenate([np.arange(0, 2000, 100), np.arange(2000, MAXLAG_M + 1, 500)])
NPAIR = 12_000_000

a = rng.integers(0, M, NPAIR); b = rng.integers(0, M, NPAIR)
ok = a != b
a, b = a[ok], b[ok]
d = np.hypot(rs[a].astype(np.int64) - rs[b], cs[a].astype(np.int64) - cs[b]) * PX_M
keep = d <= MAXLAG_M
a, b, d = a[keep], b[keep], d[keep]
bin_id = np.digitize(d, EDGES) - 1
nb = len(EDGES) - 1
cnt = np.bincount(bin_id, minlength=nb)[:nb]


def _vario(z):
    num = np.bincount(bin_id, weights=0.5 * (z[a] - z[b]) ** 2, minlength=nb)[:nb]
    with np.errstate(invalid='ignore', divide='ignore'):
        return num / cnt


def _range95(g, ctr, lo=1600.0, hi=4500.0, frac=0.95):
    """Practical range: first lag from which gamma STAYS at >= frac of the sill.

    CORRECTED 22 Oct 2026. The first version estimated the sill from lags beyond
    8 km. Both variograms here DECLINE past about 5 km -- the signature of
    large-scale non-stationarity across an elongated domain, where pairs that far
    apart lie at opposite ends of the municipality and have different means. Taking
    the sill from that declining tail put it BELOW the true plateau, so 95% of it
    was crossed in the first bin and the range came out at 50-250 m, which is
    nonsense. The sill is now the mean of the plateau itself, and the range is the
    lag from which gamma stays above the fraction rather than first touches it.
    """
    good = np.isfinite(g) & (cnt >= 500)
    pl = good & (ctr >= lo) & (ctr <= hi)
    if pl.sum() < 3:
        return np.nan, np.nan
    sill = float(np.nanmean(g[pl]))
    idx = np.where(good & (ctr <= hi))[0]
    thr = frac * sill
    for i in idx:
        if all(g[j] >= thr for j in idx if j >= i):
            return float(ctr[i]), sill
    return np.nan, sill


ctr = 0.5 * (EDGES[:-1] + EDGES[1:])
print(f"  sample: {M:,} scored pixels, {len(d):,} pairs within {MAXLAG_M/1000:.0f} km\n")
print(f"  {'lag (m)':>9} {'n pairs':>10} {'gamma indicator':>17} {'gamma residual':>16}")
print("  " + "-" * 56)
g_ind, g_res = _vario(ind), _vario(resid)
for i in range(nb):
    if cnt[i] < 500:
        continue
    if ctr[i] <= 2500 or i % 4 == 0:
        print(f"  {ctr[i]:>9.0f} {cnt[i]:>10,} {g_ind[i]:>17.5f} {g_res[i]:>16.5f}")

r_ind, s_ind = _range95(g_ind, ctr)
r_res, s_res = _range95(g_res, ctr)
r_ind99, _ = _range95(g_ind, ctr, frac=0.99)
r_res99, _ = _range95(g_res, ctr, frac=0.99)
print(f"\n  landslide indicator : sill {s_ind:.5f}, practical range "
      f"{r_ind:,.0f} m  ({r_ind/1000:.2f} km)")
print(f"  model residual      : sill {s_res:.5f}, practical range "
      f"{r_res:,.0f} m  ({r_res/1000:.2f} km)")
print(f"  99% of sill reached : indicator {r_ind99:,.0f} m, residual {r_res99:,.0f} m")
print(f"  long-lag decline    : gamma falls beyond ~5 km (non-stationarity across "
      f"an elongated domain); the sill above is the 1.6-4.5 km plateau, not the tail")
print(f"\n  block size in use   : 2,000 m")
print(f"  2 km is {'AT OR ABOVE' if r_res <= 2000 else 'BELOW'} the residual range.")
print("""
  READ: the residual range is the number that justifies the block size, because
  decorrelating the model's ERRORS is what blocked validation is for. If 2 km
  is at or above it, say so in Section 6.4 and cite this variogram. If it is
  below, 2 km is a lower bound and the blocked estimate is optimistic -- which
  is still reportable, and more honest than silence.""")

# ============================ B. BLOCK-SIZE SWEEP ==========================
print("\n" + "=" * 78)
print("B.  BLOCK-SIZE SWEEP   (inclusion rule held at 200 of each class)")
print("=" * 78)
print(f"{'size':>7} {'tiles':>6} {'scored':>7} {'% LS px kept':>13} "
      f"{'within-block AUC':>22} {'pooled on union':>16}")
print("-" * 76)
SWEEP = {}
for km in (0.5, 1.0, 2.0, 4.0, 8.0):
    tp = int(round(km * 1000 / PX_M))
    tot = sc = 0; aucs = []; Ps = []; Qs = []; kept = 0
    for r0 in range(0, ref_shape[0], tp):
        for c0 in range(0, ref_shape[1], tp):
            sl = (slice(r0, r0 + tp), slice(c0, c0 + tp))
            s = SCORED[sl]
            if not s.any():
                continue
            tot += 1
            f, o = _fos[sl], OBSERVED[sl]
            p = np.sort(f[s & o]); q = np.sort(f[s & ~o])
            if len(p) < 200 or len(q) < 200:
                continue
            sc += 1; kept += len(p)
            aucs.append(_auc(p, q)); Ps.append(p); Qs.append(q)
    aucs = np.array([x for x in aucs if np.isfinite(x)])
    if len(aucs) < 2:
        print(f"{km:>6.1f}k {tot:>6} {sc:>7}   (too few scored tiles)"); continue
    u = _auc(np.sort(np.concatenate(Ps)), np.sort(np.concatenate(Qs)))
    SWEEP[km] = (tot, sc, aucs, u)
    print(f"{km:>6.1f}k {tot:>6} {sc:>7} {100*kept/len(pos):>12.1f}% "
          f"  {aucs.mean():.3f} +/- {aucs.std(ddof=1):.3f} ({aucs.min():.3f}-{aucs.max():.3f})"
          f"   {u:.4f}")
print(f"\n  whole-domain pooled AUC for reference: {REF['auc']:.4f}")
print("""
  READ: within-block AUC should FALL as blocks shrink -- a smaller tile holds
  less regional contrast to exploit, so more of the ranking has to be earned
  slope by slope. A monotone decay is the strongest form of the paper's claim:
  it makes the pooled-to-blocked loss a continuous function of support rather
  than the consequence of one arbitrary tile size.""")

# ===================== C. DOES theta* SURVIVE AT EVERY SIZE? ===============
print("\n" + "=" * 78)
print("C.  THRESHOLD TRANSFER AT EACH BLOCK SIZE   (two-fold spatial split)")
print("=" * 78)
_gp = np.searchsorted(pos, THRESHOLDS, 'left')
_gq = np.searchsorted(neg, THRESHOLDS, 'left')
print(f"{'size':>7} {'theta* per half':<24} {'withheld':>9}")
print("-" * 44)
for km in sorted(SWEEP):
    tp = int(round(km * 1000 / PX_M))
    T = []
    for r0 in range(0, ref_shape[0], tp):
        for c0 in range(0, ref_shape[1], tp):
            sl = (slice(r0, r0 + tp), slice(c0, c0 + tp))
            s = SCORED[sl]
            if not s.any():
                continue
            f, o = _fos[sl], OBSERVED[sl]
            p = np.sort(f[s & o]); q = np.sort(f[s & ~o])
            if len(p) < 200 or len(q) < 200:
                continue
            T.append((c0, p, q))
    T.sort(key=lambda t: t[0])
    half = len(T) // 2
    ths, fr = [], []
    for band in (T[:half], T[half:]):
        if not band:
            continue
        hp = np.sort(np.concatenate([t[1] for t in band]))
        hq = np.sort(np.concatenate([t[2] for t in band]))
        HR = (_gp - np.searchsorted(hp, THRESHOLDS, 'left')) / (len(pos) - len(hp))
        FAR = (_gq - np.searchsorted(hq, THRESHOLDS, 'left')) / (len(neg) - len(hq))
        ths.append(THRESHOLDS[int(np.argmax(HR - FAR))])
        fr.append((len(hp) + len(hq)) / (len(pos) + len(neg)))
    print(f"{km:>6.1f}k {str(sorted(set(np.round(ths, 2)))):<24} {100*np.mean(fr):>8.0f}%")
print("""
  READ: theta* holding at 0.90 across every block size, while the within-block
  AUC falls steadily, is the cleanest statement of the paper's finding -- the
  calibration is indifferent to the support it is estimated on, the
  discrimination is not.
""")


A.  HOW FAR DOES THE SIGNAL -- AND THE ERROR -- STAY CORRELATED?
  sample: 60,000 scored pixels, 7,887,420 pairs within 12 km

    lag (m)    n pairs   gamma indicator   gamma residual
  --------------------------------------------------------
         50      1,826           0.02136          0.12815
        150      5,062           0.02252          0.16061
        250      8,175           0.02563          0.16807
        350     11,161           0.02562          0.17391
        450     13,924           0.02560          0.17811
        550     16,824           0.02776          0.18090
        650     19,632           0.02799          0.17915
        750     22,367           0.02676          0.18199
        850     25,049           0.02805          0.18228
        950     27,309           0.02750          0.18139
       1050     29,529           0.02645          0.18135
       1150     32,354           0.02723          0.18379
       1250     34,216           0.02730          0.18613
  

In [30]:
# ============================================================================
#  §28   BLOCKED VALIDATION: FIGURE AND PER-TILE TABLE   ->  Section 6.4
#
#  New in v11. Produces the two things Section 6.4 is missing: it is the only
#  validation subsection with neither a figure nor a table, while the pixel,
#  polygon and intensity-stratified subsections each have one or both. Forty-
#  five tiles with five statistics apiece currently exist only as running prose.
#
#  Outputs, into OUTPUT_DIR (as configured in §0):
#    fig7_blocked_validation.png   300 dpi, two panels
#        (a) the 108-tile grid, scored tiles shaded by within-block AUC,
#            unscored tiles HATCHED (not colour-coded) so the distinction
#            survives greyscale printing and colour-vision deficiency
#        (b) distribution of the 45 within-block AUCs, with the pooled value,
#            the blocked mean and chance marked
#    table_S_blocked_tiles.tex     a longtable of all 45 tiles for the
#                                  supplementary: n+, n-, AUC, theta*, HR,
#                                  FAR, J
#
#  Colour: one hue, light to dark (matplotlib "Blues"), which is the rule for a
#  sequential/magnitude encoding. No rainbow, no diverging scheme -- AUC here
#  has no meaningful midpoint to diverge about. Chance (0.5) is marked on the
#  colourbar instead. The hazard maps elsewhere in the paper use a red-green
#  classed scheme; this figure deliberately does not, so the two are not
#  confused for one another.
# ============================================================================
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

guard()

# v11 writes beside the v10 products; the pipeline defines OUTPUT_DIR in §0.
OUT = globals().get('OUT') or OUTPUT_DIR
os.makedirs(OUT, exist_ok=True)

TILE_PX = int(round(2000 / abs(ref_transform.a)))
MIN_BLOCK = 200
_fos, _th = REF['fos'], REF['theta']


def _auc(p, q):
    if len(p) == 0 or len(q) == 0:
        return np.nan
    lt = np.searchsorted(q, p, 'left'); le = np.searchsorted(q, p, 'right')
    return ((len(q) - le).sum() + 0.5 * (le - lt).sum()) / (len(p) * len(q))


nR = int(np.ceil(ref_shape[0] / TILE_PX))
nC = int(np.ceil(ref_shape[1] / TILE_PX))
AUCg = np.full((nR, nC), np.nan)      # scored tiles only
INTER = np.zeros((nR, nC), bool)      # intersects the scored domain
rows = []

for i, r0 in enumerate(range(0, ref_shape[0], TILE_PX)):
    for j, c0 in enumerate(range(0, ref_shape[1], TILE_PX)):
        sl = (slice(r0, r0 + TILE_PX), slice(c0, c0 + TILE_PX))
        s = SCORED[sl]
        if not s.any():
            continue
        INTER[i, j] = True
        f, o = _fos[sl], OBSERVED[sl]
        p = np.sort(f[s & o]); q = np.sort(f[s & ~o])
        if len(p) < MIN_BLOCK or len(q) < MIN_BLOCK:
            continue
        a = _auc(p, q)
        AUCg[i, j] = a
        hr = (p < _th).sum() / len(p)
        far = (q < _th).sum() / len(q)
        x, y = ref_transform * (c0 + TILE_PX / 2, r0 + TILE_PX / 2)
        rows.append(dict(i=i, j=j, x=x, y=y, npos=len(p), nneg=len(q),
                         auc=a, hr=hr, far=far, J=hr - far))

A = np.array([r['auc'] for r in rows])
print(f"tiles intersecting scored domain : {int(INTER.sum())}")
print(f"tiles scored                     : {len(rows)}")
print(f"within-block AUC                 : {A.mean():.3f} +/- {A.std(ddof=1):.3f}"
      f"   range {A.min():.3f}-{A.max():.3f}")
print(f"tiles below chance (AUC < 0.5)   : {int((A < 0.5).sum())}")

# ------------------------------------------------------------------ figure --
fig = plt.figure(figsize=(11.0, 4.6))
gs = fig.add_gridspec(1, 2, width_ratios=[1.35, 1.0], wspace=0.26)

# (a) map -------------------------------------------------------------------
ax = fig.add_subplot(gs[0, 0])
cmap = plt.get_cmap('Blues').copy()
im = ax.imshow(np.ma.masked_invalid(AUCg), cmap=cmap, vmin=0.5, vmax=0.95,
               interpolation='nearest')
for i in range(nR):
    for j in range(nC):
        if INTER[i, j] and not np.isfinite(AUCg[i, j]):
            ax.add_patch(Rectangle((j - .5, i - .5), 1, 1, facecolor='0.93',
                                   edgecolor='0.55', hatch='///', lw=0.4))
        if np.isfinite(AUCg[i, j]):
            v = AUCg[i, j]
            ax.text(j, i, f"{v:.2f}".lstrip('0'), ha='center', va='center',
                    fontsize=5.6, color='white' if v > 0.78 else '0.15')
ax.set_xticks([]); ax.set_yticks([])
for sp in ax.spines.values():
    sp.set_visible(False)
ax.set_title('(a)  within-block AUC, 2 km tiles', fontsize=9, loc='left')
cb = fig.colorbar(im, ax=ax, fraction=0.040, pad=0.02)
cb.set_label('AUC', fontsize=8)
cb.ax.tick_params(labelsize=7)
cb.ax.axhline(0.5, color='0.25', lw=1.0)
ax.add_patch(Rectangle((0, 0), 0, 0, facecolor='0.93', edgecolor='0.55',
                       hatch='///', lw=0.4, label='not scored'))
ax.legend(loc='lower left', fontsize=7, frameon=False,
          bbox_to_anchor=(0.0, -0.09))

# (b) distribution ----------------------------------------------------------
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(A, bins=np.arange(0.20, 1.001, 0.05), color=cmap(0.55),
         edgecolor='white', linewidth=1.2)
for v, lab, st in ((0.5, 'chance', ':'), (A.mean(), f'blocked mean {A.mean():.3f}', '-'),
                   (REF['auc'], f"pooled {REF['auc']:.3f}", '--')):
    ax2.axvline(v, color='0.20', lw=1.4, ls=st)
    ax2.text(v, ax2.get_ylim()[1] * 0.97, ' ' + lab, rotation=90, va='top',
             ha='left', fontsize=7, color='0.20')
ax2.set_xlabel('within-block AUC', fontsize=8)
ax2.set_ylabel('tiles', fontsize=8)
ax2.tick_params(labelsize=7)
ax2.grid(axis='y', color='0.90', lw=0.6)
ax2.set_axisbelow(True)
for sp in ('top', 'right'):
    ax2.spines[sp].set_visible(False)
for sp in ('left', 'bottom'):
    ax2.spines[sp].set_color('0.6')
ax2.set_title(f'(b)  distribution over the {len(rows)} scored tiles',
              fontsize=9, loc='left')

png = os.path.join(OUT, 'fig7_blocked_validation.png')
fig.savefig(png, dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig)
print(f"\nwrote {png}")

# ----------------------------------------------------------- latex table ---
rows_sorted = sorted(rows, key=lambda r: -r['auc'])
tex = [
    r"\begin{longtable}{@{}rrrrrrr@{}}",
    r"\caption{Per-tile results of the spatially blocked validation "
    r"(Section~\ref{sec:blocked_validation}). Tiles are 2~km squares of the scored "
    r"domain carrying at least 200 pixels of each class; $n^{+}$ and $n^{-}$ are "
    r"landslide and non-landslide pixel counts. HR, FAR and $J$ are evaluated at the "
    r"domain threshold $\theta^\ast = 0.90$. Ordered by within-block AUC.}"
    r"\label{tab:S_blocked}\\",
    r"\toprule",
    r"Tile & $n^{+}$ & $n^{-}$ & AUC & HR & FAR & $J$ \\",
    r"\midrule", r"\endfirsthead",
    r"\toprule", r"Tile & $n^{+}$ & $n^{-}$ & AUC & HR & FAR & $J$ \\",
    r"\midrule", r"\endhead", r"\bottomrule", r"\endfoot",
]
for k, r in enumerate(rows_sorted, 1):
    tex.append(f"{k} & {r['npos']:,} & {r['nneg']:,} & {r['auc']:.3f} & "
               f"{100*r['hr']:.1f}\\% & {100*r['far']:.1f}\\% & {r['J']:.3f} \\\\"
               .replace(',', '{,}'))
tex += [r"\midrule",
        f"\\multicolumn{{3}}{{@{{}}l}}{{Mean over {len(rows)} tiles}} & "
        f"{A.mean():.3f} & "
        f"{100*np.mean([r['hr'] for r in rows]):.1f}\\% & "
        f"{100*np.mean([r['far'] for r in rows]):.1f}\\% & "
        f"{np.mean([r['J'] for r in rows]):.3f} \\\\",
        r"\end{longtable}"]
tp = os.path.join(OUT, 'table_S_blocked_tiles.tex')
open(tp, 'w').write('\n'.join(tex) + '\n')
print(f"wrote {tp}  ({len(rows)} rows)")

print(f"\nmean HR {100*np.mean([r['hr'] for r in rows]):.1f}%, "
      f"mean FAR {100*np.mean([r['far'] for r in rows]):.1f}%, "
      f"mean J {np.mean([r['J'] for r in rows]):.3f}")
print("These should reproduce the 51.1% / 27.0% / 0.241 quoted in Section 6.4.")


tiles intersecting scored domain : 108
tiles scored                     : 45
within-block AUC                 : 0.667 +/- 0.155   range 0.211-0.894
tiles below chance (AUC < 0.5)   : 5

wrote /Users/jayroldarcede/Downloads/makilala_data/output_maps_v10/fig7_blocked_validation.png
wrote /Users/jayroldarcede/Downloads/makilala_data/output_maps_v10/table_S_blocked_tiles.tex  (45 rows)

mean HR 51.1%, mean FAR 27.0%, mean J 0.241
These should reproduce the 51.1% / 27.0% / 0.241 quoted in Section 6.4.
